In [ ]:
# #################################### STEP1 ##################################################
import streamlit as st
import pandas as pd
import datetime
from datetime import datetime,timedelta
import numpy as np
from snowflake.snowpark.context import get_active_session
import re
import yaml
import logging
import os
import time
# PLANT,DEALER, DISPATCH_QTY_TILL_MONTH, IS_PART_PLAN -- dispatch daily table


def get_run_cycle():
    # Get the current time
    current_time = datetime.now().time()
    # Define time ranges
    if current_time >= datetime.strptime("00:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("11:00:00", "%H:%M:%S").time():
        return 1,current_time
    elif current_time >= datetime.strptime("11:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("14:00:00", "%H:%M:%S").time():
        return 2,current_time
    elif current_time >= datetime.strptime("14:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("16:00:00", "%H:%M:%S").time():
        return 3,current_time
    elif current_time >= datetime.strptime("16:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("18:00:00", "%H:%M:%S").time():
        return 4,current_time
    else:
        return 5,current_time

# Example usage
cycle_id,current_time = get_run_cycle()
print(f"Current run cycle: {cycle_id}")

# Initialize logger
logging.basicConfig(level=logging.INFO,format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Add the B2G < 0 table and 

def drop_table(table_name):
    query = f'DROP TABLE IF EXISTS {table_name};'
    logger.info(f'Dropping table: {table_name}')
    session.sql(query).collect()
    

def delete_insert(table_name, cycle_id, current_date=pd.Timestamp.now().date()):
    # Construct the SQL query with both conditions
    query = f"""
        DELETE FROM {table_name} 
        WHERE RUN_DATE = '{current_date}' 
        AND RUN_CYCLE_ID = {cycle_id};
    """
    # Log the action
    logger.info(f"Deleting entries from {table_name} where RUN_DATE = {current_date} and RUN_CYCLE_ID = {cycle_id}")
    # Execute the query
    session.sql(query).collect()

    #session.sql.collect()

def write_to_snowflake(df, table_name, mode="append"):     
    logger.info(f'Writing to Snowflake table {table_name} with mode {mode}')
    agg_data_sp = session.create_dataframe(df)
    agg_data_sp.write.mode(mode).save_as_table(table_name)


#from geopy.distance import geodesic as GD

# Initialize session
session = get_active_session()


def remove_trailing_numbers(input_string):
    result = re.sub(r' \d+.', '', input_string)
    return result.strip()

config={
  "start_date": "2026-10-01",
  "customer_types": ["Individual"],
  "agg_type": "daily",
  "combine_models": "",
  "planning_month": "2026-10-01",
  "CALDAY" : "20261001",
  "CAL_DATE" : "2026-10-01",
  "database"  : {
     "mop_table" :"ANALYTICS_DATABASE.ANALYTICS_SALES.Dispatch_Plan_F_monthwise",
      "OBD_MAPPING" : "MOP_DATABASE.SOQ.OBD2_MAPPING_VIEW",
   # "mop_table": "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.MOP_SEP_2024_SAMPLE_P03",
    "ecr_sales_table": "ANALYTICS_DATABASE.ANALYTICS_SALES.CUSTOMER_RETAILS",
    "predicted_sales_table": "WORK_DATABASE.MOP.SOQ_DATA_FINAL",
    "stock_availability_table": "ANALYTICS_DATABASE.ANALYTICS_SALES.STOCK_AVAILABILITY",
    "dispatch_table" : "ANALYTICS_DATABASE.ANALYTICS_SALES.FINAL_DISPATCH_VIEW",
    "Weekly_table":"MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Weekly_dispatch_prioritisation",
    "Daily_table":"MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation",
    "ZeroB2G_table" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.ZeroB2G_table",
    "Unconstrained_table" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file",
    "DailyDispatchTable" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchTable",
    "DailyLeadTimeTable" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyLeadTimeTable",
    "runner_skus_table" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.RUNNER_SKU_TABLE"
  }
}




# Fetch Parent Dealer Mapping
def getParentDealerMapping():
    query = '''
    SELECT DISTINCT X_DEALER_CODE_HIER AS DEALER_CODE, PAR_ORG_NAME 
    FROM FIVETRAN_DATABASE.ORACLE_LDP_OLAP_SCHEMA.WC_INT_ORG_DH 
    WHERE X_DEALER_CODE_HIER IS NOT NULL
    '''
    parent_dealer_mapping = session.sql(query).to_pandas()
    parent_dealer_mapping['PARENT_DEALER_CODE'] = parent_dealer_mapping['PAR_ORG_NAME'].apply(lambda x: str(x).split("-")[0].strip())
    logger.info(f'Retrieved {len(parent_dealer_mapping)} parent dealer mappings')
    return parent_dealer_mapping

# Fetch SKU Supercedence data
def fetchSKUSupercedence():
    query = '''SELECT * FROM MOP_DATABASE.SOQ.SKU_SUPERCEDENCE'''
    logger.info('Fetching SKU Supercedence data')
    data = session.sql(query).to_pandas()
    data = data.rename(columns={"UNIQUEFAMILYCODE": 'UNIQUE FAMILY CODE'})
    logger.info(f'Retrieved {data.shape[0]} rows of SKU Supercedence data')
    return data

def fetchOBDData(session):
    OBD_MAPPING = config['database']['OBD_MAPPING']
    obd_mapping = session.table(OBD_MAPPING).to_pandas()
    obd_mapping['SKU'] = obd_mapping['PREVIOUS_OBD_SKU']
    return obd_mapping
    
# Fetch Daily ECR Sales
def getDailyECR(customer_type_to_include, start_date, config):
    customer_type_str = ",".join([f"'{types}'" for types in customer_type_to_include])
    ecr_sales_table = config['database']['ecr_sales_table']
    
    query = f'''
    SELECT  DEALER_CODE, MODEL, SKU, CAL_DATE, MONTH(CAL_DATE) AS CAL_MONTH, YEAR(CAL_DATE) AS CAL_YEAR, 
           SUM(INVOICED_SALES) AS INVOICED_SALES, SUM(CANCELLED_SALES) AS CANCELLED_SALES, SUM(RETURNED_SALES) AS RETURNED_SALES 
    FROM {ecr_sales_table}
    WHERE X_CUSTOMER_TYPE IN ({customer_type_str}) 
      AND CAL_DATE >= current_date - (3 * 365)
    GROUP BY DEALER_CODE, MODEL, SKU, CAL_DATE
    '''
    logger.info(f'Fetching Daily ECR Sales from {ecr_sales_table} starting {start_date}')
    ecr_sales = session.sql(query).to_pandas()
    obd_data=fetchOBDData(session)
   
    ecr_sales=pd.merge(ecr_sales,obd_data,on="SKU",how="left")
   
        ## Null in Current ObD SKU - means, current SKU is the same as sku
   
    ecr_sales['CURRENT_OBD_SKU'] = ecr_sales['CURRENT_OBD_SKU'].fillna(ecr_sales['SKU'])
   
    ecr_sales=ecr_sales.drop(['SKU'],axis=1)
    ecr_sales=ecr_sales.rename(columns={'CURRENT_OBD_SKU':'SKU'})
    
    ecr_sales['NET_SALES'] = ecr_sales['INVOICED_SALES'].fillna(0) + ecr_sales['CANCELLED_SALES'].fillna(0) + ecr_sales['RETURNED_SALES'].fillna(0)
    logger.info(f'Retrieved {len(ecr_sales)} rows of Daily ECR sales')
    return ecr_sales
    
def fetchMOPData(config,parent_dealer_mapping):
    mop_table = config['database']['mop_table']
    CAL_DATE  = config['CAL_DATE']
    logger.info(f'Fetching MOP Data from {mop_table}')
    query = f"""select DISTINCT DEALERCODE as DEALER_CODE, MODEL,PLANT, SKU, PLAN_QTY AS QUANTITY, VERSION    
                from {mop_table}   
                where CAL_DATE = '{CAL_DATE}' 
                and  VERSION = (
                select distinct version from {mop_table} where  CAL_DATE = '{CAL_DATE}' order by version desc limit 1);"""
    print(query)
    mop_data = session.sql(query).to_pandas()
    ("list of DEALER WITH NO PARENT DEALERS :\n", mop_data.loc[~mop_data.DEALER_CODE.isin(parent_dealer_mapping.DEALER_CODE),'DEALER_CODE'] )
    print(os.getcwd())
    #mop_data.loc[~(mop_data.DEALER_CODE.isin(parent_dealer_mapping.DEALER_CODE)),'DEALER_CODE'].to_csv('snow://notebook/MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DISPATCH_DEMAND_20240923/missing_delaers.csv', index=False)
    logger.info(f'Retrieved missing dealers with parent dealers to missing_dealers.csv file')
    
    #print(mop_data.loc[~(mop_data.DEALER_CODE.isin(parent_dealer_mapping.DEALER_CODE)),'DEALER_CODE'].shape)
    
    # --- CODE EDIT STARTS ---
    mop_data = pd.merge(mop_data, parent_dealer_mapping[['DEALER_CODE', 'PARENT_DEALER_CODE']], on="DEALER_CODE", how="left")
    
    null_before = mop_data['PARENT_DEALER_CODE'].isna().sum()
    print(f"Number of rows with NULL PARENT_DEALER_CODE before fallback: {null_before}")
    
    dealer_code_str = mop_data['DEALER_CODE'].astype(str)
    condition = dealer_code_str.str.startswith('1') & mop_data['PARENT_DEALER_CODE'].isna()
    mop_data.loc[condition, 'PARENT_DEALER_CODE'] = mop_data.loc[condition, 'DEALER_CODE']
    
    null_after = mop_data['PARENT_DEALER_CODE'].isna().sum()
    print(f"Number of rows with NULL PARENT_DEALER_CODE after fallback: {null_after}")

    if null_after>0:
        # mop_data = mop_data.dropna(subset=['PARENT_DEALER_CODE']).reset_index(drop=True)
        print(f"MOP_DATA table still has null values in parent dealer code")
   
    mop_data = pd.merge(mop_data, parent_dealer_mapping[['DEALER_CODE', 'PARENT_DEALER_CODE']], on="DEALER_CODE", how="inner")
    mop_data = mop_data.groupby(['DEALER_CODE', 'SKU', 'PLANT','VERSION', 'MODEL'])['QUANTITY'].sum().reset_index()
    mop_data.columns = ['DEALER', 'SKU','PLANT','VERSION','MODEL', 'QUANTITY']
    # obd_data = fetchOBDData(session)
    # mop_data = pd.merge(mop_data, obd_data, on="SKU", how="left")
   
    # # Null in Current OBD SKU means the current SKU is the same as SKU
    # mop_data['CURRENT_OBD_SKU'] = mop_data['CURRENT_OBD_SKU'].fillna(mop_data['SKU'])

    # mop_data.drop('SKU',axis=1,inplace=True)
    # mop_data = mop_data.rename(columns={'CURRENT_OBD_SKU': 'SKU'})
    mop_data.DEALER   = mop_data.DEALER.astype(int).astype(str)
    logger.info(f'Retrieved {len(mop_data)} rows of MOP data')
    
    return mop_data

def fetchTransitTime():
    query = f'''
    select  
    DISTINCT
    PLANT, 
    TRIM(PARENT_DEALER_CODE) AS DEALER, SKU, TRANSIT_TIME AS TRANSIT_TIME FROM MOP_DATABASE.SOQ.TRANSIT_DEALER_SKU_PLANT_MAPPING_NEW;'''

    
    #TRIM(PARENT_DEALER_CODE) AS DEALER, SKU, TRANSIT_TIME AS TRANSIT_TIME FROM MOP_DATABASE.SOQ.PARENT_DEALER_TRANSIT_TIME_SKU_NEW;'''
    Transit_time = session.sql(query).to_pandas()
    
    
    data = {
        "PLANT": ['HHHG', 'HHHU', 'HHHD', 'HHHG', 'HM4N', 'HHHG', 'HHHU', 'HM5V', 'HM5V', 'HM5V', 'HM5V', 'HM5V', 'HM5V', 'HM5V', 'HM5V', 'HM5V', 'HM5V', 'HM5V', 'HM5V', 'HM5V', 'HM5V', 'HM5V'],
        "DEALER": [12266, 12266, 12266, 17057, 12266, 17043, 17043, 12236, 12205, 11964, 11963, 11467, 11298, 11297, 11246, 10480, 10239, 11740, 12117, 11370, 11255, 11570],
        "TRANSIT_TIME": [3, 4, 3, 7, 3, 3, 3, 9, 10, 9, 10, 10, 8, 10, 9, 10, 11, 9, 9, 9, 10, 10]
    }
    

    df = pd.DataFrame(data)
    df.DEALER = df.DEALER.astype(str)
    df = pd.merge(df,Transit_time[['PLANT','DEALER','SKU']].copy(), on=['PLANT','DEALER'],how='left')

    Transit_time= pd.concat([Transit_time,df],axis=0)
    Transit_time.sort_values(['PLANT','DEALER','SKU','TRANSIT_TIME'],ascending=[1,1,1,0],inplace=True)
    Transit_time.drop_duplicates(['PLANT','DEALER','SKU'], inplace=True)
    

    return Transit_time

# Get Predicted Sales
def fetchPredictedSales(config,filter):
    predicted_sales_table = config['database']['predicted_sales_table']
    planning_month = config['planning_month']
    CAL_DATE  = config['CAL_DATE']
    #STOCK_CAL_DATE = '20260403'
    predicted_sales_query=f'''
    SELECT PARENT_DEALER_CODE AS DEALER, SKU, PREDICTED_SALES_SKU from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.PREDICTION_SKU_DATA
    WHERE PLANNING_MONTH= '{CAL_DATE}';
    '''
    

    logger.info(f'Fetching Predicted Sales from PREDICTION_SKU_DATA table for month {planning_month}')

    predicted_sales=session.sql(predicted_sales_query).to_pandas()

    logger.info(f'Shape of Predicted Sales {predicted_sales.shape}')

    ## fetch safety stock
    query = f"""    SELECT DISTINCT PARENT_DEALER_CODE AS DEALER,SKU,
        ROUND(LEAST(SAFETY_STOCK, PREDICTED_SALES_SKU * 3)) AS SAFETY_STOCK, SAFETY_STOCK_DAYS
        FROM MOP_DATABASE.SOQ.SOQ_DATA_FINAL_VERSION_WITH_AO_VERSION_4  
        where  PLANNING_MONTH = '{CAL_DATE}' AND STOCK_DATE_PERIOD='end'  
        AND DEMAND_VARIABILITY_TYPE='SKU_BASED' AND SERVICE_LEVEL=80 and run_version = 57;"""

    #RUN_VERSION=1 AND RUN_DATE='20250601'   AND STOCK_DATE_PERIOD='end' and planning_month='2025-06-01' and  SERVICE_LEVEL=90 AND DEMAND_VARIABILITY_TYPE='SKU_BASED'

    safety_stock=session.sql(query).to_pandas()
    logger.info(f'Fetching Safety Stock from {predicted_sales_table} for month {planning_month}')
    logger.info(f'Shape of Safety Stock {safety_stock.shape}')
    predicted_sales=pd.merge(predicted_sales,safety_stock,on=['DEALER','SKU'])
    logger.info(f'Shape  Safety Stock+Prediction {predicted_sales.shape}')
    

    
    logger.info(f'Fetching Predicted Sales from {predicted_sales_table} for month {planning_month}')
    now = datetime.now()
    next_month = now.replace(day=28) + timedelta(days=4)  # Guarantees we move to the next month
    last_day = next_month.replace(day=1) - timedelta(days=1)
    print(last_day.day, "last day")
        

    
    Transit_time=fetchTransitTime()
    predicted_sales=pd.merge(predicted_sales,Transit_time, on =['DEALER','SKU'], how='left')
    ###  making transit time if Null to 0 -aiswarya
   
    predicted_sales.loc[pd.isnull(predicted_sales['TRANSIT_TIME']),'TRANSIT_TIME']=0
    print(predicted_sales.columns)
    predicted_sales['AVG_LEAD_TIME_STOCK'] = np.ceil((predicted_sales.PREDICTED_SALES_SKU.fillna(0)/last_day.day) * predicted_sales.TRANSIT_TIME.fillna(0)) 
    predicted_sales['REORDER_LEVEL'] = predicted_sales.SAFETY_STOCK.fillna(0) + predicted_sales.AVG_LEAD_TIME_STOCK.fillna(0)
    
    if filter == 'True':
        predicted_sales = predicted_sales.loc[(predicted_sales.PREDICTED_SALES_SKU.fillna(0) > 0) | (predicted_sales.REORDER_LEVEL.fillna(0) > 0)]
    logger.info(f'Retrieved {predicted_sales.shape[0]} rows of Predicted Sales data')
    
    return predicted_sales

def open_order_block(session):
    """
    Create a fabricated lookup table for SKU, PLANT, DEALER, and OPEN_ORDER_QTY.
    :param final_prioritisation_df: DataFrame containing prioritisation data
    :return: Fabricated lookup DataFrame with SKU, PLANT, DEALER, and OPEN_ORDER_QTY
    """
    # Filter rows where B2G > 5 and select unique combinations of SKU, PLANT, and DEALER
    yday = (datetime.today() - timedelta(days=1)).strftime('%Y%m%d')
    CALDAY = config['CALDAY']
    query = fquery = f"""
        SELECT
            PLANT,
            DEALER_CODE AS DEALER,
            SKU,
            SUM(TOTAL_OPEN_ORDER_QTY) AS TOTAL_OPEN_ORDER_QTY
        FROM SAP_DATA_DATABASE.SAP_DATA.VW_OPEN_ORDERS_LIVE
        WHERE OPEN_ORDER_DATE >= {CALDAY}
        GROUP BY PLANT, DEALER, SKU; """
    try :
        
        open_order_block = session.sql(query).to_pandas()
        
   
        if open_order_block.shape[0] > 0 :
            open_order_block['OPEN_ORDER_BLOCK_STATUS'] = 'Blocked'
        else :
            open_order_block['OPEN_ORDER_BLOCK_STATUS'] = None   
        return open_order_block
       
    except:
        print(" Open orders data not present")
        open_order_block = pd.DataFrame(data= None, columns = ['PLANT', 'DEALER', 'SKU' ,'TOTAL_OPEN_ORDER_QTY','OPEN_ORDER_BLOCK_STATUS'])
        return open_order_block



    


def fetchDispatchData(config, parent_dealer_mapping):
    dispatch_table = config['database']['dispatch_table']
    CALDAY = config['CALDAY']

    yday = (datetime.today() - timedelta(days=1)).strftime('%Y%m%d')
    
    query = f"""select SOLD_TO as DEALER_CODE,  SKU , PLANT, sum(QUANT_B) as Quantity_dispatched 
        from {dispatch_table} where CALDAY >= {CALDAY} group by 1,2,3;"""

    try :
        Dispatch_data = session.sql(query).to_pandas()
        if Dispatch_data.shape[0] > 0:
            Dispatch_data.DEALER_CODE = Dispatch_data.DEALER_CODE.astype(int).astype(str)
            Dispatch_data = pd.merge(Dispatch_data, parent_dealer_mapping[['DEALER_CODE', 'PARENT_DEALER_CODE']], on="DEALER_CODE", how="inner")
            Dispatch_data = Dispatch_data.groupby(['DEALER_CODE', 'SKU','PLANT'])['QUANTITY_DISPATCHED'].sum().reset_index()
            Dispatch_data.columns = ['DEALER', 'SKU', 'PLANT', 'QUANTITY_DISPATCHED']
        #Dispatch_data.DEALER = Dispatch_data.DEALER.astype(int).astype(str)
            logger.info(f'Retrieved {len(Dispatch_data)} rows of Dispatch data')
            return Dispatch_data
        else:
            Dispatch_data = pd.DataFrame(data= None, columns = ['DEALER', 'SKU', 'PLANT' ,'QUANTITY_DISPATCHED'])
            return Dispatch_data  
    except:
        print(" Dispatch  data not present")
        Dispatch_data = pd.DataFrame(data= None, columns = ['DEALER', 'SKU', 'PLANT' ,'QUANTITY_DISPATCHED'])
        return Dispatch_data     


def fetchTodayDispatchData(config, parent_dealer_mapping):
    dispatch_table = config['database']['dispatch_table']
    CALDAY = config['CALDAY']

    yday = (datetime.today() - timedelta(days=1)).strftime('%Y%m%d')
    tday = (datetime.today()).strftime('%Y%m%d')
    
    query = f"""select SOLD_TO as DEALER_CODE, SKU , PLANT, sum(QUANT_B) as Quantity_dispatched 
        from {dispatch_table} where CALDAY = {tday} group by 1,2,3;"""

    try :
        Dispatch_data = session.sql(query).to_pandas()
        if Dispatch_data.shape[0] > 0:
            Dispatch_data.DEALER_CODE = Dispatch_data.DEALER_CODE.astype(int).astype(str)
            Dispatch_data = pd.merge(Dispatch_data, parent_dealer_mapping[['DEALER_CODE', 'PARENT_DEALER_CODE']], on="DEALER_CODE", how="inner")
            Dispatch_data = Dispatch_data.groupby(['DEALER_CODE', 'SKU'])['QUANTITY_DISPATCHED'].sum().reset_index()
            Dispatch_data.columns = ['DEALER', 'SKU', 'QUANTITY_DISPATCHED_TODAY']
        #Dispatch_data.DEALER = Dispatch_data.DEALER.astype(int).astype(str)
            logger.info(f'Retrieved {len(Dispatch_data)} rows of Dispatch data')
            return Dispatch_data
        else:
            Dispatch_data = pd.DataFrame(data= None, columns = ['DEALER', 'SKU' ,'QUANTITY_DISPATCHED_TODAY'])
            return Dispatch_data  
    except:
        print(" Dispatch  data not present")
        Dispatch_data = pd.DataFrame(data= None, columns = ['DEALER', 'SKU' ,'QUANTITY_DISPATCHED_TODAY'])
        return Dispatch_data     


# def fetchPlantLevelDispatchData(config, parent_dealer_mapping):
#     dispatch_table = config['database']['dispatch_table']
#     CALDAY = config['CALDAY']
#     query = f"""select SOLD_TO as DEALER_CODE, BIC_ZSD_MATNR as SKU , PLANT, sum(QUANT_B) as Quantity_dispatched 
#         from {dispatch_table} where CALDAY >= {CALDAY} group by 1,2,3;"""

#     logger.info(f'Fetching Dispatch data from {dispatch_table} after {CALDAY}')
#     Dispatch_data = session.sql(query).to_pandas()
#     Dispatch_data.DEALER_CODE = Dispatch_data.DEALER_CODE.astype(int).astype(str)
#     Dispatch_data = pd.merge(Dispatch_data, parent_dealer_mapping[['DEALER_CODE', 'PARENT_DEALER_CODE']], on="DEALER_CODE", how="inner")
#     Dispatch_data = Dispatch_data.groupby(['DEALER_CODE', 'SKU','PLANT'])['QUANTITY_DISPATCHED'].sum().reset_index()
#     Dispatch_data.columns = ['DEALER', 'SKU', 'PLANT', 'QUANTITY_DISPATCHED']
#     #Dispatch_data.DEALER = Dispatch_data.DEALER.astype(int).astype(str)
#     logger.info(f'Retrieved {len(Dispatch_data)} rows of Dispatch data')
#     return Dispatch_data


# def fetchDealerInventory(config):
#     stock_availability_table = config['database']['stock_availability_table']
#     CAL_DATE = config['CAL_DATE']
    
#     yday = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')
#     query = f"""
#         SELECT DEALER_CODE, SKU, DATE(CAL_DATE) as CLOSING_STOCK_DATE, CLOSING_STOCK AS CURRENT_AVAILABLE_STOCK 
#         FROM {stock_availability_table} WHERE DATE(CAL_DATE) = '{yday}' ;"""
#     logger.info(f'Fetching Dealer Inventory from {stock_availability_table} after {CAL_DATE}')
#     inventory_dealer = session.sql(query).to_pandas()
#     logger.info(f'Retrieved {len(inventory_dealer)} rows of Dealer Inventory data')
#     return inventory_dealer


def fetchDealerInventory(config):
    '''
    In this function we consider the stock availability of the previous OBD variant SKU as well
    to get the Stock Availability if Is_OBD is true
    '''
    CAL_DATE = config['CAL_DATE']
    stock_availability_table = config['database']['stock_availability_table']
    yday = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')
    query = f"""
    SELECT DEALER_CODE, SKU, DATE(CAL_DATE) as CLOSING_STOCK_DATE, CLOSING_STOCK AS CURRENT_AVAILABLE_STOCK 
    FROM {stock_availability_table} WHERE DATE(CAL_DATE) = '{yday}' ;"""

    inventory_dealer = session.sql(query).to_pandas()
 
    #if IS_OBD:
    obd_data = fetchOBDData(session)
    inventory_dealer = pd.merge(inventory_dealer, obd_data, on="SKU", how="left")
   
    # Null in Current OBD SKU means the current SKU is the same as SKU
    inventory_dealer['CURRENT_OBD_SKU'] = inventory_dealer['CURRENT_OBD_SKU'].fillna(inventory_dealer['SKU'])

    inventory_dealer = inventory_dealer.groupby(['CURRENT_OBD_SKU', 'DEALER_CODE','CLOSING_STOCK_DATE'])['CURRENT_AVAILABLE_STOCK'].sum().reset_index()
    inventory_dealer = inventory_dealer.rename(columns={'CURRENT_OBD_SKU': 'SKU'})
    return inventory_dealer
    

    

def aggregateInventory(inventory_dealer, parent_dealer_mapping):
    logger.info('Aggregating Inventory by Parent Dealer')
    inventory_dealer = pd.merge(inventory_dealer, parent_dealer_mapping[['DEALER_CODE', 'PARENT_DEALER_CODE']], on="DEALER_CODE", how="inner")
    aggregated_inventory = inventory_dealer.groupby(['PARENT_DEALER_CODE', 'SKU', 'CLOSING_STOCK_DATE'])['CURRENT_AVAILABLE_STOCK'].sum().reset_index()
    inventory_dealer = aggregated_inventory[['PARENT_DEALER_CODE', 'SKU', 'CLOSING_STOCK_DATE', 'CURRENT_AVAILABLE_STOCK']]
    inventory_dealer.columns = ['DEALER', 'SKU', 'CLOSING_STOCK_DATE', 'CURRENT_AVAILABLE_STOCK']
    logger.info(f'Aggregated Inventory contains {len(inventory_dealer)} rows')
    return inventory_dealer

def weekly_daily_aggregation(inventory_dealer, cut_off_date):
    logger.info(f'Performing weekly/daily aggregation with cut off date {cut_off_date}')
    inventory_dealer = inventory_dealer.loc[pd.to_datetime(inventory_dealer.CLOSING_STOCK_DATE) < cut_off_date]
    idx = inventory_dealer.groupby(['DEALER', 'SKU'])['CLOSING_STOCK_DATE'].idxmax()
    latest_inventory = inventory_dealer.loc[idx]
    latest_inventory = latest_inventory[['DEALER', 'SKU', 'CURRENT_AVAILABLE_STOCK','CLOSING_STOCK_DATE']]
    latest_inventory.columns = ['DEALER', 'SKU', 'CURRENT_AVAILABLE_STOCK','CLOSING_STOCK_DATE']
    logger.info(f'Final weekly/daily inventory aggregation contains {len(latest_inventory)} rows')
    return latest_inventory


def getECRDealerSales(config,start_date):
    logger.info(f'ECR Sales data fetching')
    customer_type_to_include = config['customer_types']
    agg_period = config['agg_type']
    concat_sim_models = config['combine_models']
    
    
    ecr_data = getDailyECR(customer_type_to_include, start_date, config)
    parent_dealer_mapping = getParentDealerMapping()
    
    ecr_sales = pd.merge(ecr_data, parent_dealer_mapping[['DEALER_CODE', 'PARENT_DEALER_CODE']], on="DEALER_CODE", how="inner")
    
    if agg_period == "monthly":
        group_by_cols = ['PARENT_DEALER_CODE', 'SKU',  'CAL_MONTH', 'CAL_YEAR'] if not concat_sim_models else ['PARENT_DEALER_CODE', 'SUB_MODEL', 'FAMILY_CODE', 'CAL_MONTH', 'CAL_YEAR']
        ecr_dealer_sales = ecr_sales.groupby(group_by_cols)['NET_SALES'].sum().reset_index()
        ecr_dealer_sales['Date'] = ecr_dealer_sales.apply(lambda row: datetime.datetime(row['CAL_YEAR'], row['CAL_MONTH'], 1), axis=1)
    else:
        group_by_cols = ['PARENT_DEALER_CODE', 'SKU', 'CAL_DATE', 'CAL_MONTH', 'CAL_YEAR'] if not concat_sim_models else ['PARENT_DEALER_CODE', 'SUB_MODEL', 'FAMILY_CODE', 'CAL_DATE', 'CAL_MONTH', 'CAL_YEAR']
        ecr_dealer_sales = ecr_sales.groupby(group_by_cols)['NET_SALES'].sum().reset_index()
        ecr_dealer_sales['Date'] = pd.to_datetime(ecr_dealer_sales['CAL_DATE'])
        ecr_dealer_sales['NET_SALES'] = np.where(ecr_dealer_sales.NET_SALES.fillna(0) <= 0, 0, ecr_dealer_sales.NET_SALES)
    logger.info(f'ECR data fetched and it contains {len(ecr_dealer_sales)} rows')
    return ecr_dealer_sales

def get_current_week_start_date(current_date):
    current_day = current_date.day
    current_month = current_date.month
    current_year = current_date.year
    week_number = (current_day - 1) // 7 + 1
    week_start_day = (week_number - 1) * 7 + 1
    #print(pd.Timestamp(current_year, current_month, week_start_day))
    return pd.Timestamp(current_year, current_month, week_start_day)

def calculate_reorder_urgency(final_data, ecr_dealer_sales, current_date,config):
    logger.info(f'Calculating weekly/daily reorder_urgency scores')
    current_date = pd.to_datetime(current_date)
    
    current_day = current_date.day
    current_month = current_date.month
    current_year = current_date.year
   
    if current_month == 12:
        last_day_of_month = pd.Timestamp(current_year, 12, 31)
    else:
        last_day_of_month = pd.Timestamp(current_year, current_month + 1, 1) - pd.Timedelta(days=1)

    days_remaining_in_month = (last_day_of_month - current_date).days + 1

    
    # Helper function to get days in the current week
    def get_days_in_week(current_day):
        if current_day <= 21:
            return 7
        else:
            return (last_day_of_month.day - current_day + 1)
    
    def get_remaining_days_in_custom_week(current_day, last_day_of_month):

        last_day_of_month_date = last_day_of_month.date()
   
        current_day_date = pd.Timestamp(last_day_of_month.year, last_day_of_month.month, current_day).date()
    
        
        if 1 <= current_day <= 7:
            return 7 - current_day + 1
        elif 8 <= current_day <= 14:
        
            return 14 - current_day + 1
        elif 15 <= current_day <= 21:
          
            return 21 - current_day + 1
        else:
            days_remaining = (last_day_of_month_date - current_day_date).days
            return days_remaining + 1
  
    def calculate_slope(sales_list):
      
        if isinstance(sales_list, (list, np.ndarray)) and len(sales_list) > 0:
            if not np.any(np.isnan(sales_list)):
                if len(sales_list) < 2:
                    return 0
                x = np.arange(len(sales_list))
                slope, _ = np.polyfit(x, sales_list, 1)
                return slope
            else:
                return 0  
        else:
            return 0  


 
    current_week_start_date = get_current_week_start_date(current_date)
    days_in_week = get_days_in_week(current_day)
    days_left_in_week = get_remaining_days_in_custom_week(current_day,last_day_of_month)
    
    print(days_in_week,"days_in_week")
    print(days_left_in_week, "days_left_in_week")
    print(last_day_of_month, "days in the month")
    

    # Merge final_data with ecr_sales based on DEALER and SKU, and date filters for sales up to the current day or week
    ecr_dealer_sales['CAL_DATE'] = pd.to_datetime(ecr_dealer_sales['CAL_DATE'])
    
    # Weekly sales (before current week start date)
    weekly_sales = ecr_dealer_sales[ecr_dealer_sales['CAL_DATE'] < current_week_start_date].groupby(['PARENT_DEALER_CODE', 'SKU'])['NET_SALES'].sum().reset_index()
    weekly_sales.columns = ['DEALER', 'SKU', 'total_sales_upto_previous_week']

    # Daily sales (before the current date)
    daily_sales = ecr_dealer_sales[ecr_dealer_sales['CAL_DATE'] < current_date].groupby(['PARENT_DEALER_CODE', 'SKU'])['NET_SALES'].sum().reset_index()
    daily_sales.columns = ['DEALER', 'SKU', 'total_sales_upto_previous_day']

    # Merge the sales data with final_data
    final_data = final_data.merge(weekly_sales, how='left', on=['DEALER', 'SKU']).fillna(0)
    final_data = final_data.merge(daily_sales, how='left', on=['DEALER', 'SKU']).fillna(0)

    # # Calculate remaining expected demand
    final_data['remaining_expected_demand_weekly'] = final_data['PREDICTED_SALES_SKU'].fillna(0) - final_data['total_sales_upto_previous_week'].fillna(0)
    final_data['remaining_expected_demand_daily'] = final_data['PREDICTED_SALES_SKU'].fillna(0) - final_data['total_sales_upto_previous_day'].fillna(0)

  
    print(last_day_of_month.day)
    final_data['PER_DAY_PREDICTED_SALES'] = final_data['PREDICTED_SALES_SKU'].fillna(0)/last_day_of_month.day
    final_data['SLOPE'] = final_data['LAST_15_SALES'].apply(calculate_slope)
    final_data['DEMAND_FACTOR_WEEKLY'] = 1 + final_data['LAST_15_SALES'].apply(calculate_slope).divide(
                                            final_data['PER_DAY_PREDICTED_SALES']).where(final_data['PER_DAY_PREDICTED_SALES'] > 0, 1)
    
    
    final_data['DEMAND_FACTOR_DAILY'] = 1 + final_data['LAST_15_SALES'].apply(calculate_slope).divide(
                                            final_data['PER_DAY_PREDICTED_SALES'].fillna(0)).where(final_data['PER_DAY_PREDICTED_SALES'].fillna(0) > 0, 1)

    final_data['expected_demand_in_day'] =  final_data['DEMAND_FACTOR_DAILY'].fillna(0) * final_data['PER_DAY_PREDICTED_SALES'].fillna(0)* days_left_in_week
    final_data['expected_demand_per_day'] =  final_data['DEMAND_FACTOR_DAILY'].fillna(0) * final_data['PER_DAY_PREDICTED_SALES'].fillna(0) 
    final_data['expected_demand_in_week'] =  final_data['expected_demand_in_day'].fillna(0)  * min(days_in_week,7)
    
    # final_data['remaining_expected_demand_weekly'] = final_data['expected_demand_in_day'] * max(days_in_week,7)
    # final_data['remaining_expected_demand_weekly'] = final_data['expected_demand_in_day'] * max(days_in_week,7)

    # Available stock after accounting for expected demand
    final_data['available_stock_weekly'] = final_data['CURRENT_WEEK_AVAILABLE_STOCK'].fillna(0) - final_data['expected_demand_in_week'].fillna(0)
    final_data['available_stock_daily'] = final_data['CURRENT_AVAILABLE_STOCK'].fillna(0) - final_data['expected_demand_in_day'].fillna(0)

    # Reorder urgency score calculation
    aplha =1
    final_data['Weekly_Reorder_level_urgency'] = np.where( final_data.SALES_PROPORTION.fillna(0) <=0,0,
                                                    (np.exp((final_data.REORDER_LEVEL.fillna(0) - final_data.available_stock_weekly.fillna(0) )
                                                  / (final_data.REORDER_LEVEL.fillna(0) + 1)) * (1 + (aplha * final_data.SALES_PROPORTION.fillna(0)) )))    
    final_data['Daily_Reorder_level_urgency'] = np.where(final_data.SALES_PROPORTION.fillna(0) <=0,0,
                                                    (np.exp((final_data.REORDER_LEVEL.fillna(0)- final_data.available_stock_daily.fillna(0) )
                                                  / (final_data.REORDER_LEVEL.fillna(0) + 1)) * (1 + (aplha * final_data.SALES_PROPORTION.fillna(0)) )))  
    
    
    final_data['Weekly_Reorder_level_urgency'] = final_data['Weekly_Reorder_level_urgency'].fillna(0)
    final_data['Daily_Reorder_level_urgency'] = final_data['Daily_Reorder_level_urgency'].fillna(0)
   
    weekly=final_data.drop(['CLOSING_STOCK_DATE','CURRENT_AVAILABLE_STOCK','Daily_Reorder_level_urgency','available_stock_daily','expected_demand_in_day','remaining_expected_demand_daily'],axis=1)
    daily = final_data.drop(['CLOSING_WEEK_STOCK_DATE','CURRENT_WEEK_AVAILABLE_STOCK','Weekly_Reorder_level_urgency','available_stock_weekly','expected_demand_in_week','remaining_expected_demand_weekly'],axis=1)

    weekly['WEEK_TILL_DATE']=current_week_start_date
    daily['TILL_DATE'] = current_date - timedelta(-1)
    
    logger.info(f'Successfully calculated weekly/daily reorder_urgency scores: Weekly table no of rows: {len(weekly)} Daily table no of rows: {len(daily)}')
    
    
    return weekly,daily

    

def rank_with_fallback(df, urgency_col, quantity_col):
    # Separate positive and non-positive quantity values
    positive_df = df[df[quantity_col] > 0].copy()
    non_positive_df = df[df[quantity_col] <= 0].copy()

    # Rank the positive quantity values first
    positive_df = positive_df.sort_values(by=[urgency_col, quantity_col], ascending=[False, False])
    positive_df['URGENCY_RANK'] = range(1, len(positive_df) + 1)

    # Rank the non-positive quantity values next
    non_positive_df = non_positive_df.sort_values(by=[urgency_col], ascending=[False])
    non_positive_df['URGENCY_RANK'] = range(len(positive_df) + 1, len(positive_df) + len(non_positive_df) + 1)

    # Combine both dataframes
    ranked_df = pd.concat([positive_df, non_positive_df])

    return ranked_df


# def LeadTimeSafetyStockDays():
#     query = f"""SELECT DISTINCT  PARENT_DEALER_CODE AS DEALER,SKU,PREDICTED_SALES_SKU,
#         SAFETY_STOCK, SAFETY_STOCK_DAYS
#         FROM MOP_DATABASE.SOQ.SOQ_DATA_FINAL_VERSION 
#         where 
#         STOCK_DATE_PERIOD='mid' 
#         AND PLANNING_MONTH='2025-02-01' 
#         AND DEMAND_VARIABILITY_TYPE='SKU_BASED'  
#         AND SERVICE_LEVEL=90"""

#     now = datetime.now()
#     next_month = now.replace(day=28) + timedelta(days=4)  # Guarantees we move to the next month
#     last_day = next_month.replace(day=1) - timedelta(days=1)
#     predicted_sales = session.sql(query).to_pandas()
#     Transit_time=fetchTransitTime()
#     predicted_sales=pd.merge(predicted_sales,Transit_time, on =['DEALER','SKU'], how='left')
#     predicted_sales['AVG_LEAD_TIME_STOCK'] = np.ceil((predicted_sales.PREDICTED_SALES_SKU/last_day.day) * predicted_sales.TRANSIT_TIME) 
#     predicted_sales['REORDER_LEVEL'] = predicted_sales.SAFETY_STOCK + predicted_sales.AVG_LEAD_TIME_STOCK
#     logger.info(f'Retrieved {predicted_sales.shape[0]} rows of Predicted Sales data')
#     return predicted_sales



def load_data(config, start_date):
    """Load all necessary data from the configuration and start date."""
    # ecr_dealer_sales = getECRDealerSales(config, start_date)
    parent_dealer_mapping = getParentDealerMapping()
    mop_data = fetchMOPData(config, parent_dealer_mapping)
    predicted_sales = fetchPredictedSales(config, "False")
    dealer_inventory = fetchDealerInventory(config)
    dispatch_data = fetchDispatchData(config, parent_dealer_mapping)
    today_dispatch_data= fetchTodayDispatchData(config, parent_dealer_mapping)
    
    return  parent_dealer_mapping, mop_data, predicted_sales, dealer_inventory, dispatch_data,today_dispatch_data

def preprocess_inventory(dealer_inventory, parent_dealer_mapping, current_date):
    """Preprocess inventory data for weekly and daily aggregation."""
    aggregated_inventory = aggregateInventory(dealer_inventory, parent_dealer_mapping)
    current_week_start_date = get_current_week_start_date(current_date)
    current_week_start_inventory = weekly_daily_aggregation(aggregated_inventory, current_week_start_date)
    current_available_inventory = weekly_daily_aggregation(aggregated_inventory, current_date)
    return current_week_start_inventory, current_available_inventory, current_week_start_date

def merge_inventory_with_sales(predicted_sales, current_week_start_inventory, current_available_inventory, current_week_start_date, current_date,today_dispatch_data):
    """Merge inventory data with predicted sales."""
    final_data = pd.merge(predicted_sales, current_week_start_inventory, on=['DEALER', 'SKU'], how='left')
    final_data['CURRENT_WEEK_AVAILABLE_STOCK'] = final_data.CURRENT_AVAILABLE_STOCK.fillna(0)
    final_data.drop(['CURRENT_AVAILABLE_STOCK'], axis=1, inplace=True)
    final_data['CLOSING_WEEK_STOCK_DATE'] = final_data.CLOSING_STOCK_DATE.fillna(current_week_start_date)
    final_data.drop(['CLOSING_STOCK_DATE'], axis=1, inplace=True)

    final_data = final_data.merge(current_available_inventory, on=['DEALER', 'SKU'], how='left')
    final_data =  final_data.merge(today_dispatch_data, on=['DEALER', 'SKU'], how='left' )
    final_data['QUANTITY_DISPATCHED_TODAY'] = final_data.QUANTITY_DISPATCHED_TODAY.fillna(0)
    final_data['CURRENT_AVAILABLE_STOCK'] = final_data.CURRENT_AVAILABLE_STOCK.fillna(0) + final_data.QUANTITY_DISPATCHED_TODAY.fillna(0)
    final_data['CLOSING_STOCK_DATE'] = final_data.CLOSING_STOCK_DATE.fillna(current_date)
    return final_data


def process_ecr_sales(ecr_sales_since_2021, current_date):
   
    # Add DEALSKU and ensure CAL_DATE is datetime
    # obd_data=fetchOBDData(session)
   
    # ecr_sales_since_2021=pd.merge(ecr_sales_since_2021,obd_data,on="SKU",how="left")
   
    #     ## Null in Current ObD SKU - means, current SKU is the same as sku
   
    # ecr_sales_since_2021['CURRENT_OBD_SKU'] = ecr_sales_since_2021['CURRENT_OBD_SKU'].fillna(ecr_sales_since_2021['SKU'])
   
    # ecr_sales_since_2021=ecr_sales_since_2021.drop(['SKU'],axis=1)
    # ecr_sales_since_2021=ecr_sales_since_2021.rename(columns={'CURRENT_OBD_SKU':'SKU'})
    
    ecr_sales_since_2021['DEALSKU'] = ecr_sales_since_2021.PARENT_DEALER_CODE.astype(str) + ecr_sales_since_2021.SKU.astype(str)
    ecr_sales_since_2021['CAL_DATE'] = pd.to_datetime(ecr_sales_since_2021['CAL_DATE'])
    
    # Filter sales data for the last 90 days
    dealer_total_sales = ecr_sales_since_2021.loc[
        ecr_sales_since_2021.CAL_DATE >= current_date - pd.Timedelta(days=90)
    ].copy()

    
    # Group by dealer to calculate total sales
    dealer_total_sales = ecr_sales_since_2021.groupby('PARENT_DEALER_CODE')['NET_SALES'].sum().reset_index()
    dealer_total_sales = dealer_total_sales.rename(columns={'NET_SALES': 'TOTAL_NET_SALES'})
    
    # Merge to calculate sales proportions
    ecr_sales_with_total = pd.merge(ecr_sales_since_2021, dealer_total_sales, on='PARENT_DEALER_CODE')
    ecr_sales_with_total['sales_proportion'] = np.where(ecr_sales_with_total['TOTAL_NET_SALES'].fillna(0)==0, 0, ecr_sales_with_total['NET_SALES'].fillna(0) / ecr_sales_with_total['TOTAL_NET_SALES'].fillna(0))
    
    # Aggregate last 3 years' sales and sales proportions
    last_3year_sales = ecr_sales_with_total.groupby(['PARENT_DEALER_CODE', 'SKU'])[
        ['NET_SALES', 'sales_proportion']
    ].sum().reset_index()
    
    # Rename columns
    last_3year_sales.columns = ['DEALER', 'SKU', 'LAST_3_YEAR_NET_SALES', 'LAST_3_MONTHS_SALES_PROPORTION']
    
    return last_3year_sales

# def process_last_10_days_sales(ecr_sales_since_2021, current_date):

#     # Filter sales data for the last 10 days
#     dealer_total_sales_10 = ecr_sales_since_2021.loc[
#         ecr_sales_since_2021.CAL_DATE >= current_date - pd.Timedelta(days=15)
#     ].copy()

#     # Group by dealer and SKU to get sales as a list
#     dealer_total_sales_10 = dealer_total_sales_10.groupby(['PARENT_DEALER_CODE', 'SKU'])['NET_SALES'].apply(
#         lambda x: list(x.fillna(0))
#     ).reset_index()
    
#     # Rename columns
#     dealer_total_sales_10.columns = ['DEALER', 'SKU', 'LAST_10_SALES']

#     return dealer_total_sales_10




def process_last_15_days_sales(ecr_sales_since_2021, current_date):
    dealer_total_sales_15 = ecr_sales_since_2021.loc[
        ecr_sales_since_2021.CAL_DATE >= current_date - pd.Timedelta(days=15)].copy()
    last_15_days = pd.date_range(end=dealer_total_sales_15['CAL_DATE'].max(), periods=15, freq='D')
    dealer_total_sales_15['CAL_DATE'] = pd.to_datetime(dealer_total_sales_15['CAL_DATE'])
    # Fill missing dates with 0 sales for each Dealer-SKU pair
    full_sales = (dealer_total_sales_15
                  .groupby(['PARENT_DEALER_CODE', 'SKU'])
                  .apply(lambda x: x.set_index('CAL_DATE')
                                    .reindex(last_15_days, fill_value=0)['NET_SALES']
                                    .tolist())
                  .reset_index())
    # Rename columns
    full_sales.columns = ['DEALER', 'SKU', 'LAST_15_SALES']

    return full_sales

def process_ecr_3_month_sales(ecr_sales_since_2021, current_date):

    import pandas as pd
    import numpy as np

    ecr_sales_since_2021 = ecr_sales_since_2021.copy()

    ecr_sales_since_2021['CAL_DATE'] = pd.to_datetime(ecr_sales_since_2021['CAL_DATE'])

    # ----------------------------
    # Filter last 90 days
    # ----------------------------
    dealer_sales = ecr_sales_since_2021.loc[
        ecr_sales_since_2021['CAL_DATE'] >= current_date - pd.Timedelta(days=90)
    ]

    # ----------------------------
    # Dealer total sales
    # ----------------------------
    dealer_total_sales = (
        dealer_sales
        .groupby('PARENT_DEALER_CODE', as_index=False)['NET_SALES']
        .sum()
        .rename(columns={'NET_SALES': 'TOTAL_NET_SALES'})
    )

    # ----------------------------
    # Merge totals
    # ----------------------------
    dealer_sales = dealer_sales.merge(
        dealer_total_sales,
        on='PARENT_DEALER_CODE',
        how='left'
    )

    dealer_sales['sales_proportion'] = np.where(
        dealer_sales['TOTAL_NET_SALES'] == 0,
        0,
        dealer_sales['NET_SALES'] / dealer_sales['TOTAL_NET_SALES']
    )

    # ----------------------------
    # Aggregate at Dealer–SKU level
    # ----------------------------
    last_3_month_sales = (
        dealer_sales
        .groupby(['PARENT_DEALER_CODE', 'SKU'], as_index=False)
        .agg(
            LAST_3_MONTH_NET_SALES=('NET_SALES', 'sum'),
            SALES_PROPORTION=('sales_proportion', 'sum')
        )
    )

    last_3_month_sales = last_3_month_sales.rename(
        columns={'PARENT_DEALER_CODE': 'DEALER'}
    )

    # ----------------------------
    # Sort & cumulative proportion at Dealer level
    # ----------------------------
    last_3_month_sales = (
        last_3_month_sales
        .sort_values(['DEALER', 'LAST_3_MONTH_NET_SALES'], ascending=[True, False])
    )

    last_3_month_sales['CUMULATIVE_PROPORTION'] = (
        last_3_month_sales
        .groupby('DEALER')['SALES_PROPORTION']
        .cumsum()
    )

    # ----------------------------
    # Classification logic
    # ----------------------------
    def classify_sku(group):
        group = group.copy()

        # Default classification
        group['SKU_CATEGORY'] = np.where(
            group['CUMULATIVE_PROPORTION'] <= 0.80, 'RUNNER',
            np.where(group['CUMULATIVE_PROPORTION'] <= 0.90, 'REPEATER', 'STRANGER')
        )

        # Business override:
        # Ensure at least one RUNNER and one REPEATER if first SKUs cross 90%
        if len(group) >= 1:
            group.iloc[0, group.columns.get_loc('SKU_CATEGORY')] = 'RUNNER'

        if len(group) >= 2 and group.iloc[1]['CUMULATIVE_PROPORTION'] > 0.80:
            group.iloc[1, group.columns.get_loc('SKU_CATEGORY')] = 'REPEATER'

        return group

    last_3_month_sales = (
        last_3_month_sales
        .groupby('DEALER', group_keys=False)
        .apply(classify_sku)
    )

    return last_3_month_sales


def main():
    
    # Load configuration
    #config = load_config()
    current_date = pd.Timestamp.now()
    weekly_table = config['database']['Weekly_table']
    daily_table =  config['database']['Daily_table']
    Unconstrained_table =  config['database']['Unconstrained_table']
    ZeroB2G_table =   config['database']["ZeroB2G_table"]
    DailyDispatchTable = config['database']["DailyDispatchTable"]
    DailyLeadTimeTable =  config['database']['DailyLeadTimeTable']
    runner_skus_table =  config['database']['runner_skus_table']
    # Fetch data based on the config
    start_date = config['start_date']
 
    parent_dealer_mapping, mop_data, predicted_sales, dealer_inventory, dispatch_data, today_dispatch_data = load_data(config, start_date)
    
    predicted_sales = pd.merge(mop_data[['DEALER','SKU','PLANT']].copy(),predicted_sales, on =['DEALER','SKU','PLANT'], how='left')
    
    #predicted_sales.loc[(predicted_sales.DEALER=='10258') & (predicted_sales.SKU=='HPLNPDRLMFIMVG')]
    current_week_start_inventory, current_available_inventory, current_week_start_date = preprocess_inventory(dealer_inventory, parent_dealer_mapping, current_date)    
   
    print(predicted_sales.shape, "before merging with daily & week starte date Inventory")

    final_data = merge_inventory_with_sales(predicted_sales, current_week_start_inventory, current_available_inventory, current_week_start_date, current_date,today_dispatch_data)
    
    print(final_data.shape, "after merging  with daily & week start date Inventory")
    open_orders = open_order_block(session)
    
    start_date = '2023-05-01'
    ecr_sales_since_2021 = getECRDealerSales(config, start_date)
    
    # Process ECR sales
    last_3year_sales = process_ecr_sales(ecr_sales_since_2021, current_date)
    
    # Merge with final data
    final_data = pd.merge(final_data, last_3year_sales, on=['DEALER', 'SKU'], how='left')
    #final_data.loc[(final_data.DEALER=='11845') & (final_data.SKU == 'HSPPPDRSCFIBKG'),:]
    print(final_data.shape,"after merging  with sales data for last 3 year sales ")

    last_10_sales = process_last_15_days_sales(ecr_sales_since_2021, current_date)

    # Merge with final data
    final_data = pd.merge(final_data, last_10_sales, on=['DEALER', 'SKU'], how='left')
    print(final_data.shape,"after merging  with sales data for last 10 day sales ")
        
    final_data['SALES_PROPORTION']=final_data.LAST_3_MONTHS_SALES_PROPORTION.fillna(0)

    aplha =1
    
    final_data['REORDER_URGENCY_SCORE'] = np.where(final_data.SALES_PROPORTION.fillna(0)==0,0,(np.exp((final_data.REORDER_LEVEL.fillna(0)-final_data.CURRENT_AVAILABLE_STOCK.fillna(0))
                                                  / (final_data.REORDER_LEVEL.fillna(0) + 1)) * (1 + (aplha * final_data.SALES_PROPORTION.fillna(0) ))))
    
    #final_data.loc[(final_data.DEALER=='11845') & (final_data.SKU == 'HSPPPDRSCFIBKG'),:]
    weekly,daily = calculate_reorder_urgency(final_data, ecr_sales_since_2021, current_date,config)
    
    MOP_Weekly=pd.merge(mop_data[['DEALER','SKU','PLANT','QUANTITY','VERSION','MODEL']],weekly, on =['DEALER','SKU','PLANT'], how ='left')
    MOP_Weekly=pd.merge(MOP_Weekly,dispatch_data, on =['DEALER','SKU','PLANT'], how ='left')
    MOP_Weekly['B2G'] = MOP_Weekly.QUANTITY - MOP_Weekly.QUANTITY_DISPATCHED.fillna(0)
    
    MOP_Daily=pd.merge(mop_data[['DEALER','SKU','PLANT','QUANTITY','VERSION','MODEL']],daily, on =['DEALER','SKU','PLANT'], how ='left')
    
    MOP_Daily=pd.merge(MOP_Daily,dispatch_data, on =['DEALER','SKU','PLANT'], how ='left')
    MOP_Daily = pd.merge(MOP_Daily, open_orders, on =['PLANT','DEALER','SKU'], how='left') 
    MOP_Daily['B2G'] = MOP_Daily.QUANTITY - MOP_Daily.QUANTITY_DISPATCHED.fillna(0) - MOP_Daily.TOTAL_OPEN_ORDER_QTY.fillna(0)
    

    
    MOP_Weekly['RUN_DATE'] = pd.Timestamp.now().date()
    MOP_Daily['RUN_DATE']  = pd.Timestamp.now().date()

    MOP_Weekly['WEEK_START_DATE']=current_week_start_date
    MOP_Daily['CURRENT_DATE'] = MOP_Daily['RUN_DATE']
 
    MOP_Weekly = rank_with_fallback(MOP_Weekly, 'Weekly_Reorder_level_urgency', 'B2G')
    MOP_Daily = rank_with_fallback(MOP_Daily, 'Daily_Reorder_level_urgency', 'B2G')   

    current_year = current_date.year
    current_month = current_date.month

    # Filter for sales data for the current month
    ecr_sales_last_7_days = ecr_sales_since_2021.loc[ecr_sales_since_2021.CAL_DATE >= current_date - pd.Timedelta(days=15),: ].copy()
    ecr_sales_last_7_days = ecr_sales_last_7_days.groupby(['PARENT_DEALER_CODE', 'SKU'])['NET_SALES'].sum().reset_index()
    ecr_sales_last_7_days.columns = ['DEALER','SKU','LAST_N_DAY_SALES']
    ecr_sales_last_7_days.LAST_N_DAY_SALES = np.where(ecr_sales_last_7_days.LAST_N_DAY_SALES.fillna(0) <= 0 , 0,ecr_sales_last_7_days.LAST_N_DAY_SALES.fillna(0) )

    ecr_sales_current_month = ecr_sales_since_2021.loc[
        (ecr_sales_since_2021['CAL_DATE'].dt.year == current_year) &
        (ecr_sales_since_2021['CAL_DATE'].dt.month == current_month), :].copy()
    ecr_sales_current_month = ecr_sales_current_month.groupby(['PARENT_DEALER_CODE', 'SKU'])['NET_SALES'].sum().reset_index()
    ecr_sales_current_month.columns = ['DEALER','SKU','CURRENT_MONTH_SALES']
    ecr_sales_current_month.CURRENT_MONTH_SALES = np.where(ecr_sales_current_month.CURRENT_MONTH_SALES <= 0 , 0,ecr_sales_current_month.CURRENT_MONTH_SALES )

    MOP_Weekly = pd.merge(MOP_Weekly,ecr_sales_last_7_days, on =['DEALER','SKU'], how='left')
    MOP_Daily = pd.merge(MOP_Daily,ecr_sales_last_7_days, on =['DEALER','SKU'], how='left')
    MOP_Weekly.LAST_N_DAY_SALES.fillna(0,inplace=True)
    MOP_Daily.LAST_N_DAY_SALES.fillna(0,inplace=True)

    MOP_Weekly = pd.merge(MOP_Weekly,ecr_sales_current_month, on =['DEALER','SKU'], how='left')
    MOP_Daily = pd.merge(MOP_Daily,ecr_sales_current_month, on =['DEALER','SKU'], how='left')
    MOP_Weekly.CURRENT_MONTH_SALES.fillna(0,inplace=True)
    MOP_Daily.CURRENT_MONTH_SALES.fillna(0,inplace=True)
    
    MOP_Weekly['DEALSKU'] = MOP_Weekly.DEALER.astype(str) + MOP_Weekly.SKU.astype(str)
    MOP_Weekly['SALES_PRESENCE_FLAG'] = np.where(MOP_Weekly.DEALSKU.isin(ecr_sales_since_2021.DEALSKU),1,0)
    
    MOP_Daily['DEALSKU'] = MOP_Daily.DEALER.astype(str) + MOP_Daily.SKU.astype(str)
    MOP_Daily['SALES_PRESENCE_FLAG'] = np.where(MOP_Daily.DEALSKU.isin(ecr_sales_since_2021.DEALSKU),1,0)

    MOP_Daily['SAFETY_STOCK'] =  np.minimum(MOP_Daily['SAFETY_STOCK'].fillna(0),(MOP_Daily['PREDICTED_SALES_SKU'].fillna(0) * 3))
                        
    
    MOP_Weekly= MOP_Weekly[["RUN_DATE","DEALER","SKU",'PLANT',"QUANTITY","MODEL","VERSION","PREDICTED_SALES_SKU","AVG_LEAD_TIME_STOCK","SAFETY_STOCK",
                            "REORDER_LEVEL","CURRENT_WEEK_AVAILABLE_STOCK","REORDER_URGENCY_SCORE", "WEEK_TILL_DATE",
                            "total_sales_upto_previous_week","total_sales_upto_previous_day","PER_DAY_PREDICTED_SALES", "DEMAND_FACTOR_WEEKLY",
                            "expected_demand_in_week","available_stock_weekly","Weekly_Reorder_level_urgency" ,"SALES_PRESENCE_FLAG","WEEK_START_DATE","SALES_PROPORTION","CURRENT_MONTH_SALES","LAST_N_DAY_SALES",
                            "QUANTITY_DISPATCHED","B2G","URGENCY_RANK"]]
    MOP_Daily = MOP_Daily[["RUN_DATE","DEALER","SKU",'PLANT',"QUANTITY","MODEL","VERSION","PREDICTED_SALES_SKU","AVG_LEAD_TIME_STOCK","SAFETY_STOCK",
                            "REORDER_LEVEL","CURRENT_AVAILABLE_STOCK", "REORDER_URGENCY_SCORE", "TILL_DATE",
                            "total_sales_upto_previous_week","total_sales_upto_previous_day","PER_DAY_PREDICTED_SALES",  "SLOPE" , 'DEMAND_FACTOR_DAILY',
                            "expected_demand_per_day","expected_demand_in_day","available_stock_daily","Daily_Reorder_level_urgency", "SALES_PRESENCE_FLAG","SALES_PROPORTION","CURRENT_MONTH_SALES","LAST_N_DAY_SALES",
                            "QUANTITY_DISPATCHED","TOTAL_OPEN_ORDER_QTY","B2G","URGENCY_RANK","QUANTITY_DISPATCHED_TODAY"]]
    
    MOP_Weekly[["REORDER_URGENCY_SCORE", "total_sales_upto_previous_week",
                "total_sales_upto_previous_day","PER_DAY_PREDICTED_SALES",
                "expected_demand_in_week","available_stock_weekly",
                "Weekly_Reorder_level_urgency"]]= MOP_Weekly[["REORDER_URGENCY_SCORE", "total_sales_upto_previous_week","total_sales_upto_previous_day",
                                                              "PER_DAY_PREDICTED_SALES","expected_demand_in_week","available_stock_weekly",
                                                              "Weekly_Reorder_level_urgency"]].apply(lambda x: round(x,3))

    MOP_Daily[["REORDER_URGENCY_SCORE", "total_sales_upto_previous_week",
               "total_sales_upto_previous_day","PER_DAY_PREDICTED_SALES",
              "expected_demand_in_day","available_stock_daily",
               "Daily_Reorder_level_urgency","expected_demand_per_day","SLOPE"]] = MOP_Daily[["REORDER_URGENCY_SCORE", "total_sales_upto_previous_week",
                                                            "total_sales_upto_previous_day","PER_DAY_PREDICTED_SALES",
                                                            "expected_demand_in_day","available_stock_daily",
                                                            "Daily_Reorder_level_urgency","expected_demand_per_day","SLOPE"]].apply(lambda x: round(x,3))

    
    MOP_Daily.QUANTITY_DISPATCHED.fillna(0,inplace=True)
    MOP_Daily.TOTAL_OPEN_ORDER_QTY.fillna(0,inplace=True)
    
    # MOP_Daily.B2G =  np.where(MOP_Daily.B2G < 0, 0, MOP_Daily.B2G)
    # MOP_Weekly.B2G =  np.where(MOP_Weekly.B2G < 0, 0, MOP_Daily.B2G)
    MOP_Daily.QUANTITY_DISPATCHED_TODAY.fillna(0)
   
    MOP_Daily['RUN_CYCLE_ID'] = cycle_id
    MOP_Daily['RUN_CYCLE_TIME'] = current_time
    MOP_Daily['QUANTITY_DISPATCHED_AS_OF_TODAY'] = MOP_Daily.QUANTITY_DISPATCHED_TODAY.fillna(0)
    
    #MOP_Daily.drop('QUANTITY_DISPATCHED_TODAY',axis=1,inplace=True)
    

    MOP_Weekly['RUN_CYCLE_ID'] = cycle_id
    MOP_Weekly['RUN_CYCLE_TIME'] = current_time
    
    # MOP_Daily.B2G =  np.where(MOP_Daily.B2G < 0, 0, MOP_Daily.B2G)
    # MOP_Weekly.B2G =  np.where(MOP_Weekly.B2G < 0, 0, MOP_Daily.B2G)
    
    dispatch_data['RUN_DATE'] = pd.Timestamp.now().date()
    dispatch_data['RUN_CYCLE_ID'] = cycle_id
    dispatch_data['RUN_CYCLE_TIME'] = current_time
    
    Unconstrained_file = MOP_Daily[['RUN_DATE','RUN_CYCLE_ID', 'RUN_CYCLE_TIME','DEALER','SKU','PLANT','QUANTITY','QUANTITY_DISPATCHED','QUANTITY_DISPATCHED_TODAY','TOTAL_OPEN_ORDER_QTY','B2G']].fillna(0).groupby(['RUN_DATE','RUN_CYCLE_ID', 'RUN_CYCLE_TIME','DEALER','SKU','PLANT'])[['QUANTITY','QUANTITY_DISPATCHED','QUANTITY_DISPATCHED_TODAY','TOTAL_OPEN_ORDER_QTY','B2G']].sum().reset_index()
    MOP_Daily.drop('QUANTITY_DISPATCHED_TODAY',axis=1,inplace=True)
    Unconstrained_file.columns = ['RUN_DATE', 'RUN_CYCLE_ID', 'RUN_CYCLE_TIME','DEALER', 'SKU','PLANT', 'MOP_QUANTITY', 'QUANTITY_DISPATCHED', 'QUANTITY_DISPATCHED_TODAY','TOTAL_OPEN_ORDER_QTY', 'B2G']
    dispatch_data['IS_PART_OF_PLAN'] = np.where((dispatch_data.PLANT.astype(str) + dispatch_data.DEALER.astype(str) + dispatch_data.SKU.astype(str)).isin(MOP_Daily.PLANT.astype(str) + MOP_Daily.DEALER.astype(str) + MOP_Daily.SKU.astype(str)),1,0)
    
    # LeadTimetable=LeadTimeSafetyStockDays()
    
    LeadTimeTable=pd.merge(mop_data[['DEALER','SKU','PLANT']],predicted_sales[[ 'DEALER',	'SKU'	,'PLANT',	'PREDICTED_SALES_SKU',	'SAFETY_STOCK',	'TRANSIT_TIME',	'AVG_LEAD_TIME_STOCK',	'REORDER_LEVEL','SAFETY_STOCK_DAYS']], on=['DEALER','SKU','PLANT'], how='left')
    
    

    LeadTimeTable['RUN_DATE'] = pd.Timestamp.now().date() 
    LeadTimeTable['RUN_CYCLE_ID'] = cycle_id
    LeadTimeTable['RUN_CYCLE_TIME'] = current_time
    LeadTimeTable = LeadTimeTable[[  'DEALER',	'SKU'	,'PLANT',	'PREDICTED_SALES_SKU',	'SAFETY_STOCK',	'TRANSIT_TIME',	'AVG_LEAD_TIME_STOCK',	'REORDER_LEVEL',	'RUN_DATE',	'RUN_CYCLE_ID',	'RUN_CYCLE_TIME',	'SAFETY_STOCK_DAYS']].copy()

    Runner_skus=process_ecr_3_month_sales(ecr_sales_since_2021, current_date)

    Runner_skus['RUN_DATE'] = pd.Timestamp.now().date() 
    Runner_skus['RUN_CYCLE_ID'] = cycle_id
    Runner_skus['RUN_CYCLE_TIME'] = current_time
    
    delete_insert(DailyLeadTimeTable,cycle_id,current_date=pd.Timestamp.now().date())
    delete_insert(DailyDispatchTable,cycle_id,current_date=pd.Timestamp.now().date())
    delete_insert(weekly_table,cycle_id, current_date=pd.Timestamp.now().date())
    delete_insert(daily_table,cycle_id,current_date=pd.Timestamp.now().date())
    delete_insert(Unconstrained_table,cycle_id, current_date=pd.Timestamp.now().date())
    delete_insert(ZeroB2G_table,cycle_id,current_date=pd.Timestamp.now().date())
    delete_insert( runner_skus_table,cycle_id,current_date=pd.Timestamp.now().date())

# weekly_table = config['database']['Weekly_table']
# daily_table =  config['database']['Daily_table']
# Unconstrained_table =  config['database']['Unconstrained_table']
# ZeroB2G_table =   config['database']["ZeroB2G_table"]

    # drop_table(DailyDispatchTable)
    # drop_table(weekly_table)
    # drop_table(daily_table)
    # drop_table(Unconstrained_table)
    # drop_table(ZeroB2G_table)

        
    
    write_to_snowflake(LeadTimeTable ,DailyLeadTimeTable)
    if Unconstrained_file.loc[Unconstrained_file.B2G <= 0,:].shape[0] > 0 :
        write_to_snowflake(Unconstrained_file.loc[Unconstrained_file.B2G <= 0,:].drop(['QUANTITY_DISPATCHED_TODAY','PLANT'],axis=1) ,ZeroB2G_table)

    #Unconstrained_file=Unconstrained_file.loc[Unconstrained_file.B2G > 0,:].copy()
    write_to_snowflake(Runner_skus,runner_skus_table)
    write_to_snowflake(MOP_Weekly,weekly_table)
    write_to_snowflake(MOP_Daily,daily_table)
    write_to_snowflake(Unconstrained_file,Unconstrained_table)
    if dispatch_data.shape[0] > 0 :
        write_to_snowflake(dispatch_data,DailyDispatchTable)
   
    return MOP_Weekly, MOP_Daily,dispatch_data, Unconstrained_file,LeadTimeTable,Runner_skus
    #return final_data
    
if __name__ == "__main__":
    MOP_Weekly, MOP_Daily,dispatch_data,Unconstrained_file,LeadTimeTable,Runner_skus = main()
    #final_data = main()

time.sleep(20)

In [ ]:
# #################################### STEP2 ##################################################
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
from snowflake.snowpark.context import get_active_session
import logging
import time

pd.options.mode.chained_assignment = None

# ===================== LOGGER =====================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

session = get_active_session()

# ===================== RUN CYCLE =====================
def get_run_cycle():
    current_time = datetime.now().time()
    if current_time < datetime.strptime("11:00:00", "%H:%M:%S").time():
        return 1, current_time
    elif current_time < datetime.strptime("14:00:00", "%H:%M:%S").time():
        return 2, current_time
    elif current_time < datetime.strptime("16:00:00", "%H:%M:%S").time():
        return 3, current_time
    elif current_time < datetime.strptime("18:00:00", "%H:%M:%S").time():
        return 4, current_time
    else:
        return 5, current_time

cycle_id, current_time = get_run_cycle()

# ===================== CONFIG =====================
config = {
    "CALDAY": 20261001,
    "Blocked_status_table": "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_block_status_table",
    "Unblocked_table": "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_unblocked_table",
    "FundBlocked": "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.FundBlocked",
    "DemandBlocked": "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DemandBlocked",
    "OpenOrdersBlocked": "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.OpenOrdersBlocked",
    "PlantBlockedStats": "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.PlantBlockedStats",
    "OverallBlockedStats": "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.OverallBlockedStats",
    "is_fund_dynamic": False
}

# ===================== PREMIA BLOCK CONFIG =====================
PREMIA_DEALER_MIN = 17000
PREMIA_DEALER_MAX = 18000

PREMIA_SKU_KEYWORDS = [
    'KZMCDDSCF','X25ADDSCF','XOOFDSSCF','XPLBBDSSF','XPLBTDSSF',
    'KZMBDDSCF','X25ACDSCF','X44BTDSCF','XPLGCDSSF','XPLGDDSSF','XPLGHDSSF'
]

# ===================== COMMON UTILS =====================
def delete_insert(table_name, cycle_id, run_date):
    session.sql(
        f"""
        DELETE FROM {table_name}
        WHERE RUN_DATE = '{run_date}'
        AND RUN_CYCLE_ID = {cycle_id};
        """
    ).collect()

def write_to_snowflake(df, table_name):
    session.create_dataframe(df).write.mode("append").save_as_table(table_name)

def fetch_data(query):
    return session.sql(query).to_pandas()

# ===================== DATA LOAD =====================


    
def process_prioritisation_df():
    query = """
    SELECT *
    FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation
    WHERE (run_date, run_cycle_id) IN (
        SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation
        WHERE run_date = (SELECT MAX(run_date) FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation)
        GROUP BY run_date
    );
    """
    df = fetch_data(query)
    df['QUANTITY_DISPATCHED'] = df['QUANTITY_DISPATCHED'].fillna(0)
    df['DEALER'] = df['DEALER'].astype(str)
    return df

def process_demand_blocks():
    query = """
    SELECT DEALER_CODE, PLANT_HHHD, PLANT_HHHG, PLANT_HHHU, PLANT_HM4N, PLANT_HM5V, PLANT_HM6C
    FROM SAP_DATA_DATABASE.SAP_DATA.DEMAND_BLOCKERS_OPEN
    WHERE DATE IN (SELECT MAX(DATE) FROM SAP_DATA_DATABASE.SAP_DATA.DEMAND_BLOCKERS_OPEN)
    AND LEN(DEALER_CODE) = 5;
    """
    df = fetch_data(query)
    plant_cols = [c for c in df.columns if c.startswith('PLANT_')]
    df.rename(columns={'DEALER_CODE': 'DEALER'}, inplace=True)
    df['DEALER'] = df['DEALER'].astype(str)

    df = df.groupby('DEALER').agg(
        {c: lambda x: 'Blocked' if 'Blocked' in x.values else 'Not Blocked' for c in plant_cols}
    ).reset_index()

    df = df.melt(id_vars=['DEALER'], value_vars=plant_cols,
                 var_name='PLANT', value_name='DEMAND_BLOCK_STATUS')
    df['PLANT'] = df['PLANT'].str.replace('PLANT_', '')
    return df

def process_fund_blocks():
    query = """
    SELECT *
    FROM SAP_DATA_DATABASE.SAP_DATA.FUND_AVAILABILITY_HOURLY_DATA
    WHERE REFRESH_DATE IN (
        SELECT MAX(REFRESH_DATE) FROM SAP_DATA_DATABASE.SAP_DATA.FUND_AVAILABILITY_HOURLY_DATA
    );
    """
    df = fetch_data(query)
    df['DEALER'] = df['DEALER'].astype(str)
    df['OBL_CR_LIMIT_AVAIL'] = pd.to_numeric(df['OBL_CR_LIMIT_AVAIL'], errors='coerce').fillna(0)
    df['FUND_BLOCK_STATUS'] = np.where(df['OBL_CR_LIMIT_AVAIL'] < 1_000_000, 'Blocked', 'Not Blocked')
    return df[['DEALER', 'OBL_CR_LIMIT_AVAIL', 'FUND_BLOCK_STATUS']]

def open_order_block():
    query = f"""
    SELECT PLANT, DEALER_CODE AS DEALER, SKU, SUM(TOTAL_OPEN_ORDER_QTY) AS TOTAL_OPEN_ORDER_QTY
    FROM SAP_DATA_DATABASE.SAP_DATA.VW_OPEN_ORDERS_LIVE
    WHERE TOTAL_OPEN_ORDER_QTY > 0
    AND OPEN_ORDER_DATE >= {config['CALDAY']}
    GROUP BY 1,2,3;
    """
    try:
        df = fetch_data(query)
        df['DEALER'] = df['DEALER'].astype(str)
        df['OPEN_ORDER_BLOCK_STATUS'] = 'Blocked'
        return df
    except:
        return pd.DataFrame(columns=['PLANT','DEALER','SKU','TOTAL_OPEN_ORDER_QTY','OPEN_ORDER_BLOCK_STATUS'])

def fetch_runner_skus():
    query = f"""
    SELECT *
    FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.RUNNER_SKU_TABLE
    WHERE (run_date, run_cycle_id) IN (
        SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.RUNNER_SKU_TABLE
        WHERE run_date = (SELECT MAX(run_date) FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.RUNNER_SKU_TABLE)
        GROUP BY run_date
    );
    """
    df = fetch_data(query)

    df['DEALER'] = df['DEALER'].astype(str)
    return df
    
# ===================== MERGE =====================
def merge_data(prioritisation_df, demand_blocks, fund_blocks, open_orders):
    df = prioritisation_df.merge(
        demand_blocks, on=['DEALER','PLANT'], how='left'
    )

    df = df.merge(
        fund_blocks[['DEALER','FUND_BLOCK_STATUS']], on='DEALER', how='left'
    )

    df = df.merge(
        open_orders[['DEALER','PLANT','SKU','OPEN_ORDER_BLOCK_STATUS']],
        on=['DEALER','PLANT','SKU'], how='left'
    )

    df['DEMAND_BLOCK_STATUS'] = df['DEMAND_BLOCK_STATUS'].fillna('Not Blocked')
    df['FUND_BLOCK_STATUS'] = df['FUND_BLOCK_STATUS'].fillna('Not Blocked')
    df['OPEN_ORDER_BLOCK_STATUS'] = df['OPEN_ORDER_BLOCK_STATUS'].fillna('Not Blocked')

    return df


# ===================== MAIN =====================
def main():
    run_date = pd.Timestamp.now().date()
    
    Blocked_status_table = config['Blocked_status_table']
    Unblocked_table =  config['Unblocked_table']
    FundBlocked = config['FundBlocked']
    DemandBlocked =  config['DemandBlocked']
    OpenOrdersBlocked = config['OpenOrdersBlocked']  
    PlantBlockedStats = config['PlantBlockedStats'] 
    OverallBlockedStats = config['OverallBlockedStats'] 
    is_fund_dynamic = config['is_fund_dynamic']
    
    prioritisation_df = process_prioritisation_df()
    demand_blocks = process_demand_blocks()
    fund_blocks = process_fund_blocks()
    open_order_df = open_order_block()
    runner_skus = fetch_runner_skus()
    runner_skus
    

    blocked_df = merge_data(prioritisation_df, demand_blocks, fund_blocks, open_order_df)

    # ================= PREMIA DEALER SKU BLOCK =================
    blocked_df['DEALER_INT'] = pd.to_numeric(blocked_df['DEALER'], errors='coerce')

    
    premia_mask = (
        blocked_df['DEALER_INT'].between(PREMIA_DEALER_MIN, PREMIA_DEALER_MAX) &
        blocked_df['SKU'].astype(str).apply(lambda x: any(k in x for k in PREMIA_SKU_KEYWORDS))
    )

    blocked_df.loc[premia_mask, [
        'DEMAND_BLOCK_STATUS','FUND_BLOCK_STATUS','OPEN_ORDER_BLOCK_STATUS'
    ]] = 'Blocked'

    blocked_df.drop(columns=['DEALER_INT'], inplace=True)

    blocked_df['RUN_DATE'] = run_date
    blocked_df['RUN_CYCLE_ID'] = cycle_id
    blocked_df['RUN_CYCLE_TIME'] = current_time

    unblocked_df = blocked_df.loc[
        (blocked_df.DEMAND_BLOCK_STATUS == 'Not Blocked') &
        (blocked_df.FUND_BLOCK_STATUS == 'Not Blocked') &
        (blocked_df.OPEN_ORDER_BLOCK_STATUS == 'Not Blocked')
    ].drop(
        ['DEMAND_BLOCK_STATUS','FUND_BLOCK_STATUS','OPEN_ORDER_BLOCK_STATUS'],
        axis=1
    )

   
    runner_skus= runner_skus.loc[runner_skus.SKU_CATEGORY=='RUNNER',:]
    runner_skus['DEALER_SKU'] = runner_skus.DEALER.astype(str) + runner_skus.SKU
    unblocked_df['DEALER_SKU'] = unblocked_df.DEALER.astype(str) + unblocked_df.SKU
    #unblocked_df = unblocked_df.loc[unblocked_df.DEALER_SKU.isin(runner_skus.DEALER_SKU)]
    unblocked_df.drop(['DEALER_SKU'],axis=1,inplace=True)
    
    unblocked_df['RUN_DATE'] = run_date
    unblocked_df['RUN_CYCLE_ID'] = cycle_id
    unblocked_df['RUN_CYCLE_TIME'] = current_time

    demand_blocks['RUN_DATE'] = pd.Timestamp.now().date()
    demand_blocks['RUN_CYCLE_ID'] = cycle_id
    demand_blocks['RUN_CYCLE_TIME'] = current_time
    
    fund_blocks['RUN_DATE'] = pd.Timestamp.now().date()
    fund_blocks['RUN_CYCLE_ID'] = cycle_id
    fund_blocks['RUN_CYCLE_TIME'] = current_time
    
    open_order_df['RUN_DATE'] = pd.Timestamp.now().date()
    open_order_df['RUN_CYCLE_ID'] = cycle_id
    open_order_df['RUN_CYCLE_TIME'] = current_time


    # ================= DELETE + INSERT =================
    delete_insert(config['Blocked_status_table'], cycle_id, run_date)
    delete_insert(config['Unblocked_table'], cycle_id, run_date)
    delete_insert(DemandBlocked,cycle_id,run_date=pd.Timestamp.now().date())
    delete_insert(FundBlocked,cycle_id,run_date=pd.Timestamp.now().date())
    delete_insert(OpenOrdersBlocked,cycle_id,run_date=pd.Timestamp.now().date())
    

    write_to_snowflake(blocked_df, config['Blocked_status_table'])
    write_to_snowflake(unblocked_df, config['Unblocked_table'])
    write_to_snowflake(open_order_df,OpenOrdersBlocked )
    write_to_snowflake(demand_blocks,DemandBlocked)
    write_to_snowflake(fund_blocks,FundBlocked)

    return blocked_df, unblocked_df

# ===================== RUN =====================
final_blocked_df, final_unblocked_df = main()
time.sleep(20)


In [ ]:
session.sql("USE ROLE DS1_ROLE").collect()
session.sql("USE WAREHOUSE DS_SPO_WH").collect()

In [ ]:
# #################################### STEP 2A ##################################################
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
from snowflake.snowpark.context import get_active_session
import logging

pd.options.mode.chained_assignment = None

# ------------------------------------------------
# Logger & Snowflake Session
# ------------------------------------------------
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
session = get_active_session()

# ------------------------------------------------
# Utility Functions
# ------------------------------------------------
def drop_table(table_name):
    logger.info(f"Dropping table: {table_name}")
    session.sql(f"DROP TABLE IF EXISTS {table_name}").collect()

def write_to_snowflake(df, table_name, mode="append"):
    logger.info(f"Writing to Snowflake table: {table_name}")
    session.create_dataframe(df).write.mode(mode).save_as_table(table_name)

def fetch_data(query):
    return session.sql(query).to_pandas()

def get_run_cycle():
    t = datetime.now().time()
    if t < datetime.strptime("11:00:00", "%H:%M:%S").time():
        return 1, t
    elif t < datetime.strptime("14:00:00", "%H:%M:%S").time():
        return 2, t
    elif t < datetime.strptime("16:00:00", "%H:%M:%S").time():
        return 3, t
    elif t < datetime.strptime("18:00:00", "%H:%M:%S").time():
        return 4, t
    else:
        return 5, t

cycle_id, _ = get_run_cycle()

# ------------------------------------------------
# Config
# ------------------------------------------------
config = {
    "PlantB2GTable": "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.PlantB2GTable"
}

# ================================================================
# 1. Fetch Base Prioritisation Data
# ================================================================
def fetch_prioritisation():
    query = """
        SELECT *
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
        WHERE (run_date, run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        )
    """
    return fetch_data(query)

# ================================================================
# 2. Fetch Dealer–SKU Dispatch
# ================================================================
def fetch_dealer_sku_dispatch():
    query = """
        SELECT
            DEALER,
            SKU,
            SUM(QUANTITY_DISPATCHED) AS TOTAL_DISPATCH
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchTable
        WHERE (run_date, run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        )
        GROUP BY DEALER, SKU
    """
    return fetch_data(query)

# ================================================================
# 3. Fetch Dealer–SKU–Plant Open Orders
# ================================================================
def fetch_dealer_sku_openorders():
    query = """
        SELECT
            DEALER,
            SKU,
            PLANT,
            TOTAL_OPEN_ORDER_QTY AS TOTAL_OPEN_ORDERS
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.OpenOrdersBlocked
        WHERE (run_date, run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        )
    """
    return fetch_data(query)

# ================================================================
# 4. Load & Join Data
# ================================================================
df = fetch_prioritisation()
dispatch_df = fetch_dealer_sku_dispatch()
openorders = fetch_dealer_sku_openorders()

df = df.merge(dispatch_df, on=["DEALER", "SKU"], how="left")
df["TOTAL_DISPATCH"] = df["TOTAL_DISPATCH"].fillna(0)

df = df.merge(openorders, on=["DEALER", "SKU", "PLANT"], how="left")
df["TOTAL_OPEN_ORDERS"] = df["TOTAL_OPEN_ORDERS"].fillna(0)

# ================================================================
# 5. Date Handling
# ================================================================
df["RUN_DATE"] = pd.to_datetime(df["RUN_DATE"], errors="coerce")
df["MONTH_PROGRESS_FACTOR"] = df["RUN_DATE"].dt.day / df["RUN_DATE"].dt.days_in_month

# ================================================================
# 6. MTD_B2G_PLANT
# ================================================================
df["MTD_B2G_PLANT"] = (
    df["MONTH_PROGRESS_FACTOR"] * df["MOP_QUANTITY"]
) - df["TOTAL_DISPATCH"] - df["TOTAL_OPEN_ORDERS"]

# ================================================================
# 7. Dealer–SKU Aggregations
# ================================================================
openorders_dealer = (
    openorders.groupby(["DEALER", "SKU"])["TOTAL_OPEN_ORDERS"]
    .sum()
    .reset_index()
    .rename(columns={"TOTAL_OPEN_ORDERS": "TOTAL_OPENORDER_DEALER_SKU"})
)

dealer_sku_agg = (
    df.groupby(["DEALER", "SKU"], as_index=False)
      .agg(
          TOTAL_QUANTITY_DEALER_SKU=("MOP_QUANTITY", "sum"),
          TOTAL_DISPATCH_DEALER_SKU=("TOTAL_DISPATCH", "sum")
      )
)

df = (
    df.merge(dealer_sku_agg, on=["DEALER", "SKU"], how="left")
      .merge(openorders_dealer, on=["DEALER", "SKU"], how="left")
)

df["TOTAL_OPENORDER_DEALER_SKU"] = df["TOTAL_OPENORDER_DEALER_SKU"].fillna(0)

# ================================================================
# 8. MTD_B2G_OVERALL
# ================================================================
df["MTD_B2G_OVERALL"] = (
    df["MONTH_PROGRESS_FACTOR"] * df["TOTAL_QUANTITY_DEALER_SKU"]
) - df["TOTAL_DISPATCH_DEALER_SKU"] - df["TOTAL_OPENORDER_DEALER_SKU"]

# ================================================================
# 9. B2G-BASED PROPORTIONATION
# ================================================================
df["POSITIVE_MTD_B2G"] = np.where(df["MTD_B2G_PLANT"] > 0, df["MTD_B2G_PLANT"], 0)

positive_b2g_agg = (
    df.groupby(["DEALER", "SKU"], as_index=False)
      .agg(TOTAL_POSITIVE_MTD_B2G=("POSITIVE_MTD_B2G", "sum"))
)

df = df.merge(positive_b2g_agg, on=["DEALER", "SKU"], how="left")
df["TOTAL_POSITIVE_MTD_B2G"] = df["TOTAL_POSITIVE_MTD_B2G"].fillna(0)

df["EFFECTIVE_B2G"] = np.where(
    (df["POSITIVE_MTD_B2G"] > 0) & (df["TOTAL_POSITIVE_MTD_B2G"] > 0),
    (df["POSITIVE_MTD_B2G"] / df["TOTAL_POSITIVE_MTD_B2G"]) * df["MTD_B2G_OVERALL"],
    0
)

df["EFFECTIVE_B2G"] = np.floor(df["EFFECTIVE_B2G"])

# ================================================================
# 10. Final Output
# ================================================================
final_df = df[
    [
        "DEALER",
        "SKU",
        "PLANT",
        "MOP_QUANTITY",
        "TOTAL_DISPATCH",
        "TOTAL_DISPATCH_DEALER_SKU",
        "TOTAL_OPEN_ORDERS",
        "B2G",
        "MTD_B2G_PLANT",
        "MTD_B2G_OVERALL",
        "EFFECTIVE_B2G",
    ]
].copy()

# ================================================================
# 11. Write to Snowflake
# ================================================================
drop_table(config["PlantB2GTable"])
write_to_snowflake(final_df, config["PlantB2GTable"])


In [ ]:
# #################################### STEP3 ##################################################
import streamlit as st
import pandas as pd
import datetime
from datetime import datetime,timedelta
import numpy as np
from snowflake.snowpark.context import get_active_session
import re
import yaml
import logging
import os
import time
pd.options.mode.chained_assignment = None

# add the b2g in inv snapshot-- done
# add total dispatched from plant ,sku in the inv snapshot

# Initialize logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
session = get_active_session()

def get_run_cycle():
    # Get the current time
    current_time = datetime.now().time()
    # Define time ranges
    if current_time >= datetime.strptime("00:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("11:00:00", "%H:%M:%S").time():
        return 1,current_time
    elif current_time >= datetime.strptime("11:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("14:00:00", "%H:%M:%S").time():
        return 2,current_time
    elif current_time >= datetime.strptime("14:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("16:00:00", "%H:%M:%S").time():
        return 3,current_time
    elif current_time >= datetime.strptime("16:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("18:00:00", "%H:%M:%S").time():
        return 4,current_time
    else:
        return 5,current_time
        
# Example usage
cycle_id,current_time = get_run_cycle()
print(f"Current run cycle: {cycle_id}")

holiday_data = {
    "HHHD": ["2/2/2025", "2/9/2025", "2/16/2025", "2/23/2025"],
    "HHHG": ["2/2/2025", "2/9/2025", "2/16/2025", "2/23/2025"],
    "HHHU": ["2/2/2025", "2/9/2025", "2/16/2025", "2/23/2025"],
    "HM4N": ["2/2/2025", "2/9/2025", "2/16/2025", "2/23/2025"],
    "HM5V": ["2/2/2025", "2/9/2025", "2/16/2025", "2/23/2025"],
    "HM6C": ["2/2/2025", "2/9/2025", "2/16/2025", "2/23/2025"]
}

holiday_data = {
    "HHHD": ["3/2/2025", "3/9/2025", "3/14/2025", "3/16/2025", "3/23/2025", "3/30/2025"],
    "HHHG": ["3/2/2025", "3/9/2025", "3/14/2025", "3/16/2025", "3/23/2025", "3/30/2025"],
    "HHHU": ["3/2/2025", "3/9/2025", "3/14/2025", "3/16/2025", "3/23/2025", "3/30/2025"],
    "HM4N": ["3/2/2025", "3/9/2025", "3/14/2025", "3/16/2025", "3/23/2025", "3/30/2025"],
    "HM5V": ["3/2/2025", "3/9/2025", "3/14/2025", "3/16/2025", "3/23/2025", "3/30/2025"],
    "HM6C": ["3/2/2025", "3/9/2025", "3/14/2025", "3/16/2025", "3/23/2025", "3/30/2025"]
}

holiday_data = {
    "HHHD": ["4/1/2025", "4/6/2025", "4/13/2025", "4/20/2025", "4/27/2025"],
    "HHHG": ["4/1/2025", "4/6/2025", "4/13/2025", "4/20/2025", "4/27/2025"],
    "HHHU": ["4/1/2025", "4/6/2025", "4/13/2025", "4/20/2025", "4/27/2025"],
    "HM4N": ["4/1/2025", "4/6/2025", "4/13/2025", "4/20/2025", "4/27/2025"],
    "HM5V": ["4/1/2025", "4/6/2025", "4/13/2025", "4/20/2025", "4/27/2025"],
    "HM6C": ["4/6/2025", "4/13/2025", "4/20/2025", "4/27/2025"]
}

holiday_data = {
    "HHHD": [ "5/4/2025", "5/11/2025", "5/18/2025", "5/25/2025"],
    "HHHG": [ "5/4/2025", "5/11/2025", "5/18/2025", "5/25/2025"],
    "HHHU": ["5/4/2025", "5/11/2025", "5/18/2025", "5/25/2025"],
    "HM4N": [ "5/4/2025", "5/11/2025", "5/18/2025", "5/25/2025"],
    "HM5V": [ "5/4/2025", "5/11/2025", "5/18/2025", "5/25/2025"],
    "HM6C": ["5/1/2025", "5/4/2025", "5/11/2025", "5/18/2025", "5/25/2025"]
}

holiday_data = {
    "HHHD": ["6/1/2025", "6/8/2025", "6/15/2025", "6/22/2025", "6/29/2025"],
    "HHHG": ["6/1/2025", "6/8/2025", "6/15/2025", "6/22/2025", "6/29/2025"],
    "HHHU": ["6/1/2025", "6/8/2025", "6/15/2025", "6/22/2025", "6/29/2025"],
    "HM4N": ["6/1/2025", "6/8/2025", "6/15/2025", "6/22/2025", "6/29/2025"],
    "HM5V": ["6/1/2025", "6/8/2025", "6/15/2025", "6/22/2025", "6/29/2025"],
    "HM6C": ["6/1/2025", "6/8/2025", "6/15/2025", "6/22/2025", "6/29/2025"]
}


holiday_data = {
    "HHHD": ["7/2/2025", "7/6/2025", "7/13/2025", "7/20/2025", "7/27/2025"],
    "HHHG": ["7/2/2025", "7/6/2025", "7/13/2025", "7/20/2025", "7/27/2025"],
    "HHHU": ["7/2/2025", "7/6/2025", "7/13/2025",
             "7/19/2025",             
             "7/20/2025",               
             "7/21/2025", "7/22/2025", "7/23/2025", "7/24/2025"],
    "HM4N": ["7/2/2025", "7/6/2025", "7/13/2025", "7/20/2025", "7/27/2025"],
    "HM5V": ["7/2/2025", "7/6/2025", "7/13/2025", "7/20/2025", "7/27/2025"],
    "HM6C": ["7/2/2025", "7/6/2025", "7/13/2025", "7/20/2025", "7/27/2025"]
}

holiday_data = {
    "HHHD": ["8/3/2025", "8/9/2025", "8/15/2025", "8/16/2025", "8/24/2025", "8/31/2025"],
    "HHHG": ["8/3/2025", "8/9/2025", "8/15/2025", "8/16/2025", "8/24/2025", "8/31/2025"],
    "HHHU": ["8/3/2025", "8/9/2025", "8/10/2025", "8/15/2025", "8/24/2025", "8/31/2025"],
    "HM4N": ["8/3/2025", "8/9/2025", "8/15/2025", "8/16/2025", "8/24/2025", "8/31/2025"],
    "HM5V": ["8/3/2025", "8/9/2025", "8/15/2025", "8/16/2025", "8/17/2025", "8/24/2025","8/31/2025"],
    "HM6C": ["8/3/2025", "8/10/2025", "8/15/2025", "8/17/2025", "8/27/2025", "8/31/2025"]
}

holiday_data = {
    "HHHD": ["9/7/2025", "9/14/2025", "9/21/2025", "9/28/2025"],
    "HHHG": ["9/7/2025", "9/14/2025", "9/21/2025", "9/28/2025"],
    "HHHU": ["9/7/2025", "9/14/2025", "9/21/2025", "9/28/2025"],
    "HM4N": ["9/7/2025", "9/14/2025", "9/21/2025", "9/28/2025"],
    "HM5V": ["9/7/2025", "9/14/2025", "9/21/2025", "9/28/2025"],
    "HM6C": ["9/7/2025", "9/14/2025", "9/21/2025", "9/28/2025"]
}

holiday_data = {
    "HHHD": ["10/2/2025", "10/10/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HHHG": ["10/2/2025", "10/10/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HHHU": ["10/2/2025", "10/10/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HM4N": ["10/2/2025", "10/10/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HM5V": ["10/2/2025", "10/12/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HM6C": ["10/2/2025", "10/12/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"]
}

holiday_data = {
    "HHHD": ["10/2/2025", "10/10/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HHHG": ["10/2/2025", "10/10/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HHHU": ["10/2/2025", "10/10/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HM4N": ["10/2/2025", "10/10/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HM5V": ["10/2/2025", "10/12/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HM6C": ["10/2/2025", "10/12/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"]
}

holiday_data = {
    "HHHD": ["11/2/2025", "11/5/2025", "11/9/2025", "11/16/2025", "11/23/2025"],
    "HHHG": ["11/2/2025", "11/5/2025", "11/9/2025", "11/16/2025", "11/23/2025"],
    "HHHU": ["11/2/2025", "11/9/2025", "11/16/2025", "11/23/2025"],
    "HM4N": ["11/2/2025", "11/9/2025", "11/16/2025", "11/23/2025"],
    "HM5V": ["11/2/2025", "11/9/2025", "11/16/2025", "11/23/2025"],
    "HM6C": ["11/2/2025", "11/9/2025", "11/16/2025", "11/23/2025"]
}

holiday_data = {
    "HHHD": ["12/1/2025", "12/7/2025", "12/14/2025", "12/21/2025", "12/28/2025", "12/29/2025", "12/30/2025", "12/31/2025"],
    "HHHG": ["12/1/2025", "12/7/2025", "12/14/2025", "12/21/2025", "12/28/2025", "12/29/2025", "12/30/2025", "12/31/2025"],
    "HHHU": ["12/1/2025", "12/7/2025", "12/14/2025", "12/21/2025", "12/28/2025", "12/29/2025", "12/30/2025", "12/31/2025"],
    "HM4N": ["12/1/2025", "12/7/2025", "12/14/2025", "12/21/2025", "12/28/2025", "12/29/2025", "12/30/2025", "12/31/2025"],
    "HM5V": ["12/1/2025", "12/7/2025", "12/14/2025", "12/21/2025", "12/28/2025", "12/29/2025", "12/30/2025", "12/31/2025"],
    "HM6C": ["12/1/2025", "12/7/2025", "12/14/2025", "12/21/2025", "12/28/2025", "12/29/2025", "12/30/2025", "12/31/2025"]
}


holiday_data = {
    "HHHD": ["1/1/2026", "1/2/2026", "1/3/2026", "1/11/2026", "1/18/2026", "1/26/2026"],
    "HHHG": ["1/1/2026", "1/2/2026", "1/3/2026", "1/11/2026", "1/18/2026", "1/26/2026"],
    "HHHU": ["1/1/2026", "1/2/2026", "1/3/2026", "1/11/2026", "1/18/2026", "1/26/2026"],
    "HM4N": ["1/1/2026", "1/2/2026", "1/3/2026", "1/11/2026", "1/18/2026", "1/26/2026"],
    "HM5V": ["1/1/2026", "1/2/2026", "1/11/2026", "1/14/2026", "1/15/2026", "1/18/2026", "1/26/2026"],
    "HM6C": ["1/1/2026", "1/2/2026", "1/3/2026", "1/11/2026", "1/14/2026", "1/15/2026", "1/16/2026", "1/26/2026"]
}

holiday_data = {
    "HHHD": ["2/1/2026", "2/8/2026", "2/15/2026", "2/22/2026"],
    "HHHG": ["2/1/2026", "2/8/2026", "2/15/2026", "2/22/2026"],
    "HHHU": ["2/1/2026", "2/8/2026", "2/15/2026", "2/22/2026"],
    "HM4N": ["2/1/2026", "2/8/2026", "2/15/2026", "2/22/2026"],
    "HM5V": ["2/1/2026", "2/8/2026", "2/15/2026", "2/22/2026"],
    "HM6C": ["2/1/2026", "2/8/2026", "2/15/2026", "2/22/2026"]
}

holiday_data = {
    "HHHD": ["3/1/2026", "3/4/2026", "3/8/2026", "3/15/2026", "3/22/2026", "3/29/2026"],
    "HHHG": ["3/1/2026", "3/4/2026", "3/8/2026", "3/15/2026", "3/22/2026", "3/29/2026"],
    "HHHU": ["3/1/2026", "3/4/2026", "3/8/2026", "3/15/2026", "3/22/2026", "3/29/2026"],
    "HM4N": ["3/1/2026", "3/4/2026", "3/8/2026", "3/15/2026", "3/22/2026", "3/29/2026"],
    "HM5V": ["3/1/2026", "3/4/2026", "3/8/2026", "3/15/2026", "3/22/2026", "3/29/2026"],
    "HM6C": ["3/1/2026", "3/8/2026", "3/15/2026", "3/19/2026", "3/22/2026", "3/29/2026"]
}

holiday_data = {
    "HHHD": ["4/1/2026", "4/5/2026", "4/12/2026", "4/19/2026", "4/26/2026"],
    "HHHG": ["4/1/2026", "4/5/2026", "4/12/2026", "4/19/2026", "4/26/2026"],
    "HHHU": ["4/1/2026", "4/5/2026", "4/12/2026", "4/19/2026", "4/26/2026"],
    "HM4N": ["4/1/2026", "4/5/2026", "4/12/2026", "4/19/2026", "4/26/2026"],
    "HM5V": ["4/1/2026", "4/5/2026", "4/12/2026", "4/19/2026", "4/26/2026"],
    "HM6C": ["4/1/2026", "4/5/2026", "4/12/2026", "4/19/2026", "4/26/2026"]
}

holiday_data = {
    "HHHD": ["5/3/2026", "5/10/2026", "5/17/2026", "5/24/2026", "5/31/2026"],
    "HHHG": ["5/3/2026", "5/10/2026", "5/17/2026", "5/24/2026", "5/31/2026"],
    "HHHU": ["5/3/2026", "5/10/2026", "5/17/2026", "5/24/2026", "5/31/2026"],
    "HM4N": ["5/3/2026", "5/10/2026", "5/17/2026", "5/24/2026", "5/31/2026"],
    "HM5V": ["5/3/2026", "5/10/2026", "5/17/2026", "5/24/2026", "5/31/2026"],
    "HM6C": ["5/1/2026", "5/3/2026", "5/10/2026", "5/17/2026", "5/24/2026", "5/31/2026"]
}

holiday_data = {
    "HHHD": ["6/7/2026", "6/14/2026", "6/21/2026", "6/28/2026"],
    "HHHG": ["6/7/2026", "6/14/2026", "6/21/2026", "6/28/2026"],
    "HHHU": ["6/7/2026", "6/14/2026", "6/21/2026", "6/28/2026"],
    "HM4N": ["6/7/2026", "6/14/2026", "6/21/2026", "6/28/2026"],
    "HM5V": ["6/7/2026", "6/14/2026", "6/21/2026", "6/28/2026"],
    "HM6C": ["6/7/2026", "6/14/2026", "6/21/2026", "6/28/2026"]
}

holiday_data = {
    "HHHD": ["7/5/2026", "7/12/2026", "7/19/2026", "7/26/2026"],
    "HHHG": ["7/5/2026", "7/12/2026", "7/19/2026", "7/26/2026"],
    "HHHU": ["7/5/2026", "7/12/2026", "7/19/2026", "7/26/2026"],
    "HM4N": ["7/5/2026", "7/12/2026", "7/19/2026", "7/26/2026"],
    "HM5V": ["7/5/2026", "7/12/2026", "7/19/2026", "7/26/2026"],
    "HM6C": ["7/5/2026", "7/12/2026", "7/19/2026", "7/26/2026"]
}

holiday_data = {
    "HHHD": ["8/2/2026", "8/9/2026", "8/16/2026", "8/23/2026","8/30/2026"],
    "HHHG": ["8/2/2026", "8/9/2026", "8/16/2026", "8/23/2026","8/30/2026"],
    "HHHU": ["8/2/2026", "8/9/2026", "8/16/2026", "8/23/2026","8/30/2026"],
    "HM4N": ["8/2/2026", "8/9/2026", "8/16/2026", "8/23/2026","8/30/2026"],
    "HM5V": ["8/2/2026", "8/9/2026", "8/16/2026", "8/23/2026","8/30/2026"],
    "HM6C": ["8/2/2026", "8/9/2026", "8/16/2026", "8/23/2026","8/30/2026"]
}

holiday_data = {
    "HHHD": ["9/6/2026", "9/13/2026", "9/20/2026", "9/27/2026"],
    "HHHG": ["9/6/2026", "9/13/2026", "9/20/2026", "9/27/2026"],
    "HHHU": ["9/6/2026", "9/13/2026", "9/20/2026", "9/27/2026"],
    "HM4N": ["9/6/2026", "9/13/2026", "9/20/2026", "9/27/2026"],
    "HM5V": ["9/6/2026", "9/13/2026", "9/20/2026", "9/27/2026"],
    "HM6C": ["9/6/2026", "9/13/2026", "9/20/2026", "9/27/2026"]
}

holiday_data = {
    "HHHD": ["10/2/2026","10/4/2026","10/11/2026", "10/18/2026","10/25/2026"],
    "HHHG": ["10/2/2026","10/4/2026","10/11/2026", "10/18/2026","10/25/2026"],
    "HHHU": ["10/2/2026","10/4/2026","10/11/2026", "10/18/2026","10/25/2026"],
    "HM4N": ["10/2/2026","10/4/2026","10/11/2026", "10/18/2026","10/25/2026"],
    "HM5V": ["10/2/2026","10/4/2026","10/11/2026", "10/18/2026","10/25/2026"],
    "HM6C": ["10/2/2026","10/4/2026","10/11/2026", "10/18/2026","10/25/2026"]
}


config={
    "PropotionedB2GLessThanZero" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.PropotionedB2GLessThanZero",
    "InventorySnapshot" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.InventorySnapshot",
    "DispatchableInstances" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances",
    "MissingModelNamesSKUs" :  "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.MissingModelNamesSKUs",
    "Max_Quantity_increment_percentage": 1.20
  }

def fetch_data(query, session):
    """
    Fetch data from the database using a SQL query.
    :param query: SQL query to execute
    :param session: Active database session
    :return: DataFrame with fetched data
    """
    return session.sql(query).to_pandas()

def drop_table(table_name):
    query = f'DROP TABLE IF EXISTS {table_name};'
    logger.info(f'Dropping table: {table_name}')
    session.sql(query).collect()
    

def delete_insert(table_name, cycle_id, current_date=pd.Timestamp.now().date()):
    # Construct the SQL query with both conditions
    query = f"""
        DELETE FROM {table_name} 
        WHERE RUN_DATE = '{current_date}' 
        AND RUN_CYCLE_ID = {cycle_id};
    """
    # Log the action
    logger.info(f"Deleting entries from {table_name} where RUN_DATE = {current_date} and RUN_CYCLE_ID = {cycle_id}")
    # Execute the query
    session.sql(query).collect()
    #session.sql.collect()

def write_to_snowflake(df, table_name, mode="append"):     
    logger.info(f'Writing to Snowflake table {table_name} with mode {mode}')
    agg_data_sp = session.create_dataframe(df)
    agg_data_sp.write.mode(mode).save_as_table(table_name)



def rank_with_fallback(df, urgency_col, quantity_col):
    # Separate positive and non-positive quantity values
    positive_df = df[df[quantity_col] > 0].copy()
    non_positive_df = df[df[quantity_col] <= 0].copy()

    # Rank the positive quantity values first
    positive_df = positive_df.sort_values(by=[urgency_col, quantity_col], ascending=[False, False])
    positive_df['MODIFIED_URGENCY_RANK'] = range(1, len(positive_df) + 1)

    # Rank the non-positive quantity values next
    non_positive_df = non_positive_df.sort_values(by=[urgency_col], ascending=[False])
    non_positive_df['MODIFIED_URGENCY_RANK'] = range(len(positive_df) + 1, len(positive_df) + len(non_positive_df) + 1)

    # Combine both dataframes
    ranked_df = pd.concat([positive_df, non_positive_df])

    return ranked_df
  

def fetch_config_values(config):
    """Fetch configuration values from the config dictionary."""
    return (
        config['PropotionedB2GLessThanZero'],
        config['MissingModelNamesSKUs'],
        config['DispatchableInstances'],
        config['InventorySnapshot']
    )


def fetch_plant_inventory(session):
    """Fetch plant inventory data."""
    plant_inventory_query = """
     SELECT a.PLANT AS PLANT, a.SKU, SUM(a.QTY) AS AVAILABLE_PLANT_STOCK 
    FROM SAP_DATA_DATABASE.SAP_DATA.VW_PLANT_INVENTORY a
    group by 1,2;
    """
    return fetch_data(plant_inventory_query, session)


def fetch_prioritisation(session):
    """Fetch plant inventory data."""
    plant_inventory_query = """
       SELECT * 
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DAILY_DISPATCH_PRIORITISATION t
        WHERE (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        );
    """
    return fetch_data(plant_inventory_query, session)


def fetch_PlantB2G (session):
    """Fetch Plant B2G data."""
    PlantB2G_query = """
       SELECT * 
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.PlantB2GTable ;
    """
    return fetch_data(PlantB2G_query, session)


def fetch_today_production_stock(session):
    """Fetch today's production stock data."""
    plant_plan_today_query = """
       select PLANT, SKU, SUM(TODAY_PRODUCTION_STOCK) as TODAY_PRODUCTION_STOCK from
       (SELECT 
    CASE
        WHEN ZLINE LIKE '1%' THEN 'HHHD'
        WHEN ZLINE LIKE '2%' THEN 'HHHG'
        WHEN ZLINE LIKE '3%' THEN 'HHHU'
        WHEN ZLINE LIKE '4%' THEN  'HM4N'
        WHEN ZLINE LIKE '5%' THEN  'HM5V'
        WHEN ZLINE LIKE '6%' THEN  'HM6C'
          
            END AS PLANT,
            Plan AS TODAY_PRODUCTION_STOCK,
            MCODE AS SKU
            
        from (
        
                select A.ZLINE,A.Plan, B.MCODE
                from
                (select ZLINE,MODEL_KEY, sum(ZQTY) as Plan from ANALYTICS_DATABASE.ANALYTICS_OPERATION.PRD_PLAN_F_FUTURE_DATA 
                where DATE_KEY = TO_CHAR(CURRENT_DATE, 'yyyymmdd') group by ZLINE,MODEL_KEY ) A
                left join 
                (select KEY, MCODE from ANALYTICS_DATABASE.ANALYTICS_SALES.MODEL_D) B
                ON
                A.MODEL_KEY = B.KEY
        
        )) group by 1,2;
    """
    return fetch_data(plant_plan_today_query, session)


def fetch_lot_size(session):
    """Fetch lot size data."""
    lot_size_query = """SELECT * FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.MODEL_LOAD_SIZE;"""
    return fetch_data(lot_size_query, session)

def fetch_open_order(session):
    open_order_query = """select PLANT, SKU, sum(TOTAL_OPEN_ORDER_QTY) as OPEN_ORDER_QTY from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.OpenOrdersBlocked t
                           WHERE (t.run_date, t.run_cycle_id) IN (
                          SELECT run_date, MAX(run_cycle_id)
                             FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_unblocked_table
                     WHERE run_date = (
                        SELECT MAX(run_date)
                        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_unblocked_table
                            )
                        GROUP BY run_date)  group by 1,2;"""
    return fetch_data(open_order_query, session)
    
def combine_inventory_and_production(plant_inventory, plant_plan_today,InventorySnapshot,open_order,unblocked_data, prioritisation):
    """Combine inventory and production data."""
    total_inventory = pd.merge(
        plant_inventory, plant_plan_today, on=['PLANT', 'SKU'], how='outer'
    ).fillna(0)
   
    total_inventory= pd.merge(total_inventory, open_order, on =['PLANT','SKU'], how = 'left')
    total_inventory.OPEN_ORDER_QTY.fillna(0,inplace=True)
    total_inventory['EOD_AVAILABLE_QUANTITY'] = (
        total_inventory['AVAILABLE_PLANT_STOCK'].fillna(0) + total_inventory['TODAY_PRODUCTION_STOCK'].fillna(0) - total_inventory['OPEN_ORDER_QTY'].fillna(0)
    )
 
    total_inventory['RUN_DATE'] = pd.Timestamp.now().date()
    total_inventory.EOD_AVAILABLE_QUANTITY =  np.where( total_inventory.EOD_AVAILABLE_QUANTITY.fillna(0) > 0, total_inventory.EOD_AVAILABLE_QUANTITY.fillna(0), 0)
    total_inventory=total_inventory.loc[total_inventory.EOD_AVAILABLE_QUANTITY > 0,:].copy()
    total_inventory = total_inventory[['RUN_DATE','PLANT','SKU','AVAILABLE_PLANT_STOCK','TODAY_PRODUCTION_STOCK','OPEN_ORDER_QTY','EOD_AVAILABLE_QUANTITY']].copy()
  
    
    filtered_prioritisation = prioritisation[prioritisation['B2G'] > 0]
    initial_b2g = filtered_prioritisation.groupby(['PLANT', 'SKU'])['B2G'].sum().reset_index()
    initial_b2g.columns =['PLANT','SKU','INITIAL_B2G']

    mop=prioritisation.groupby(['PLANT','SKU'])['QUANTITY'].sum().reset_index()
    mop.columns = ['PLANT','SKU', 'PLANNED_QUANTITY']
    b2g=unblocked_data.groupby(['PLANT','SKU'])['B2G'].sum().reset_index()
    total_inventory=pd.merge(total_inventory,initial_b2g, on =['PLANT','SKU'], how='left')
    total_inventory=pd.merge(total_inventory,b2g, on =['PLANT','SKU'], how='left')
    total_inventory=pd.merge(total_inventory,mop, on =['PLANT','SKU'], how='inner')
    total_inventory.B2G.fillna(0, inplace=True)
    total_inventory.PLANNED_QUANTITY.fillna(0, inplace=True)
    total_inventory.INITIAL_B2G.fillna(0, inplace=True)
 
    
    #drop_table(InventorySnapshot)
    total_inventory['RUN_CYCLE_ID'] = cycle_id
    total_inventory['RUN_CYCLE_TIME'] = current_time
    delete_insert(InventorySnapshot, cycle_id, current_date=datetime.now().date())
    write_to_snowflake(total_inventory, InventorySnapshot)
    return total_inventory.drop(['RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME','B2G','PLANNED_QUANTITY','INITIAL_B2G'],axis=1)


def fetch_unblocked_data(session):
    """Fetch unblocked data."""
    unblocked_query = """
      SELECT * 
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_unblocked_table t
        WHERE (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_unblocked_table
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_unblocked_table
            )
            GROUP BY run_date
        );
    """
    unblocked_data= fetch_data(unblocked_query, session)
    #unblocked_data.loc[unblocked_data.SKU=='HDESEDSZCFIMGV',:]
    unblocked_data['RUN_CYCLE_ID'] = cycle_id
    unblocked_data['RUN_CYCLE_TIME'] = current_time
    unblocked_data=unblocked_data.loc[unblocked_data.B2G.fillna(0)>0,:]
    #unblocked_data.loc[(unblocked_data.DEALER=='11845') & (unblocked_data.SKU == 'HSPPPDRSCFIBKG'),:]
    return unblocked_data


# def handle_b2g_less_than_zero(unblocked_data, B2GLessThanZero):
#     """Handle B2G less than zero."""
#     delete_insert(B2GLessThanZero, current_date=datetime.now().date())
#     write_to_snowflake(unblocked_data[unblocked_data['B2G'] <= 0], B2GLessThanZero)
#     print(unblocked_data.shape, " Before filtering B2G > 0 ")
#     print(unblocked_data.loc[unblocked_data['B2G'] > 0].shape, " Post filtering B2G > 0 ")
#     return unblocked_data[unblocked_data['B2G'] > 0]





def calculate_remaining_working_days_for_plants(current_date, holiday_data):
    next_month = 1 if current_date.month == 12 else current_date.month + 1
    next_year = current_date.year + 1 if current_date.month == 12 else current_date.year
    total_days_in_month = (datetime(next_year, next_month, 1) - timedelta(days=1)).day
    total_working_days_in_month = {}
    remaining_working_days = {}
    working_days_past = {}

    for plant, holidays in holiday_data.items():
        # Convert holiday strings to datetime.date objects
        holidays = [datetime.strptime(date, '%m/%d/%Y').date() for date in holidays]
        
        # Calculate total working days in the month
        total_working_days = total_days_in_month - len(
            [day for day in range(1, total_days_in_month + 1)
             if datetime(current_date.year, current_date.month, day).date() in holidays]
        )
        
        # Calculate days past as working days
        days_past = [
            day for day in range(1, current_date.day )
            if datetime(current_date.year, current_date.month, day).date() not in holidays
        ]
        
        # Calculate remaining working days
        remaining_days_in_month = [
            day for day in range(current_date.day, total_days_in_month + 1)
            if datetime(current_date.year, current_date.month, day).date() not in holidays
        ]
        
        # Store results
        total_working_days_in_month[plant] = total_working_days
        working_days_past[plant] = len(days_past) + 1
        remaining_working_days[plant] = len(remaining_days_in_month)
    return total_working_days_in_month, working_days_past, remaining_working_days

def merge_with_plant_B2G(unblocked_data, plant_B2G):
    """Merge unblocked data with lot size and handle missing models."""
    unblocked_data = pd.merge(unblocked_data, plant_B2G[["DEALER","SKU","PLANT","MTD_B2G_OVERALL", "EFFECTIVE_B2G","TOTAL_DISPATCH","TOTAL_DISPATCH_DEALER_SKU"]], on=["DEALER","SKU","PLANT"], how='left')
    
    # if unblocked_data.loc[unblocked_data.EFFECTIVE_B2G.isnull(), :].shape[0] > 0:
        
    #     delete_insert(MissingModelNamesSKUs, cycle_id, current_date=datetime.now().date())
    #     #drop_table(MissingModelNamesSKUs)
    #     write_to_snowflake(
    #         unblocked_data.loc[(unblocked_data.LOAD_SIZE_MULTIPLES.isnull()), :][
    #             ['RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME', 'SKU', 'MODEL', 'LOAD_SIZE_MULTIPLES']
    #         ].drop_duplicates(),
    #         MissingModelNamesSKUs
    #     )
    # print("Load size multiples not found for the following SKUs:",
    #       unblocked_data.loc[unblocked_data.LOAD_SIZE_MULTIPLES.isnull(), 'SKU'].unique())
    unblocked_data['EFFECTIVE_B2G'].fillna(0,inplace=True)
    unblocked_data['MTD_B2G_OVERALL'].fillna(0,inplace=True)
    unblocked_data['TOTAL_DISPATCH'].fillna(0,inplace=True)
    unblocked_data['TOTAL_DISPATCH_DEALER_SKU'].fillna(0,inplace=True)
    return unblocked_data.copy()



def merge_with_lot_size(unblocked_data, LOT_SIZE, MissingModelNamesSKUs):
    """Merge unblocked data with lot size and handle missing models."""
    unblocked_data = pd.merge(unblocked_data, LOT_SIZE[['MODEL', 'LOAD_SIZE_MULTIPLES']], on='MODEL', how='left')
    
    if unblocked_data.loc[unblocked_data.LOAD_SIZE_MULTIPLES.isnull(), :].shape[0] > 0:
        
        delete_insert(MissingModelNamesSKUs, cycle_id, current_date=datetime.now().date())
        #drop_table(MissingModelNamesSKUs)
        write_to_snowflake(
            unblocked_data.loc[(unblocked_data.LOAD_SIZE_MULTIPLES.isnull()), :][
                ['RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME', 'SKU', 'MODEL', 'LOAD_SIZE_MULTIPLES']
            ].drop_duplicates(),
            MissingModelNamesSKUs
        )
    print("Load size multiples not found for the following SKUs:",
          unblocked_data.loc[unblocked_data.LOAD_SIZE_MULTIPLES.isnull(), 'SKU'].unique())
    return unblocked_data.loc[(unblocked_data.LOAD_SIZE_MULTIPLES.notnull()), :].copy()


def compute_metrics(unblocked_data, total_inventory):
    """Compute metrics for unblocked data."""
    unblocked_data = pd.merge(unblocked_data, total_inventory, on=['PLANT', 'SKU'], how='left')
    inventory_columns = ['AVAILABLE_PLANT_STOCK', 'TODAY_PRODUCTION_STOCK', 'EOD_AVAILABLE_QUANTITY']
    unblocked_data[inventory_columns] = unblocked_data[inventory_columns].fillna(0)
    
    unblocked_data['SALES_VELOCITY_7DAYS'] = np.round(
        unblocked_data['LAST_N_DAY_SALES'].fillna(0) / 15, 5
    )
    unblocked_data['SKU_SALES_PROPORTION_OVERALL'] = np.round(
        unblocked_data['LAST_N_DAY_SALES'].fillna(0) /
        unblocked_data.groupby('SKU')['LAST_N_DAY_SALES'].transform('sum').fillna(1), 5
    )
    return unblocked_data


def calculate_reorder_urgency_scores(unblocked_data, beta=0.5, gamma=0.5):
    """Calculate reorder urgency scores."""
    unblocked_data['RECENCY_FACTOR'] = (
        beta * unblocked_data['SALES_VELOCITY_7DAYS'].fillna(0) +
        (1 - beta) * unblocked_data['SKU_SALES_PROPORTION_OVERALL'].fillna(0)
    )
    unblocked_data['MODIFIED_REORDER_URGENCY_SCORE'] = (
        gamma * unblocked_data['Daily_Reorder_level_urgency'].fillna(0) +
        (1 - gamma) * unblocked_data['RECENCY_FACTOR'].fillna(0)
    )
    unblocked_data['MODIFIED_REORDER_URGENCY_SCORE_PLANT_PROPORTION'] = np.round(
        unblocked_data['MODIFIED_REORDER_URGENCY_SCORE'].fillna(0) /
        unblocked_data.groupby(['SKU', 'PLANT'])['MODIFIED_REORDER_URGENCY_SCORE'].transform('sum').fillna(1), 5
    )
    
    unblocked_data['B2G_PROPORTION'] = np.round(unblocked_data['EFFECTIVE_B2G'] /
    unblocked_data.groupby(['PLANT', 'SKU'])['EFFECTIVE_B2G'].transform('sum').fillna(1), 5
    )
    unblocked_data['B2G_URGENCY_SCORE'] = (
    unblocked_data['B2G_PROPORTION'].fillna(0) + unblocked_data['MODIFIED_REORDER_URGENCY_SCORE_PLANT_PROPORTION'].fillna(0)
    )
    unblocked_data['NORMALISED_B2G_URGENCY_SCORE'] = np.round(
    unblocked_data['B2G_URGENCY_SCORE'].fillna(0) /
    unblocked_data.groupby(['PLANT', 'SKU'])['B2G_URGENCY_SCORE'].transform('sum').fillna(1), 5
    )
    return unblocked_data


def finalize_b2g(unblocked_data, PropotionedB2GLessThanZero):
    """Finalize B2G values and handle proportioned B2G less than zero."""
    current_date = datetime.now()
    remaining_days_map, days_gone, remaining_working_days = calculate_remaining_working_days_for_plants(current_date, holiday_data)
    #remaining_days_map, days_gone, remaining_working_days
    unblocked_data['Prorated_Quantity'] = unblocked_data.apply(lambda row: np.ceil(calculate_prorated_quantity(row, remaining_days_map, days_gone)), axis=1)

    
    # unblocked_data['FINAL_B2G'] = np.minimum(
    #     np.floor(unblocked_data['NORMALISED_B2G_URGENCY_SCORE'].fillna(0) * unblocked_data['EOD_AVAILABLE_QUANTITY'].fillna(0)),
    #     unblocked_data['B2G'].fillna(0)
    # )

    unblocked_data['Prorated_Quantity_B2G'] = (unblocked_data['Prorated_Quantity'].fillna(0) - unblocked_data['TOTAL_DISPATCH'].fillna(0) -  unblocked_data['TOTAL_OPEN_ORDER_QTY'].fillna(0)) 

    unblocked_data['FINAL_B2G'] = np.minimum(
                                    np.minimum(
                                        np.floor(unblocked_data['NORMALISED_B2G_URGENCY_SCORE'].fillna(0) * unblocked_data['EOD_AVAILABLE_QUANTITY'].fillna(0)),
                                        unblocked_data['EFFECTIVE_B2G'].fillna(0)
                                    ),unblocked_data['Prorated_Quantity_B2G'].fillna(0))
    
    print(unblocked_data.loc[unblocked_data['FINAL_B2G'] > 0].shape, " Post filtering final B2G > 0 ")
    unblocked_data.drop(['Prorated_Quantity','Prorated_Quantity_B2G'],axis=1,inplace=True)
    #drop_table(PropotionedB2GLessThanZero)
    delete_insert(PropotionedB2GLessThanZero, cycle_id, current_date=datetime.now().date())
    write_to_snowflake(unblocked_data[unblocked_data['FINAL_B2G'] <= 0], PropotionedB2GLessThanZero)
    return unblocked_data[unblocked_data['FINAL_B2G'] > 0]

def calculate_prorated_quantity(row, remaining_days_map, days_gone):
    plant = row['PLANT']  # Get the plant for the current row
    total_days_in_month = remaining_days_map.get(plant, 0)  # Get remaining days for the plant
    
    if total_days_in_month == 0:  # Avoid division by zero
        return 0
    
    prorated_quantity = (row['QUANTITY'] / total_days_in_month) * days_gone.get(plant,0)  # Calculate prorated quantity
    return prorated_quantity


def save_dispatchable_instances(unblocked_data, total_inventory,DispatchableInstances,current_date, holiday_data,Max_Quantity_increment_percentage):
    """Save dispatchable instances."""
    remaining_days_map, days_gone, remaining_working_days = calculate_remaining_working_days_for_plants(current_date, holiday_data)
    #remaining_days_map, days_gone, remaining_working_days
    unblocked_data['Prorated_Quantity'] = unblocked_data.apply(lambda row: np.ceil(calculate_prorated_quantity(row, remaining_days_map, days_gone)), axis=1)
    
    unblocked_data['Prorated_Quantity_B2G'] = (unblocked_data['Prorated_Quantity'].fillna(0) - unblocked_data['TOTAL_DISPATCH'].fillna(0) - unblocked_data['TOTAL_OPEN_ORDER_QTY'].fillna(0))
   
    unblocked_data['MAX_QTY_V1'] =((unblocked_data['Prorated_Quantity'].fillna(0) - unblocked_data['TOTAL_DISPATCH'].fillna(0)-  unblocked_data['TOTAL_OPEN_ORDER_QTY'].fillna(0)).combine(unblocked_data['FINAL_B2G'], max))
   
    max_qty_v1_agg = unblocked_data.groupby(['PLANT', 'SKU'])['MAX_QTY_V1'].sum().reset_index()
    
    eod_qty_agg = total_inventory.groupby(['PLANT', 'SKU'])['EOD_AVAILABLE_QUANTITY'].sum().reset_index()
    
    merged_data = pd.merge(max_qty_v1_agg, eod_qty_agg, on=['PLANT', 'SKU'], how='left')
    
    merged_data['is_understocked'] = merged_data['EOD_AVAILABLE_QUANTITY'] < merged_data['MAX_QTY_V1']
    
    unblocked_data = pd.merge(unblocked_data, merged_data[['PLANT', 'SKU', 'is_understocked']], on=['PLANT', 'SKU'], how='left')
    
    unblocked_data['SUM_MAX_QTY_V1'] = unblocked_data.groupby(['PLANT', 'SKU'])['MAX_QTY_V1'].transform('sum')
    
    unblocked_data['Normalized_MAX_QTY'] = unblocked_data.apply(lambda row: (row['MAX_QTY_V1'] / row['SUM_MAX_QTY_V1']) if row['is_understocked'] else 1, axis=1)

    unblocked_data['MAX_QTY'] = np.floor(unblocked_data.apply(
        lambda row: row['EOD_AVAILABLE_QUANTITY'] * row['Normalized_MAX_QTY'] if row['is_understocked'] 
        else row['MAX_QTY_V1'], axis=1))
    
    unblocked_data.MAX_QTY.fillna(0,inplace=True)
    
    unblocked_data.drop(columns=['SUM_MAX_QTY_V1', 'is_understocked'], inplace=True)

    unblocked_data["MAX_QTY"] = np.minimum(np.minimum(np.maximum(unblocked_data["MAX_QTY"], unblocked_data["FINAL_B2G"]),
                                           np.floor(unblocked_data['FINAL_B2G'] * Max_Quantity_increment_percentage)), unblocked_data['B2G'])


    unblocked_data = rank_with_fallback(unblocked_data, 'B2G_URGENCY_SCORE', 'FINAL_B2G')
    
    unblocked_data = unblocked_data.sort_values('MODIFIED_URGENCY_RANK')

    unblocked_data['N'] = 15
    unblocked_data['PRORITY_RANK_PLANT_WISE'] = unblocked_data.groupby('PLANT')['MODIFIED_URGENCY_RANK'].rank( method='first' )
    
    unblocked_data = unblocked_data.sort_values(['PLANT','PRORITY_RANK_PLANT_WISE'])

    unblocked_data = unblocked_data.rename(columns={"expected_demand_in_day": "EXPECTED_DEMAND_REMAINDER_WEEK", "SALES_VELOCITY_7DAYS": "LAST_N_DAY_PER_DAY_SALES",
                                                   "MODIFIED_URGENCY_RANK" : "PRIORITY_RANK_OVERALL","OPEN_ORDER_QTY":"PLANT_SKU_OPEN_ORDER_QTY"})

    unblocked_data.PLANT_SKU_OPEN_ORDER_QTY.fillna(0,inplace=True)
    unblocked_data.drop('Prorated_Quantity_B2G',axis=1,inplace=True)
    #drop_table(DispatchableInstances)
    delete_insert(DispatchableInstances, cycle_id, current_date=datetime.now().date())
    write_to_snowflake(unblocked_data, DispatchableInstances)
    return unblocked_data


# Main Execution
def main(config, session):
    current_date = datetime.now()
    PropotionedB2GLessThanZero, MissingModelNamesSKUs, DispatchableInstances,InventorySnapshot = fetch_config_values(config)

    Max_Quantity_increment_percentage =  config['Max_Quantity_increment_percentage']

    plant_inventory = fetch_plant_inventory(session)
    plant_plan_today = fetch_today_production_stock(session)
    LOT_SIZE = fetch_lot_size(session)
    unblocked_data = fetch_unblocked_data(session)
    prioritisation= fetch_prioritisation(session)
    
    PlantB2G = fetch_PlantB2G(session)
    
    open_order = fetch_open_order(session)
    total_inventory = combine_inventory_and_production(plant_inventory, plant_plan_today,InventorySnapshot,open_order,unblocked_data,prioritisation)
    
    unblocked_data= merge_with_plant_B2G(unblocked_data, PlantB2G)
    unblocked_data = merge_with_lot_size(unblocked_data, LOT_SIZE, MissingModelNamesSKUs)
    #unblocked_data.loc[(unblocked_data.SKU == 'HPLNDDRVCFIBTL'),:]
    unblocked_data = compute_metrics(unblocked_data, total_inventory)
    #unblocked_data.loc[(unblocked_data.SKU == 'HPLNDDRVCFIBTL'),:]
    unblocked_data = calculate_reorder_urgency_scores(unblocked_data)
    #unblocked_data.loc[(unblocked_data.SKU == 'HSPSADSSCFIMBN'),:]
    
    unblocked_data = finalize_b2g(unblocked_data, PropotionedB2GLessThanZero)
    #unblocked_data.loc[(unblocked_data.SKU == 'HSPSADSSCFIMBN'),:]
    dispatchable_data = save_dispatchable_instances(unblocked_data, total_inventory,DispatchableInstances,current_date,holiday_data,Max_Quantity_increment_percentage)
    print(dispatchable_data.shape, dispatchable_data.B2G.sum())
    return dispatchable_data,plant_inventory,plant_plan_today,total_inventory

dispatchable_data,plant_inventory,plant_plan_today,total_inventory = main(config, session)
time.sleep(20)

In [ ]:
# #################################### STEP4 ##################################################
import streamlit as st
import pandas as pd
import datetime
from datetime import datetime
import numpy as np
from snowflake.snowpark.context import get_active_session
import re
import yaml
import logging
import os
import time


pd.options.mode.chained_assignment = None

# Complete trucks should be flagged - DOne
# Initialize logger

# For Min qty rows filled, max_row_filled_no, total_QTY , no of trucks  - done

# Get the key_identifier created - done

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
session = get_active_session()

config={
    
    "AllProcessedTrucksMinQty":"MOP_DATABASE.DISPATCH_AUTOMATION_OBL.AllProcessedTrucksMinQty",
    "UnderOrderInstances" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.UnderOrderInstances",
    "DailyMinQtyTrucks":"MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyMinQtyTrucks",
    "FundBlockedPostMinTrucks":"MOP_DATABASE.DISPATCH_AUTOMATION_OBL.FundBlockedPostMinTrucks"
}

def get_run_cycle():
    # Get the current time
    current_time = datetime.now().time()
    # Define time ranges
    if current_time >= datetime.strptime("00:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("11:00:00", "%H:%M:%S").time():
        return 1,current_time
    elif current_time >= datetime.strptime("11:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("14:00:00", "%H:%M:%S").time():
        return 2,current_time
    elif current_time >= datetime.strptime("14:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("16:00:00", "%H:%M:%S").time():
        return 3,current_time
    elif current_time >= datetime.strptime("16:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("18:00:00", "%H:%M:%S").time():
        return 4,current_time
    else:
        return 5,current_time
        
# Example usage
cycle_id,current_time = get_run_cycle()
print(f"Current run cycle: {cycle_id}")



def fetch_data(query, session):
    """
    Fetch data from the database using a SQL query.
    :param query: SQL query to execute
    :param session: Active database session
    :return: DataFrame with fetched data
    """
    return session.sql(query).to_pandas()

def drop_table(table_name):
    query = f'DROP TABLE IF EXISTS {table_name};'
    logger.info(f'Dropping table: {table_name}')
    session.sql(query).collect()
    

def delete_insert(table_name, cycle_id, current_date=pd.Timestamp.now().date()):
    # Construct the SQL query with both conditions
    query = f"""
        DELETE FROM {table_name} 
        WHERE RUN_DATE = '{current_date}' 
        AND RUN_CYCLE_ID = {cycle_id};
    """
    # Log the action
    logger.info(f"Deleting entries from {table_name} where RUN_DATE = {current_date} and RUN_CYCLE_ID = {cycle_id}")
    # Execute the query
    session.sql(query).collect()
    #session.sql.collect()

def write_to_snowflake(df, table_name, mode="append"):     
    logger.info(f'Writing to Snowflake table {table_name} with mode {mode}')
    agg_data_sp = session.create_dataframe(df)
    agg_data_sp.write.mode(mode).save_as_table(table_name)


def fetch_fund_data(session):
    fund_dealer = """ select DEALER, OBL_CR_LIMIT_AVAIL AS fund_available
    from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.FundBlocked t
     WHERE (t.run_date, t.run_cycle_id) IN (
        SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
        WHERE run_date = (
            SELECT MAX(run_date)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
        )
        GROUP BY run_date
        ) 
        AND FUND_BLOCK_STATUS = 'Not Blocked';
        """
       
    return fetch_data(fund_dealer, session)
    
    
def fetch_dispatchable_data(session):
    """Fetch unblocked data."""
    DispatchInstancesQuery = """
            SELECT * 
    FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances t
    WHERE (t.run_date, t.run_cycle_id) IN (
        SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation
        WHERE run_date = (
            SELECT MAX(run_date)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation
        )
        GROUP BY run_date
        );

    """
       
    return fetch_data(DispatchInstancesQuery, session)


def split_min_qty_into_equal_rows(min_qty):
    return [1 for _ in range(min_qty)]

def get_inventory_remaining(df,sku):
    dat=df[df['SKU']==sku].sort_values(by='index',ascending=True)
    dat['prev_inventory_used_truck']=dat.reset_index(drop=True).index
    dat['remaining_inventory']=dat['AVAILABLE_QTY'] - dat['prev_inventory_used_truck']
    return dat

def getmaxRow(df, sku, use_max=False):
    num_unique_not_null_row_id = df.loc[~pd.isnull(df['row_id']), 'row_id'].nunique()

    if num_unique_not_null_row_id > 0:
        valid_sku_types = [sku + "_MIN_QTY"]
        if use_max:
            valid_sku_types.append(sku + "_MAX_QTY")
        df['sku_qty'] = df['SKU'] + "_" + df['QTY_TYPE']
        agg_df = df.groupby("row_id").agg({
            "sku_qty": list,
        }).reset_index()
        agg_df['is_valid'] = agg_df.apply(
            lambda row: 1 if any(valid_type in row['sku_qty'] for valid_type in valid_sku_types) else 0,
            axis=1
        )
        return agg_df.loc[agg_df['is_valid'] == 1, 'row_id'].tolist()

    return [100000]

def fill_rows(df, max_skus, remaining_fund_dict, max_rows=10, row_filled=0, max_row_multiplier={}, is_truck_filling=False):
    max_row_multiplier = {}
    done_data_list = []
    rows_to_ignore = []
    ran = max_skus
    to_iterate = True
    multiplier_data={"4X":[4]*10,"5X":[5]*10,"3X+2X":[3,2]*5}
    
    DEALER = df.DEALER.iloc[0]
    
    current_fund_quantity = int(np.floor(remaining_fund_dict.get(DEALER,0)/72000))
    if current_fund_quantity >= 3 :
   
        while to_iterate:
          
            for i in range(0, ran + 1):
                
                filter_df = df[~df['index'].isin(rows_to_ignore)]
                
                sku_row_filled = 0
                temp = filter_df[filter_df['RANK'] == i]
              
                if temp.shape[0] > 0:
                    sku = temp['SKU'].tolist()[0]
                    multiplier_order = temp['MULTIPLIER'].tolist()[0]
                    multiplier = multiplier_data[multiplier_order]
                    
                    if multiplier_order in max_row_multiplier:
                        rows_done_multiplier = int(max_row_multiplier[multiplier_order])
                        multiplier = multiplier[rows_done_multiplier:]
                        
                    same_multiplier_data = filter_df[filter_df['MULTIPLIER'] == multiplier_order]
                    same_multiplier_data = same_multiplier_data.sort_values(by="index", ascending=True)
                    cumulative_sum = [sum(multiplier[:i + 1]) for i in range(len(multiplier))]
    
                    same_multiplier_data['row_index'] = same_multiplier_data.reset_index(drop=True).index + 1
                    same_multiplier_data['is_modulus'] = same_multiplier_data['row_index'].apply(
                        lambda x: 1 if x in cumulative_sum else np.nan)
                    same_multiplier_data['cum_rows'] = same_multiplier_data['is_modulus'].fillna(0).cumsum()
                    same_multiplier_data['row_id'] = same_multiplier_data.apply(
                        lambda row: np.nan if pd.isnull(row['is_modulus']) else row['cum_rows'] + row_filled, axis=1)
                    same_multiplier_data['row_id'] = same_multiplier_data['row_id'].bfill()
                    if is_truck_filling == False:
                        valid_row_id = getmaxRow(same_multiplier_data, sku)
                        same_multiplier_data = same_multiplier_data[same_multiplier_data['row_id'].isin(valid_row_id)]
                    if is_truck_filling == False:
                        same_multiplier_data = same_multiplier_data[same_multiplier_data['row_id'] <= max_rows]
                    else:
                        max_additional_rows = row_filled + max_rows
                        same_multiplier_data = same_multiplier_data[same_multiplier_data['row_id'] <= max_additional_rows] 
                    if same_multiplier_data.shape[0] > 0:
                        done_data_list.append(same_multiplier_data)
                        for val in same_multiplier_data['index'].tolist():
                            rows_to_ignore.append(val)
                        valid_row_id = same_multiplier_data.loc[~pd.isnull(same_multiplier_data['row_id']), 'row_id'].unique()
                        if len(valid_row_id) > 0:
                            row_filled = max(valid_row_id)
                        if multiplier_order not in max_row_multiplier:
                            max_row_multiplier[multiplier_order] = len(same_multiplier_data['row_id'].unique())
                        else:
                            max_row_multiplier[multiplier_order] += len(same_multiplier_data['row_id'].unique())               
                if i == ran:
                    to_iterate = False
                if row_filled == max_rows:
                    to_iterate = False
            done_data = pd.DataFrame()
            remaining_data = pd.DataFrame()
            if len(done_data_list) > 0:
                done_data = pd.concat(done_data_list)
                done_data=done_data.iloc[0:current_fund_quantity].copy()
                row_filled = done_data.loc[done_data['is_modulus'] == 1, 'row_id'].max()
                row_filled = 0 if pd.isna(row_filled) else int(row_filled)
                if row_filled > 0:
                    done_data = done_data.loc[done_data.row_id <= row_filled,:].copy()
                    rows_to_ignore = done_data['index'].tolist()
                    if current_fund_quantity <=  len(done_data):
                        remaining_data = pd.DataFrame()
                    else:    
                        remaining_data = df.loc[~df['index'].isin(rows_to_ignore),['QTY','PLANT','DEALER','SKU','AVAILABLE_QTY','MULTIPLIER','SCORE','RANK','MAX_QTY','QTY_TYPE','index']].copy()     
            remaining_rows = max_rows - row_filled
    else:
        row_filled,done_data,rows_to_ignore,remaining_rows,max_row_multiplier,remaining_data = 0,pd.DataFrame(),[],10,0,pd.DataFrame()
    return row_filled, done_data, rows_to_ignore, remaining_rows, max_row_multiplier, remaining_data

def preprocess_data(merged_data):
    """
    Preprocesses the merged_data DataFrame.
    """
    merged_data['LOAD_SIZE_MULTIPLES'] = np.where(
        merged_data['LOAD_SIZE_MULTIPLES'] == '3X+2X (MAX 13)', 
        '3X+2X', 
        merged_data['LOAD_SIZE_MULTIPLES']
    )
    
    merged_data=merged_data[['DEALER','PLANT','SKU','LOAD_SIZE_MULTIPLES','FINAL_B2G','MAX_QTY','EOD_AVAILABLE_QUANTITY','B2G_URGENCY_SCORE']].copy()
    merged_data.columns = ['DEALER', 'PLANT', 'SKU', 'MULTIPLIER', 'MIN_QTY', 'MAX_QTY', 'AVAILABLE_QTY', 'SCORE']
    return merged_data


def split_and_rank_data(data):
    """
    Splits the MIN_QTY and MAX_QTY into rows and ranks them by SCORE.
    """
    df_list, df_list_2 = [], []

    max_skus = data['ORDER'].max()
    #print(max_skus,"max skus")

    if not isinstance(max_skus, int):
        raise TypeError(f"{max_skus} is not an integer")

    for i in range(0, max_skus + 1):
        temp_df = data[data['ORDER'] == i]

        # MIN_QTY rows
        min_rows = split_min_qty_into_equal_rows(int(temp_df['MIN_QTY'].tolist()[0]))
        df_1 = pd.DataFrame({'QTY': min_rows})
        append_common_columns(df_1, temp_df, "MIN_QTY")
        df_list.append(df_1)

        # MAX_QTY rows
        temp_df['REMAINING_QTY'] = temp_df['MAX_QTY'] - temp_df['MIN_QTY']
        max_rows = split_min_qty_into_equal_rows(int(temp_df['REMAINING_QTY'].tolist()[0]))
        df_2 = pd.DataFrame({'QTY': max_rows})
        append_common_columns(df_2, temp_df, "MAX_QTY")
        df_list_2.append(df_2)
    df_1 = pd.concat(df_list)
    df_2 = pd.concat(df_list_2)
    return pd.concat([df_1, df_2], axis=0)

def append_common_columns(df, temp_df, qty_type):
    """
    Appends common columns to the split rows.
    """
    df['PLANT'] = temp_df['PLANT'].tolist()[0]
    df['DEALER'] = temp_df['DEALER'].to_list()[0]
    df['SKU'] = temp_df['SKU'].tolist()[0]
    df['AVAILABLE_QTY'] = temp_df['AVAILABLE_QTY'].tolist()[0]
    df['MULTIPLIER'] = temp_df['MULTIPLIER'].tolist()[0]
    df['SCORE'] = temp_df['SCORE'].tolist()[0]
    df['RANK'] = temp_df['ORDER'].tolist()[0]
    df['MAX_QTY'] = temp_df['MAX_QTY'].tolist()[0]
    df['QTY_TYPE'] = qty_type

def process_min_qty_rows(df_min, remaining_inventory_dict,remaining_fund_dict):
    """
    Processes the rows with MIN_QTY and computes the results.
    """
    results_list = []
    under_order_list = []
    filtered_inventory_table = pd.DataFrame()
    incomplete_truck_remaining_data = pd.DataFrame()
    nill_truck_remaining_data = pd.DataFrame()
    while not df_min.empty:
        df_min = df_min.sort_values(by=["SCORE"], ascending=[False]).reset_index(drop=True)
        top_row = df_min.iloc[0]
        plant, dealer = top_row['PLANT'], top_row['DEALER']
        group = df_min[(df_min['PLANT'] == plant) & (df_min['DEALER'] == dealer)]
        df_min = df_min[~((df_min['PLANT'] == plant) & (df_min['DEALER'] == dealer))]
        group_data = group.copy()
        group_data['index'] = group_data.reset_index(drop=True).index
        group_data['AVAILABLE_QTY'] = group_data.apply(
            lambda row: remaining_inventory_dict.get((row['PLANT'], row['SKU']), row['AVAILABLE_QTY']),axis=1)
        inv_df_list = [get_inventory_remaining(group_data, sku) for sku in group_data['SKU'].unique()]
        group_inv_data = pd.concat(inv_df_list)
        inv_less_df = group_inv_data[group_inv_data['remaining_inventory'] <= 0]
        
        group_inv_data = group_inv_data[group_inv_data['remaining_inventory'] > 0]
        
        filtered_inventory_table = pd.concat([filtered_inventory_table, inv_less_df], axis=0)
        max_skus = group_inv_data['SKU'].nunique()
        group_inv_data['RANK'] = (
            pd.factorize(group_inv_data[['PLANT', 'DEALER', 'SKU']].apply(tuple, axis=1))[0]
        )
        
        if group_inv_data.shape[0] > 0 :
            row_filled, done_data, rows_to_ignore, remaining_rows, max_row_multiplier, remaining_data = fill_rows(group_inv_data, max_skus,remaining_fund_dict)
            
            if row_filled == 10:
                df_min = pd.concat([remaining_data, df_min], axis=0)
            elif row_filled > 0:
                incomplete_truck_remaining_data = pd.concat(
                    [incomplete_truck_remaining_data, remaining_data], axis=0
                )
            elif row_filled == 0:
                nill_truck_remaining_data = pd.concat([nill_truck_remaining_data, remaining_data], axis=0)
            if len(done_data) > 0:
                current_plant = done_data['PLANT'].iloc[0]
                current_dealer = done_data['DEALER'].iloc[0]
                existing_truck_nos = [ df['Truck_no'].max()
                    for df in results_list
                    if df['PLANT'].iloc[0] == current_plant and df['DEALER'].iloc[0] == current_dealer]          
                if existing_truck_nos:
                    done_data['Truck_no'] = max(existing_truck_nos) + 1       
                else:
                    done_data['Truck_no'] = 1        
                results_list.append(done_data)
                min_inventory_group = done_data.groupby(['PLANT', 'SKU'])['remaining_inventory'].min().reset_index()
                for _, row in min_inventory_group.iterrows():
                    plant, sku, min_inventory = row['PLANT'], row['SKU'], row['remaining_inventory']
                    remaining_inventory_dict[(plant, sku)] = min_inventory - 1
                remaining_fund_dict[dealer] = remaining_fund_dict.get(dealer,0) - (len(done_data) * 72000)              
            if row_filled == 0:
                total_qty = group_inv_data['QTY'].sum()
                unique_skus = group_inv_data['SKU'].nunique()
                under_order_list.append({
                    'PLANT': plant,
                    'DEALER': dealer,
                    'STATUS': "Under-Ordered",
                    'NUNIQUE_SKU': unique_skus,
                    'SUM_QTY': total_qty
                })
        else:
            continue
    final_done_data = pd.concat(results_list) if results_list else pd.DataFrame()
    
    under_order_df = pd.DataFrame(under_order_list)
    
    return final_done_data, under_order_df,nill_truck_remaining_data,incomplete_truck_remaining_data,remaining_fund_dict

def main():
    
    AllProcessedTrucksMinQty = config['AllProcessedTrucksMinQty']
    UnderOrderInstances =  config['UnderOrderInstances']
    DailyMinQtyTrucks = config['DailyMinQtyTrucks']
    FundBlockedPostMinTrucks=config['FundBlockedPostMinTrucks']
   
    
    dispatchable_data = fetch_dispatchable_data(session)
    fund_data =fetch_fund_data(session)
    merged_data = preprocess_data(dispatchable_data)
    dispatchable_data.loc[(dispatchable_data.DEALER=='10059') & (dispatchable_data.SKU=='HSPLMDRSCFIBHG')]
    score=dispatchable_data.groupby(['DEALER','SKU'])['B2G_URGENCY_SCORE'].mean().reset_index()
    #dat = merged_data.loc[(merged_data.PLANT == 'HHHG'), :].copy()
    dat = merged_data.copy()
    dat = dat.sort_values(by="SCORE", ascending=False).reset_index(drop=True)
    dat['ORDER'] = dat.index
    inventory_data = merged_data[['PLANT', 'SKU', 'AVAILABLE_QTY']].drop_duplicates()
    
    remaining_inventory_dict = {
        (row['PLANT'], row['SKU']): row['AVAILABLE_QTY']
        for _, row in inventory_data.iterrows()}
    remaining_fund_dict = {
        row['DEALER'] : row['FUND_AVAILABLE']
        for _, row in fund_data.iterrows()}
    df = split_and_rank_data(dat)
    df_min = df.loc[df.QTY_TYPE == 'MIN_QTY'].copy()

    
    #df_min = df_min.loc[(df_min.DEALER == '11682'),:].copy()

    final_done_data, under_order_df, nill_truck_remaining_data, incomplete_truck_remaining_data,remaining_fund_dict = process_min_qty_rows(df_min, remaining_inventory_dict,remaining_fund_dict)
    # under_order_df
    # nill_truck_remaining_data
    # incomplete_truck_remaining_data
    # remaining_fund_dict
    row_id_max_data = final_done_data.groupby(['DEALER', 'PLANT', 'Truck_no'])['row_id'].max().reset_index()
    row_id_max_data.rename(columns={'row_id': 'row_id_max'}, inplace=True)
    final_done_data = final_done_data.merge(row_id_max_data, on=['DEALER', 'PLANT', 'Truck_no'], how='left')



    
    min_agg = final_done_data.groupby(['DEALER', 'PLANT', 'SKU', 'MULTIPLIER', 'Truck_no']).agg(
        QTY=('QTY', 'sum'),
        row_id_max=('row_id_max', 'first')).reset_index()
    min_agg['IS_COMPLETE_TRUCK'] = (min_agg['row_id_max'] == 10).astype(int)
    
    min_agg['RUN_DATE'] = pd.Timestamp.now().date()
    min_agg['RUN_CYCLE_ID'] = cycle_id
    min_agg['RUN_CYCLE_TIME'] = current_time
    min_agg['KEY_IDENTIFIER'] = min_agg.PLANT + '_' +  min_agg.DEALER + '_' + min_agg.Truck_no.astype(int).astype(str)
    min_agg= min_agg[[ 'RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME','KEY_IDENTIFIER','DEALER','SKU','PLANT','MULTIPLIER','Truck_no','QTY','IS_COMPLETE_TRUCK']].copy()
    min_agg.columns = ['RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME','KEY_IDENTIFIER','DEALER','SKU','PLANT','MULTIPLIER','Truck_no','UNITS_TO_BE_DISPATCHED','IS_COMPLETE_TRUCK']
  
    min_agg=pd.merge(min_agg,fund_data.loc[fund_data.DEALER.isin(min_agg.DEALER),['DEALER','FUND_AVAILABLE']],on='DEALER',how='left')
    min_agg['CUMMULATIVE_COST']=min_agg.groupby('DEALER')['UNITS_TO_BE_DISPATCHED'].cumsum() * 72000 

    min_agg=pd.merge(min_agg,score, on =['DEALER','SKU'], how='left')
    
    agg_values = min_agg.groupby('KEY_IDENTIFIER')['B2G_URGENCY_SCORE'].agg(
                    Score_max=lambda x: x.dropna().max(),
                    Score_Median=lambda x: x.dropna().median()).fillna(0)
    
    min_agg['Max_Score'] = min_agg['KEY_IDENTIFIER'].map(agg_values['Score_max'])
    min_agg['Median_Score'] = min_agg['KEY_IDENTIFIER'].map(agg_values['Score_Median'])
    
    min_agg['Max_Score_Rank'] = min_agg.groupby(['PLANT'])['Max_Score'].rank(method='dense',ascending=False)
    min_agg['Median_Score_Rank'] = min_agg.groupby(['PLANT'])['Median_Score'].rank(method='dense',ascending=False)
    
    min_agg = min_agg.sort_values(['PLANT','Max_Score_Rank','KEY_IDENTIFIER'])

   
    min_agg['CUMMULATIVE_COST']=min_agg.groupby('DEALER')['UNITS_TO_BE_DISPATCHED'].cumsum() * 72000 
  
    
    under_order_df['RUN_DATE'] = pd.Timestamp.now().date()
    under_order_df['RUN_CYCLE_ID'] = cycle_id
    under_order_df['RUN_CYCLE_TIME'] = current_time  
    
    remaining_fund_dict = pd.DataFrame([{'DEALER': dealer, 'FUND_AVAILABLE': fund} for dealer, fund in remaining_fund_dict.items()])
    
    remaining_fund_dict['RUN_DATE'] = pd.Timestamp.now().date()
    remaining_fund_dict['RUN_CYCLE_ID'] = cycle_id
    remaining_fund_dict['RUN_CYCLE_TIME'] = current_time  
    
    drop_table(AllProcessedTrucksMinQty)
    write_to_snowflake(final_done_data,AllProcessedTrucksMinQty)

    #drop_table(FundBlockedPostMinTrucks)
    delete_insert(FundBlockedPostMinTrucks, cycle_id, current_date=datetime.now().date())
    write_to_snowflake(remaining_fund_dict,FundBlockedPostMinTrucks)
    if len(under_order_df) > 0:
         #drop_table(UnderOrderInstances)
        delete_insert(UnderOrderInstances, cycle_id, current_date=datetime.now().date())
        write_to_snowflake(under_order_df,UnderOrderInstances)
    delete_insert(DailyMinQtyTrucks, cycle_id, current_date=datetime.now().date())
    #drop_table(DailyMinQtyTrucks)
    write_to_snowflake(min_agg,DailyMinQtyTrucks)    
    return final_done_data, under_order_df, nill_truck_remaining_data, incomplete_truck_remaining_data,min_agg,remaining_fund_dict, dispatchable_data  

final_done_data, under_order_df, nill_truck_remaining_data, incomplete_truck_remaining_data,min_agg,remaining_fund_dict , dispatchable_data= main()
time.sleep(20)
# min_trucks= final_done_data.copy()


In [ ]:
# #################################### STEP5 ##################################################

import streamlit as st
import pandas as pd
import datetime
from datetime import datetime
import numpy as np
from snowflake.snowpark.context import get_active_session
import re
import yaml
import logging
import os
import time

# Initialize logger

# ALl trucks need the median score, min score, median score  rank, min score rank based rank
# Get the key_identifier created

pd.options.mode.chained_assignment = None

# Initialize logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
session = get_active_session()

def get_run_cycle():
    # Get the current time
    current_time = datetime.now().time()
    # Define time ranges
    if current_time >= datetime.strptime("00:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("11:00:00", "%H:%M:%S").time():
        return 1,current_time
    elif current_time >= datetime.strptime("11:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("14:00:00", "%H:%M:%S").time():
        return 2,current_time
    elif current_time >= datetime.strptime("14:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("16:00:00", "%H:%M:%S").time():
        return 3,current_time
    elif current_time >= datetime.strptime("16:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("18:00:00", "%H:%M:%S").time():
        return 4,current_time
    else:
        return 5,current_time

# Example usage
cycle_id,current_time = get_run_cycle()
print(f"Current run cycle: {cycle_id}")

config={
    "FullTrucksMinQty":"MOP_DATABASE.DISPATCH_AUTOMATION_OBL.FullTrucksMinQty",
    "UnderOrderInstances" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.UnderOrderInstances",
    "AllProcessedTrucksMinMaxQty":"MOP_DATABASE.DISPATCH_AUTOMATION_OBL.AllProcessedTrucksMinMaxQty",
    "DailyMaxQtyTrucks" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyMaxQtyTrucks",
    "FundBlockedPostMaxTrucks":"MOP_DATABASE.DISPATCH_AUTOMATION_OBL.FundBlockedPostMaxTrucks",
    "NoRowsIncrease" : 3
}


def fetch_data(query, session):
    """
    Fetch data from the database using a SQL query.
    :param query: SQL query to execute
    :param session: Active database session
    :return: DataFrame with fetched data
    """
    return session.sql(query).to_pandas()

def drop_table(table_name):
    query = f'DROP TABLE IF EXISTS {table_name};'
    logger.info(f'Dropping table: {table_name}')
    session.sql(query).collect()
    

def delete_insert(table_name, cycle_id, current_date=pd.Timestamp.now().date()):
    # Construct the SQL query with both conditions
    query = f"""
        DELETE FROM {table_name} 
        WHERE RUN_DATE = '{current_date}' 
        AND RUN_CYCLE_ID = {cycle_id};
    """
    # Log the action
    logger.info(f"Deleting entries from {table_name} where RUN_DATE = {current_date} and RUN_CYCLE_ID = {cycle_id}")
    # Execute the query
    session.sql(query).collect()
    #session.sql.collect()

def write_to_snowflake(df, table_name, mode="append"):     
    logger.info(f'Writing to Snowflake table {table_name} with mode {mode}')
    agg_data_sp = session.create_dataframe(df)
    agg_data_sp.write.mode(mode).save_as_table(table_name)



def fetch_all_trucks_data(session):
    """Fetch all trucks data."""
    AllProcessedTrucks_query = """
        SELECT *
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.AllProcessedTrucksMinQty;
    """
    return fetch_data(AllProcessedTrucks_query, session)

def fetch_fund_data(session):
    fund_dealer = """ select DEALER, OBL_CR_LIMIT_AVAIL AS fund_available
    from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.FundBlocked t
     WHERE (t.run_date, t.run_cycle_id) IN (
        SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation
        WHERE run_date = (
            SELECT MAX(run_date)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation
        )
        GROUP BY run_date
        ) 
        AND FUND_BLOCK_STATUS = 'Not Blocked';
        """
       
    return fetch_data(fund_dealer, session)

def fetch_inventory_data(session):
    """Fetch Inventory data."""
    Inventory_query = """
                SELECT * 
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.InventorySnapshot t
        WHERE (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.InventorySnapshot
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.InventorySnapshot
            )
            GROUP BY run_date
        );
    """
    return fetch_data(Inventory_query, session)

def fetch_dispatchable_data(session):
    """Fetch unblocked data."""
    DispatchInstancesQuery = """
               SELECT * 
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances t
        WHERE (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation
            )
            GROUP BY run_date
        );
    """
    return fetch_data(DispatchInstancesQuery, session)

def preprocess_dispatch_data(DispatchInstances, AllProcessedTrucks):
    """Process Dispatch data."""
    max_row_ids = AllProcessedTrucks.groupby(['PLANT', 'DEALER', 'Truck_no'])['row_id'].transform('max')
    filtered_trucks = AllProcessedTrucks[max_row_ids == 10].copy()

    filtered_trucks = filtered_trucks.groupby(['PLANT', 'SKU', 'DEALER'])['QTY'].sum().reset_index()
    filtered_trucks.columns = ['PLANT', 'SKU', 'DEALER', 'MIN_COMPLETE_TRUCKS_QTY']

    DispatchInstances = pd.merge(DispatchInstances, filtered_trucks, on=['PLANT', 'SKU', 'DEALER'], how='left')
    DispatchInstances.MIN_COMPLETE_TRUCKS_QTY.fillna(0, inplace=True)

    DispatchInstances['UPDATED_MIN_QTY'] = DispatchInstances.FINAL_B2G - DispatchInstances.MIN_COMPLETE_TRUCKS_QTY
    DispatchInstances['UPDATED_MAX_QTY'] = np.maximum(DispatchInstances['UPDATED_MIN_QTY'],
                                                     DispatchInstances['MAX_QTY'] - DispatchInstances['MIN_COMPLETE_TRUCKS_QTY'])

    print(DispatchInstances.shape, "Before filtering updated max qty > 0")

    DispatchInstances = DispatchInstances.loc[DispatchInstances.UPDATED_MAX_QTY > 0, :].copy()

    MinQtyFullTrucks = AllProcessedTrucks[max_row_ids == 10].copy()
    print(DispatchInstances.shape, "Post filtering updated max qty > 0")

    return DispatchInstances, MinQtyFullTrucks, filtered_trucks

def preprocess_inventory_data(Inventory, filtered_trucks):
    """Process Inventory data."""
    Inventory = pd.merge(Inventory, filtered_trucks.groupby(['PLANT', 'SKU'])['MIN_COMPLETE_TRUCKS_QTY'].sum().reset_index(),
                         on=['PLANT', 'SKU'], how='left')

    Inventory = Inventory[['RUN_DATE', 'PLANT', 'SKU', 'AVAILABLE_PLANT_STOCK', 'TODAY_PRODUCTION_STOCK',
                           'OPEN_ORDER_QTY', 'MIN_COMPLETE_TRUCKS_QTY', 'EOD_AVAILABLE_QUANTITY']].copy()
    Inventory.EOD_AVAILABLE_QUANTITY = Inventory.EOD_AVAILABLE_QUANTITY - Inventory.MIN_COMPLETE_TRUCKS_QTY.fillna(0)
    Inventory.columns = ['RUN_DATE', 'PLANT', 'SKU', 'AVAILABLE_PLANT_STOCK', 'TODAY_PRODUCTION_STOCK',
                         'OPEN_ORDER_QTY', 'MIN_COMPLETE_TRUCKS_QTY', 'AVAILABLE_QTY']
    return Inventory

def preprocess_data(merged_data, Inventory):
    """
    Preprocesses the merged_data DataFrame.
    """
    merged_data['LOAD_SIZE_MULTIPLES'] = np.where(
        merged_data['LOAD_SIZE_MULTIPLES'] == '3X+2X (MAX 13)',
        '3X+2X',
        merged_data['LOAD_SIZE_MULTIPLES']
    )

    merged_data = merged_data[['DEALER', 'PLANT', 'SKU', 'LOAD_SIZE_MULTIPLES', 'UPDATED_MIN_QTY', 'UPDATED_MAX_QTY', 'B2G_URGENCY_SCORE']].copy()
    merged_data
    merged_data = pd.merge(merged_data, Inventory[['PLANT', 'SKU', 'AVAILABLE_QTY']], on=['PLANT', 'SKU'], how='left')

    merged_data = merged_data[['DEALER', 'PLANT', 'SKU', 'LOAD_SIZE_MULTIPLES', 'UPDATED_MIN_QTY', 'UPDATED_MAX_QTY', 'AVAILABLE_QTY', 'B2G_URGENCY_SCORE']]
    merged_data.columns = ['DEALER', 'PLANT', 'SKU', 'MULTIPLIER', 'MIN_QTY', 'MAX_QTY', 'AVAILABLE_QTY', 'SCORE']
    return merged_data

def split_min_qty_into_equal_rows(min_qty):
    return [1 for _ in range(min_qty)]

def get_inventory_remaining(df, sku):
    dat = df[df['SKU'] == sku].sort_values(by='index', ascending=True)
    dat['prev_inventory_used_truck'] = dat.reset_index(drop=True).index
    dat['remaining_inventory'] = dat['AVAILABLE_QTY'] - dat['prev_inventory_used_truck']
    return dat

def fill_rows(df, max_skus, remaining_fund_dict, max_rows=10, row_filled=0, max_row_multiplier={}, is_truck_filling=False, use_max=True):
    max_row_multiplier = {}
    done_data_list = []
    rows_to_ignore = []
    ran = max_skus
    to_iterate = True
    multiplier_data = {"4X": [4] * 10, "5X": [5] * 10, "3X+2X": [3, 2] * 5}
    
    DEALER = df.DEALER.iloc[0]
    current_fund_quantity = int(np.floor(remaining_fund_dict.get(DEALER,0)/72000))
    
    if current_fund_quantity >= 3 :
        while to_iterate:
            for i in range(0, ran + 1):
                filter_df = df[~df['index'].isin(rows_to_ignore)]
                sku_row_filled = 0
                temp = filter_df[filter_df['RANK'] == i]
    
                if temp.shape[0] > 0:
                    sku = temp['SKU'].tolist()[0]
                    multiplier_order = temp['MULTIPLIER'].tolist()[0]
                    multiplier = multiplier_data[multiplier_order]
    
                    if multiplier_order in max_row_multiplier:
                        rows_done_multiplier = int(max_row_multiplier[multiplier_order])
                        multiplier = multiplier[rows_done_multiplier:]
    
                    same_multiplier_data = filter_df[filter_df['MULTIPLIER'] == multiplier_order]
                    same_multiplier_data = same_multiplier_data.sort_values(by="index", ascending=True)
                    cumulative_sum = [sum(multiplier[:i + 1]) for i in range(len(multiplier))]
    
                    same_multiplier_data['row_index'] = same_multiplier_data.reset_index(drop=True).index + 1
                    same_multiplier_data['is_modulus'] = same_multiplier_data['row_index'].apply(
                        lambda x: 1 if x in cumulative_sum else np.nan)
                    same_multiplier_data['cum_rows'] = same_multiplier_data['is_modulus'].fillna(0).cumsum()
                    same_multiplier_data['row_id'] = same_multiplier_data.apply(
                        lambda row: np.nan if pd.isnull(row['is_modulus']) else row['cum_rows'] + row_filled, axis=1)
                    same_multiplier_data['row_id'] = same_multiplier_data['row_id'].bfill()
    
                    if is_truck_filling == False:
                        valid_row_id = getmaxRow(same_multiplier_data, sku, use_max)
                        same_multiplier_data = same_multiplier_data[same_multiplier_data['row_id'].isin(valid_row_id)]
                        same_multiplier_data = same_multiplier_data[same_multiplier_data['row_id'] <= max_rows]
    
                    else:
                        max_additional_rows = row_filled + max_rows
                        same_multiplier_data = same_multiplier_data[same_multiplier_data['row_id'] <= max_additional_rows]
    
                    if same_multiplier_data.shape[0] > 0:
                        done_data_list.append(same_multiplier_data)
                        for val in same_multiplier_data['index'].tolist():
                            rows_to_ignore.append(val)
                        valid_row_id = same_multiplier_data.loc[~pd.isnull(same_multiplier_data['row_id']), 'row_id'].unique()
                        if len(valid_row_id) > 0:
                            row_filled = max(valid_row_id)
                        if multiplier_order not in max_row_multiplier:
                            max_row_multiplier[multiplier_order] = len(same_multiplier_data['row_id'].unique())
                        else:
                            max_row_multiplier[multiplier_order] += len(same_multiplier_data['row_id'].unique())
                if i == ran:
                    to_iterate = False
                if row_filled == max_rows:
                    to_iterate = False
        
            done_data = pd.DataFrame()
            remaining_data = pd.DataFrame()
            if len(done_data_list) > 0:
                done_data = pd.concat(done_data_list)
                done_data=done_data.iloc[0:current_fund_quantity].copy()
                row_filled = done_data.loc[done_data['is_modulus'] == 1, 'row_id'].max()
                row_filled = 0 if pd.isna(row_filled) else int(row_filled)
                if row_filled > 0:
                    done_data = done_data.loc[done_data.row_id <= row_filled,:].copy()
                    rows_to_ignore = done_data['index'].tolist()
                    if current_fund_quantity <=  len(done_data):
                        remaining_data = pd.DataFrame()
                    else:    
                        remaining_data = df.loc[~df['index'].isin(rows_to_ignore),['QTY','PLANT','DEALER','SKU','AVAILABLE_QTY','MULTIPLIER','SCORE','RANK','MAX_QTY','QTY_TYPE','index']].copy()     
            remaining_rows = max_rows - row_filled
    else:
        row_filled,done_data,rows_to_ignore,remaining_rows,max_row_multiplier,remaining_data = 0,pd.DataFrame(),[],10,0,pd.DataFrame()
    return row_filled, done_data, rows_to_ignore, remaining_rows, max_row_multiplier, remaining_data

def getmaxRow(df, sku, use_max=False):
    num_unique_not_null_row_id = df.loc[~pd.isnull(df['row_id']), 'row_id'].nunique()

    if num_unique_not_null_row_id > 0:
        valid_sku_types = [sku + "_MIN_QTY"]
        if use_max:
            valid_sku_types.append(sku + "_MAX_QTY")
        df['sku_qty'] = df['SKU'] + "_" + df['QTY_TYPE']
        agg_df = df.groupby("row_id").agg({
            "sku_qty": list,
        }).reset_index()
        agg_df['is_valid'] = agg_df.apply(
            lambda row: 1 if any(valid_type in row['sku_qty'] for valid_type in valid_sku_types) else 0,
            axis=1
        )
        return agg_df.loc[agg_df['is_valid'] == 1, 'row_id'].tolist()

    return [100000]

def split_and_rank_data(data):
    """
    Splits the MIN_QTY and MAX_QTY into rows and ranks them by SCORE.
    """
    df_list, df_list_2 = [], []

    max_skus = data['ORDER'].max()
    
    for i in range(0, max_skus + 1):
        temp_df = data[data['ORDER'] == i]

        min_rows = split_min_qty_into_equal_rows(int(temp_df['MIN_QTY'].tolist()[0]))
        df_1 = pd.DataFrame({'QTY': min_rows})
        append_common_columns(df_1, temp_df, "MIN_QTY")
        df_list.append(df_1)

        temp_df['REMAINING_QTY'] = temp_df['MAX_QTY'] - temp_df['MIN_QTY']
        max_rows = split_min_qty_into_equal_rows(int(temp_df['REMAINING_QTY'].tolist()[0]))
        df_2 = pd.DataFrame({'QTY': max_rows})
        append_common_columns(df_2, temp_df, "MAX_QTY")
        df_list_2.append(df_2)

    df_1 = pd.concat(df_list)
    df_2 = pd.concat(df_list_2)
    return pd.concat([df_1, df_2], axis=0)

def append_common_columns(df, temp_df, qty_type):
    """
    Appends common columns to the split rows.
    """
    df['PLANT'] = temp_df['PLANT'].tolist()[0]
    df['DEALER'] = temp_df['DEALER'].tolist()[0]
    df['SKU'] = temp_df['SKU'].tolist()[0]
    df['AVAILABLE_QTY'] = temp_df['AVAILABLE_QTY'].tolist()[0]
    df['MULTIPLIER'] = temp_df['MULTIPLIER'].tolist()[0]
    df['SCORE'] = temp_df['SCORE'].tolist()[0]
    df['RANK'] = temp_df['ORDER'].tolist()[0]
    df['MAX_QTY'] = temp_df['MAX_QTY'].tolist()[0]
    df['QTY_TYPE'] = qty_type

def process_max_qty_rows(df_min, remaining_inventory_dict, remaining_fund_dict, use_max=True):
    """
    Processes the rows with MIN_QTY and computes the results.
    """
    results_list = []
    under_order_list = []
    filtered_inventory_table = pd.DataFrame()
    incomplete_truck_remaining_data = pd.DataFrame()
    nill_truck_remaining_data = pd.DataFrame()

    while not df_min.empty:
        df_min = df_min.sort_values(by=["SCORE"], ascending=[False]).reset_index(drop=True)
        top_row = df_min.iloc[0]
        plant, dealer = top_row['PLANT'], top_row['DEALER']
        group = df_min[(df_min['PLANT'] == plant) & (df_min['DEALER'] == dealer)]
        df_min = df_min[~((df_min['PLANT'] == plant) & (df_min['DEALER'] == dealer))]
        group_data = group.copy()
       
        
        group_data['index'] = group_data.reset_index(drop=True).index
        group_data['AVAILABLE_QTY'] = group_data.apply(
            lambda row: remaining_inventory_dict.get((row['PLANT'], row['SKU']), row['AVAILABLE_QTY']), axis=1)
        inv_df_list = [get_inventory_remaining(group_data, sku) for sku in group_data['SKU'].unique()]
        group_inv_data = pd.concat(inv_df_list)
        
        inv_less_df = group_inv_data[group_inv_data['remaining_inventory'] <= 0]
        group_inv_data = group_inv_data[group_inv_data['remaining_inventory'] > 0]
     
       
        filtered_inventory_table = pd.concat([filtered_inventory_table, inv_less_df], axis=0)

        max_skus = group_inv_data['SKU'].nunique()

        group_inv_data['RANK'] = (
            pd.factorize(group_inv_data[['PLANT', 'DEALER', 'SKU']].apply(tuple, axis=1))[0]
        )
        
        if len(group_inv_data) > 0:
            row_filled, done_data, rows_to_ignore, remaining_rows, max_row_multiplier, remaining_data = fill_rows(group_inv_data, max_skus, remaining_fund_dict,use_max=True)
    
            if row_filled == 10:
                df_min = pd.concat([remaining_data, df_min], axis=0)
            elif row_filled > 0:
                incomplete_truck_remaining_data = pd.concat(
                    [incomplete_truck_remaining_data, remaining_data], axis=0
                )
            elif row_filled == 0:
                nill_truck_remaining_data = pd.concat([nill_truck_remaining_data, remaining_data], axis=0)
    
            if len(done_data) > 0:
                current_plant = done_data['PLANT'].iloc[0]
                current_dealer = done_data['DEALER'].iloc[0]
                
                existing_truck_nos = [df['Truck_no'].max() for df in results_list 
                                     if (df['PLANT'] == current_plant).all() and (df['DEALER'] == current_dealer).all()]
    
                if existing_truck_nos:
                    done_data['Truck_no'] = max(existing_truck_nos) + 1
                else:
                    done_data['Truck_no'] = 1
    
                results_list.append(done_data)
                min_inventory_group = done_data.groupby(['PLANT', 'SKU'])['remaining_inventory'].min().reset_index()
                for _, row in min_inventory_group.iterrows():
                    plant, sku, min_inventory = row['PLANT'], row['SKU'], row['remaining_inventory']
                    remaining_inventory_dict[(plant, sku)] = min_inventory - 1
                remaining_fund_dict[dealer] = remaining_fund_dict.get(dealer,0) - (len(done_data) * 72000)  
            if row_filled == 0:
                total_qty = group_inv_data['QTY'].sum()
                unique_skus = group_inv_data['SKU'].nunique()
                under_order_list.append({
                    'PLANT': plant,
                    'DEALER': dealer,
                    'STATUS': "Under-Ordered",
                    'NUNIQUE_SKU': unique_skus,
                    'SUM_QTY': total_qty
                })

    final_done_data = pd.concat(results_list) if results_list else pd.DataFrame()
    under_order_df = pd.DataFrame(under_order_list)
    return final_done_data, under_order_df, nill_truck_remaining_data, incomplete_truck_remaining_data

def Truck_correction(MinQtyFullTrucks,final_done_data):
    min_full_trucks= MinQtyFullTrucks.groupby(['PLANT','DEALER']).Truck_no.max().reset_index()
    min_full_trucks.columns = ['PLANT','DEALER','MIN_QTY_LAST_TRUCK_NO']
    
    
    df=pd.merge(final_done_data,min_full_trucks, on =['PLANT','DEALER'], how='left')
    
    df.Truck_no= df.Truck_no + df.MIN_QTY_LAST_TRUCK_NO.fillna(0)

    df= df.drop(['MIN_QTY_LAST_TRUCK_NO'],axis=1)

    return df

def main(session, config):
    # Fetch data

    FullTrucksMinQty = config['FullTrucksMinQty']
    AllProcessedTrucksMinMaxQty = config['AllProcessedTrucksMinMaxQty']
    DailyMaxQtyTrucks = config['DailyMaxQtyTrucks']
    FundBlockedPostMaxTrucks = config['FundBlockedPostMaxTrucks']
    NoRowsIncrease = config['NoRowsIncrease']
    
    AllProcessedTrucks = fetch_all_trucks_data(session)
    Inventory = fetch_inventory_data(session)
    DispatchInstances = fetch_dispatchable_data(session)
    fund_data= fetch_fund_data(session)

    

    score=DispatchInstances.groupby(['DEALER','SKU'])['B2G_URGENCY_SCORE'].mean().reset_index()

    # Get FullTrucksMinQty from config


    # Process Dispatch and Inventory Data
    DispatchInstances, MinQtyFullTrucks, filtered_trucks = preprocess_dispatch_data(DispatchInstances, AllProcessedTrucks)
    
    Inventory = preprocess_inventory_data(Inventory, filtered_trucks)
    
    FundUsedMinCompleteTrucks=MinQtyFullTrucks.groupby(['DEALER'])['QTY'].sum().reset_index().assign(QTY=lambda x: x['QTY'] * 72000)
    
    FundUsedMinCompleteTrucks.columns = ['DEALER', 'FUND_USED']
    fund_data= pd.merge(fund_data,FundUsedMinCompleteTrucks, on ='DEALER', how='left')
    fund_data['FUND_REMAINING_POST_MIN_COMPLETE_TRUCKS']  =   fund_data['FUND_AVAILABLE'] -  fund_data['FUND_USED'].fillna(0)
    
    # Drop and write to Snowflake
    drop_table(FullTrucksMinQty)
    write_to_snowflake(MinQtyFullTrucks, FullTrucksMinQty)

    with_n_rows = (
        AllProcessedTrucks
          .groupby(['PLANT', 'DEALER', 'Truck_no'])['row_id']
          .max()
          .reset_index()
          .query(f"row_id >= 10 - {NoRowsIncrease}  and row_id < 10")[['PLANT', 'DEALER']]
          .drop_duplicates()
    )
    
    # # keep only those in your dispatchable instances
    DispatchInstances = DispatchInstances.merge(
        with_n_rows,
        on=['PLANT','DEALER'],
        how='inner'
    )

    
    merged_data = preprocess_data(DispatchInstances, Inventory)
    
    # Sort and rank data
    dat = merged_data.copy()
    dat = dat.sort_values(by="SCORE", ascending=False).reset_index(drop=True)
    dat['ORDER'] = dat.index
    inventory_data = Inventory[['PLANT', 'SKU', 'AVAILABLE_QTY']].drop_duplicates()
    remaining_inventory_dict = {
        (row['PLANT'], row['SKU']): row['AVAILABLE_QTY']
        for _, row in inventory_data.iterrows()}

    remaining_fund_dict = {
        row['DEALER'] : row['FUND_REMAINING_POST_MIN_COMPLETE_TRUCKS']
        for _, row in fund_data.iterrows()}
    
    df = split_and_rank_data(dat)
    df_min = df.copy()
    
    # df_min=df_min.loc[df_min.DEALER=='12251',:]
    # Process max qty rows
    final_done_data, under_order_df, nill_truck_remaining_data, incomplete_truck_remaining_data = process_max_qty_rows(df_min, remaining_inventory_dict,remaining_fund_dict,use_max=True)

    
    # Drop and write to Snowflake
    final_done_data= Truck_correction(MinQtyFullTrucks,final_done_data)
    first_truck_no = final_done_data.groupby(
        ['PLANT','DEALER']
    )['Truck_no'].transform('min')
    
    # keep only those rows whose Truck_no equals that minimum
    final_done_data = final_done_data[first_truck_no == final_done_data['Truck_no']]
    
    row_id_max_data = final_done_data.groupby(['DEALER', 'PLANT', 'Truck_no'])['row_id'].max().reset_index()
    row_id_max_data.rename(columns={'row_id': 'row_id_max'}, inplace=True)
    
    final_done_data = final_done_data.merge(row_id_max_data, on=['DEALER', 'PLANT', 'Truck_no'], how='left')
    
    min_agg = final_done_data.groupby(['DEALER', 'PLANT', 'SKU', 'MULTIPLIER', 'Truck_no']).agg(
                                QTY=('QTY', 'sum'),
                             row_id_max=('row_id_max', 'first')).reset_index()
    min_agg['IS_COMPLETE_TRUCK'] = (min_agg['row_id_max'] == 10).astype(int)
    min_agg['RUN_DATE'] = pd.Timestamp.now().date()
    min_agg['RUN_CYCLE_ID'] = cycle_id
    min_agg['RUN_CYCLE_TIME'] = current_time
    min_agg['KEY_IDENTIFIER'] = min_agg.PLANT + '_' +  min_agg.DEALER + '_' + min_agg.Truck_no.astype(int).astype(str)
    min_agg= min_agg[[ 'RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME','KEY_IDENTIFIER','DEALER','SKU','PLANT','MULTIPLIER','Truck_no','QTY','IS_COMPLETE_TRUCK']].copy()
    min_agg.columns = ['RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME','KEY_IDENTIFIER','DEALER','SKU','PLANT','MULTIPLIER','Truck_no','UNITS_TO_BE_DISPATCHED','IS_COMPLETE_TRUCK']
    min_agg=pd.merge(min_agg,fund_data.loc[fund_data.DEALER.isin(min_agg.DEALER),['DEALER','FUND_REMAINING_POST_MIN_COMPLETE_TRUCKS']],on='DEALER',how='left')

    min_agg['CUMMULATIVE_COST']=min_agg.groupby('DEALER')['UNITS_TO_BE_DISPATCHED'].cumsum() * 72000 

    min_agg=pd.merge(min_agg,score, on =['DEALER','SKU'], how='left')
    
    agg_values = min_agg.groupby('KEY_IDENTIFIER')['B2G_URGENCY_SCORE'].agg(
                    Score_max=lambda x: x.dropna().max(),
                    Score_Median=lambda x: x.dropna().median()).fillna(0)
    
    min_agg['Max_Score'] = min_agg['KEY_IDENTIFIER'].map(agg_values['Score_max'])
    min_agg['Median_Score'] = min_agg['KEY_IDENTIFIER'].map(agg_values['Score_Median'])
    
    min_agg['Max_Score_Rank'] = min_agg.groupby(['PLANT'])['Max_Score'].rank(method='dense',ascending=False)
    min_agg['Median_Score_Rank'] = min_agg.groupby(['PLANT'])['Median_Score'].rank(method='dense',ascending=False)
   
    min_agg = min_agg.sort_values(['PLANT','Max_Score_Rank','KEY_IDENTIFIER'])

    min_agg['CUMMULATIVE_COST']=min_agg.groupby('DEALER')['UNITS_TO_BE_DISPATCHED'].cumsum() * 72000 
      
    remaining_fund_dict = pd.DataFrame([{'DEALER': dealer, 'FUND_AVAILABLE': fund} for dealer, fund in remaining_fund_dict.items()])
    
    remaining_fund_dict['RUN_DATE'] = pd.Timestamp.now().date()
    remaining_fund_dict['RUN_CYCLE_ID'] = cycle_id
    remaining_fund_dict['RUN_CYCLE_TIME'] = current_time  
    
    drop_table(AllProcessedTrucksMinMaxQty)
    write_to_snowflake(final_done_data, AllProcessedTrucksMinMaxQty)

    #drop_table(FundBlockedPostMaxTrucks)
    delete_insert(FundBlockedPostMaxTrucks, cycle_id, current_date=datetime.now().date())
    write_to_snowflake(remaining_fund_dict,FundBlockedPostMaxTrucks)

    delete_insert(DailyMaxQtyTrucks, cycle_id, current_date=datetime.now().date())
    #drop_table(DailyMaxQtyTrucks)
    write_to_snowflake(min_agg,DailyMaxQtyTrucks)

    # Return the processed data
    return DispatchInstances, Inventory,final_done_data,min_agg

if __name__ == "__main__":
    DispatchInstances, Inventory,final_done_data,min_agg = main(session, config)
#time.sleep(20)

In [ ]:
#--------------- Development-------------------

# #################################### STEP6 ##################################################


import streamlit as st
import pandas as pd
import datetime
from datetime import datetime
import numpy as np
from snowflake.snowpark.context import get_active_session
import re
import yaml
import logging
import os
import time

# suppress pandas SettingWithCopyWarning
pd.options.mode.chained_assignment = None

# Initialize logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Get active Snowflake session
session = get_active_session()

def get_run_cycle():
    """Determine current run cycle based on time of day."""
    current_time = datetime.now().time()
    if current_time >= datetime.strptime("00:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("11:00:00", "%H:%M:%S").time():
        return 1, current_time
    elif current_time >= datetime.strptime("11:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("14:00:00", "%H:%M:%S").time():
        return 2, current_time
    elif current_time >= datetime.strptime("14:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("16:00:00", "%H:%M:%S").time():
        return 3, current_time
    elif current_time >= datetime.strptime("16:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("18:00:00", "%H:%M:%S").time():
        return 4, current_time
    else:
        return 5, current_time

# Example usage
cycle_id, current_time = get_run_cycle()
print(f"Current run cycle: {cycle_id}")

# Configuration for Snowflake tables
config = {
    
    
    "ReProcessedMinTrucks" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.ReProcessedMinTrucks",
    "DailyMinQtyTrucks": "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyMinQtyTrucks",
    "DailyMaxQtyTrucks": "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyMaxQtyTrucks",
    "FundBlockedPostReProcessedTrucks": "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.FundBlockedPostReProcessedTrucks",
    "DailyReProcessedTrucks" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyReProcessedTrucks",
    "NoRowsIncrease" : 3
}

def fetch_data(query, session):
    """Execute SQL and return pandas DataFrame."""
    return session.sql(query).to_pandas()

def drop_table(table_name):
    """Drop a Snowflake table if it exists."""
    query = f'DROP TABLE IF EXISTS {table_name};'
    logger.info(f'Dropping table: {table_name}')
    session.sql(query).collect()

def delete_insert(table_name, cycle_id, current_date=pd.Timestamp.now().date()):
    """Delete existing rows for this run_date/run_cycle_id before insert."""
    query = f"""
        DELETE FROM {table_name} 
        WHERE RUN_DATE = '{current_date}' 
        AND RUN_CYCLE_ID = {cycle_id};
    """
    logger.info(f"Deleting entries from {table_name} where RUN_DATE = {current_date} and RUN_CYCLE_ID = {cycle_id}")
    session.sql(query).collect()

def write_to_snowflake(df, table_name, mode="append"):
    """Write pandas DataFrame to Snowflake table."""
    logger.info(f'Writing to Snowflake table {table_name} with mode {mode}')
    agg_data_sp = session.create_dataframe(df)
    agg_data_sp.write.mode(mode).save_as_table(table_name)

def fetch_min_trucks_data(session):
    """Fetch processed trucks data."""
    query = "SELECT * FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.AllProcessedTrucksMinQty;"
    return fetch_data(query, session)

def fetch_max_trucks_data(session):
    """Fetch processed trucks data."""
    query = "SELECT * FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.AllProcessedTrucksMinMaxQty;"
    return fetch_data(query, session)

def fetch_fund_data(session):
    """Fetch current fund availability."""
    query = """
        SELECT DEALER, OBL_CR_LIMIT_AVAIL AS FUND_AVAILABLE
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.FundBlocked t
        WHERE (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation
            )
            GROUP BY run_date
        ) 
        AND FUND_BLOCK_STATUS = 'Not Blocked';
    """
    return fetch_data(query, session)

def fetch_inventory_data(session):
    """Fetch latest inventory snapshot."""
    query = """
        SELECT * 
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.InventorySnapshot t
        WHERE (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.InventorySnapshot
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.InventorySnapshot
            )
            GROUP BY run_date
        );
    """
    return fetch_data(query, session)

def fetch_dispatchable_data(session):
    """Fetch latest dispatchable instances."""
    query = """
        SELECT * 
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances t
        WHERE (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_dispatch_prioritisation
            )
            GROUP BY run_date
        );
    """
    return fetch_data(query, session)

def preprocess_dispatch_data(DispatchInstances, AllProcessedMinTrucks, AllProcessedMinMaxTrucks):
    """Process Dispatch data."""

    # with_n_rows = (
    #     AllProcessedMinTrucks
    #       .groupby(['PLANT', 'DEALER', 'Truck_no'])['row_id']
    #       .max()
    #       .reset_index()
    #       .query("row_id = 10")[['PLANT', 'DEALER','Truck_no']]
    #       .drop_duplicates()
    # )

    with_n_rows = (
        AllProcessedMinTrucks
          .groupby(['PLANT', 'DEALER', 'Truck_no'])['row_id']
          .max()
          .reset_index()
          .query("row_id == 10")[['PLANT', 'DEALER']]
          .drop_duplicates() )
    
    AllProcessedMinTrucks=  AllProcessedMinTrucks.merge(with_n_rows,  on=['PLANT','DEALER'],
        how='inner'
    )
    
  
    AllProcessedTrucks = pd.concat([AllProcessedMinTrucks,AllProcessedMinMaxTrucks],axis=0)
    
    # max_row_ids = AllProcessedTrucks.groupby(['PLANT', 'DEALER', 'Truck_no'])['row_id'].transform('max')
    # filtered_trucks = AllProcessedTrucks[max_row_ids == 10].copy()
    filtered_trucks = AllProcessedTrucks.copy()

    filtered_trucks = filtered_trucks.groupby(['PLANT', 'SKU', 'DEALER'])['QTY'].sum().reset_index()
    filtered_trucks.columns = ['PLANT', 'SKU', 'DEALER', 'MIN_MAX_TRUCKS_QTY']
    
    
    DispatchInstances = pd.merge(DispatchInstances, filtered_trucks, on=['PLANT', 'SKU', 'DEALER'], how='left')
    DispatchInstances.MIN_MAX_TRUCKS_QTY.fillna(0, inplace=True)
    
    DispatchInstances['UPDATED_MIN_QTY'] = DispatchInstances.FINAL_B2G - DispatchInstances.MIN_MAX_TRUCKS_QTY
    DispatchInstances['UPDATED_MAX_QTY'] = np.maximum(
        DispatchInstances['UPDATED_MIN_QTY'],
        DispatchInstances['MAX_QTY'] - DispatchInstances['MIN_MAX_TRUCKS_QTY']
    )

    print(DispatchInstances.shape, "Before filtering updated max qty > 0")
    DispatchInstances = DispatchInstances.loc[DispatchInstances.UPDATED_MAX_QTY > 0, :].copy()
    # MinQtyFullTrucks = AllProcessedTrucks[max_row_ids == 10].copy()
    print(DispatchInstances.shape, "Post filtering updated max qty > 0")

    return DispatchInstances,AllProcessedTrucks, filtered_trucks

def preprocess_inventory_data(Inventory, filtered_trucks):
    """Process Inventory data."""
    Inventory = pd.merge(
        Inventory,
        filtered_trucks.groupby(['PLANT', 'SKU'])['MIN_MAX_TRUCKS_QTY'].sum().reset_index(),
        on=['PLANT', 'SKU'], how='left'
    )

    Inventory = Inventory[[
        'RUN_DATE', 'PLANT', 'SKU', 'AVAILABLE_PLANT_STOCK',
        'TODAY_PRODUCTION_STOCK', 'OPEN_ORDER_QTY',
        'MIN_MAX_TRUCKS_QTY', 'EOD_AVAILABLE_QUANTITY'
    ]].copy()

    Inventory.EOD_AVAILABLE_QUANTITY = Inventory.EOD_AVAILABLE_QUANTITY - Inventory.MIN_MAX_TRUCKS_QTY.fillna(0)
    Inventory.columns = [
        'RUN_DATE', 'PLANT', 'SKU', 'AVAILABLE_PLANT_STOCK',
        'TODAY_PRODUCTION_STOCK', 'OPEN_ORDER_QTY',
        'MIN_MAX_TRUCKS_QTY', 'AVAILABLE_QTY'
    ]
    return Inventory

def preprocess_data(merged_data, Inventory):
    """Prepare merged data for ranking."""
    merged_data['LOAD_SIZE_MULTIPLES'] = np.where(
        merged_data['LOAD_SIZE_MULTIPLES'] == '3X+2X (MAX 13)',
        '3X+2X',
        merged_data['LOAD_SIZE_MULTIPLES']
    )

    merged_data = merged_data[[
        'DEALER', 'PLANT', 'SKU', 'LOAD_SIZE_MULTIPLES',
        'UPDATED_MIN_QTY', 'UPDATED_MAX_QTY', 'B2G_URGENCY_SCORE'
    ]].copy()

    merged_data = pd.merge(
        merged_data,
        Inventory[['PLANT', 'SKU', 'AVAILABLE_QTY']],
        on=['PLANT', 'SKU'], how='left'
    )

    merged_data.columns = [
        'DEALER', 'PLANT', 'SKU', 'MULTIPLIER',
        'MIN_QTY', 'MAX_QTY', 'AVAILABLE_QTY', 'SCORE'
    ]
    return merged_data

def split_min_qty_into_equal_rows(min_qty):
    return [1 for _ in range(min_qty)]

def get_inventory_remaining(df,sku):
    dat=df[df['SKU']==sku].sort_values(by='index',ascending=True)
    dat['prev_inventory_used_truck']=dat.reset_index(drop=True).index
    dat['remaining_inventory']=dat['AVAILABLE_QTY'] - dat['prev_inventory_used_truck']
    return dat

def getmaxRow(df, sku, use_max=False):
    num_unique_not_null_row_id = df.loc[~pd.isnull(df['row_id']), 'row_id'].nunique()

    if num_unique_not_null_row_id > 0:
        valid_sku_types = [sku + "_MIN_QTY"]
        if use_max:
            valid_sku_types.append(sku + "_MAX_QTY")
        df['sku_qty'] = df['SKU'] + "_" + df['QTY_TYPE']
        agg_df = df.groupby("row_id").agg({
            "sku_qty": list,
        }).reset_index()
        agg_df['is_valid'] = agg_df.apply(
            lambda row: 1 if any(valid_type in row['sku_qty'] for valid_type in valid_sku_types) else 0,
            axis=1
        )
        return agg_df.loc[agg_df['is_valid'] == 1, 'row_id'].tolist()

    return [100000]

def fill_rows(df, max_skus, remaining_fund_dict, max_rows=10, row_filled=0, max_row_multiplier={}, is_truck_filling=False):
    
    if df is None or df.empty:
        # nothing to do, return "no trucks formed" state
        return 0, pd.DataFrame(), [], max_rows, {}, pd.DataFrame()
    
    max_row_multiplier = {}
    done_data_list = []
    rows_to_ignore = []
    ran = max_skus
    to_iterate = True
    multiplier_data={"4X":[4]*10,"5X":[5]*10,"3X+2X":[3,2]*5}
    
    DEALER = df.DEALER.iloc[0]
    current_fund_quantity = int(np.floor(remaining_fund_dict.get(DEALER,0)/72000))
    if current_fund_quantity >= 3 :
        while to_iterate:
            for i in range(0, ran + 1):
                filter_df = df[~df['index'].isin(rows_to_ignore)]
                sku_row_filled = 0
                temp = filter_df[filter_df['RANK'] == i]
              
                if temp.shape[0] > 0:
                    sku = temp['SKU'].tolist()[0]
                    multiplier_order = temp['MULTIPLIER'].tolist()[0]
                    multiplier = multiplier_data[multiplier_order]
                    
                    if multiplier_order in max_row_multiplier:
                        rows_done_multiplier = int(max_row_multiplier[multiplier_order])
                        multiplier = multiplier[rows_done_multiplier:]
                        
                    same_multiplier_data = filter_df[filter_df['MULTIPLIER'] == multiplier_order]
                    same_multiplier_data = same_multiplier_data.sort_values(by="index", ascending=True)
                    cumulative_sum = [sum(multiplier[:i + 1]) for i in range(len(multiplier))]
    
                    same_multiplier_data['row_index'] = same_multiplier_data.reset_index(drop=True).index + 1
                    same_multiplier_data['is_modulus'] = same_multiplier_data['row_index'].apply(
                        lambda x: 1 if x in cumulative_sum else np.nan)
                    same_multiplier_data['cum_rows'] = same_multiplier_data['is_modulus'].fillna(0).cumsum()
                    same_multiplier_data['row_id'] = same_multiplier_data.apply(
                        lambda row: np.nan if pd.isnull(row['is_modulus']) else row['cum_rows'] + row_filled, axis=1)
                    same_multiplier_data['row_id'] = same_multiplier_data['row_id'].bfill()
                    if is_truck_filling == False:
                        valid_row_id = getmaxRow(same_multiplier_data, sku)
                        same_multiplier_data = same_multiplier_data[same_multiplier_data['row_id'].isin(valid_row_id)]
                    if is_truck_filling == False:
                        same_multiplier_data = same_multiplier_data[same_multiplier_data['row_id'] <= max_rows]
                    else:
                        max_additional_rows = row_filled + max_rows
                        same_multiplier_data = same_multiplier_data[same_multiplier_data['row_id'] <= max_additional_rows] 
                    if same_multiplier_data.shape[0] > 0:
                        done_data_list.append(same_multiplier_data)
                        for val in same_multiplier_data['index'].tolist():
                            rows_to_ignore.append(val)
                        valid_row_id = same_multiplier_data.loc[~pd.isnull(same_multiplier_data['row_id']), 'row_id'].unique()
                        if len(valid_row_id) > 0:
                            row_filled = max(valid_row_id)
                        if multiplier_order not in max_row_multiplier:
                            max_row_multiplier[multiplier_order] = len(same_multiplier_data['row_id'].unique())
                        else:
                            max_row_multiplier[multiplier_order] += len(same_multiplier_data['row_id'].unique())               
                if i == ran:
                    to_iterate = False
                if row_filled == max_rows:
                    to_iterate = False
            done_data = pd.DataFrame()
            remaining_data = pd.DataFrame()
            if len(done_data_list) > 0:
                done_data = pd.concat(done_data_list)
                done_data=done_data.iloc[0:current_fund_quantity].copy()
                row_filled = done_data.loc[done_data['is_modulus'] == 1, 'row_id'].max()
                row_filled = 0 if pd.isna(row_filled) else int(row_filled)
                if row_filled > 0:
                    done_data = done_data.loc[done_data.row_id <= row_filled,:].copy()
                    rows_to_ignore = done_data['index'].tolist()
                    if current_fund_quantity <=  len(done_data):
                        remaining_data = pd.DataFrame()
                    else:    
                        remaining_data = df.loc[~df['index'].isin(rows_to_ignore),['QTY','PLANT','DEALER','SKU','AVAILABLE_QTY','MULTIPLIER','SCORE','RANK','MAX_QTY','QTY_TYPE','index']].copy()     
            remaining_rows = max_rows - row_filled
    else:
        row_filled,done_data,rows_to_ignore,remaining_rows,max_row_multiplier,remaining_data = 0,pd.DataFrame(),[],10,0,pd.DataFrame()
    return row_filled, done_data, rows_to_ignore, remaining_rows, max_row_multiplier, remaining_data

# def preprocess_data(merged_data):
#     """
#     Preprocesses the merged_data DataFrame.
#     """
#     merged_data['LOAD_SIZE_MULTIPLES'] = np.where(
#         merged_data['LOAD_SIZE_MULTIPLES'] == '3X+2X (MAX 13)', 
#         '3X+2X', 
#         merged_data['LOAD_SIZE_MULTIPLES']
#     )
    
#     merged_data=merged_data[['DEALER','PLANT','SKU','LOAD_SIZE_MULTIPLES','FINAL_B2G','MAX_QTY','EOD_AVAILABLE_QUANTITY','B2G_URGENCY_SCORE']].copy()
#     merged_data.columns = ['DEALER', 'PLANT', 'SKU', 'MULTIPLIER', 'MIN_QTY', 'MAX_QTY', 'AVAILABLE_QTY', 'SCORE']
#     return merged_data


def split_and_rank_data(data):
    """
    Splits the MIN_QTY and MAX_QTY into rows and ranks them by SCORE.
    """
    df_list, df_list_2 = [], []

    max_skus = data['ORDER'].max()
    for i in range(0, max_skus + 1):
        temp_df = data[data['ORDER'] == i]

        # MIN_QTY rows
        min_rows = split_min_qty_into_equal_rows(int(temp_df['MIN_QTY'].tolist()[0]))
        df_1 = pd.DataFrame({'QTY': min_rows})
        append_common_columns(df_1, temp_df, "MIN_QTY")
        df_list.append(df_1)

        # MAX_QTY rows
        temp_df['REMAINING_QTY'] = temp_df['MAX_QTY'] - temp_df['MIN_QTY']
        max_rows = split_min_qty_into_equal_rows(int(temp_df['REMAINING_QTY'].tolist()[0]))
        df_2 = pd.DataFrame({'QTY': max_rows})
        append_common_columns(df_2, temp_df, "MAX_QTY")
        df_list_2.append(df_2)
    df_1 = pd.concat(df_list)
    df_2 = pd.concat(df_list_2)
    return pd.concat([df_1, df_2], axis=0)

def append_common_columns(df, temp_df, qty_type):
    """
    Appends common columns to the split rows.
    """
    df['PLANT'] = temp_df['PLANT'].tolist()[0]
    df['DEALER'] = temp_df['DEALER'].to_list()[0]
    df['SKU'] = temp_df['SKU'].tolist()[0]
    df['AVAILABLE_QTY'] = temp_df['AVAILABLE_QTY'].tolist()[0]
    df['MULTIPLIER'] = temp_df['MULTIPLIER'].tolist()[0]
    df['SCORE'] = temp_df['SCORE'].tolist()[0]
    df['RANK'] = temp_df['ORDER'].tolist()[0]
    df['MAX_QTY'] = temp_df['MAX_QTY'].tolist()[0]
    df['QTY_TYPE'] = qty_type

def process_min_qty_rows(df_min, remaining_inventory_dict,remaining_fund_dict):
    """
    Processes the rows with MIN_QTY and computes the results.
    """
    results_list = []
    under_order_list = []
    filtered_inventory_table = pd.DataFrame()
    incomplete_truck_remaining_data = pd.DataFrame()
    nill_truck_remaining_data = pd.DataFrame()
    while not df_min.empty:
        df_min = df_min.sort_values(by=["SCORE"], ascending=[False]).reset_index(drop=True)
        top_row = df_min.iloc[0]
        plant, dealer = top_row['PLANT'], top_row['DEALER']
        group = df_min[(df_min['PLANT'] == plant) & (df_min['DEALER'] == dealer)]
        df_min = df_min[~((df_min['PLANT'] == plant) & (df_min['DEALER'] == dealer))]
        group_data = group.copy()
        group_data['index'] = group_data.reset_index(drop=True).index
        group_data['AVAILABLE_QTY'] = group_data.apply(
            lambda row: remaining_inventory_dict.get((row['PLANT'], row['SKU']), row['AVAILABLE_QTY']),axis=1)
        inv_df_list = [get_inventory_remaining(group_data, sku) for sku in group_data['SKU'].unique()]
        group_inv_data = pd.concat(inv_df_list)
        inv_less_df = group_inv_data[group_inv_data['remaining_inventory'] <= 0]
        group_inv_data = group_inv_data[group_inv_data['remaining_inventory'] > 0]
        filtered_inventory_table = pd.concat([filtered_inventory_table, inv_less_df], axis=0)
        max_skus = group_inv_data['SKU'].nunique()
        group_inv_data['RANK'] = (
            pd.factorize(group_inv_data[['PLANT', 'DEALER', 'SKU']].apply(tuple, axis=1))[0]
        )
        row_filled, done_data, rows_to_ignore, remaining_rows, max_row_multiplier, remaining_data = fill_rows(group_inv_data, max_skus,remaining_fund_dict)
        if row_filled == 10:
            df_min = pd.concat([remaining_data, df_min], axis=0)
        elif row_filled > 0:
            incomplete_truck_remaining_data = pd.concat(
                [incomplete_truck_remaining_data, remaining_data], axis=0
            )
        elif row_filled == 0:
            nill_truck_remaining_data = pd.concat([nill_truck_remaining_data, remaining_data], axis=0)
        if len(done_data) > 0:
            current_plant = done_data['PLANT'].iloc[0]
            current_dealer = done_data['DEALER'].iloc[0]
            existing_truck_nos = [ df['Truck_no'].max()
                for df in results_list
                if df['PLANT'].iloc[0] == current_plant and df['DEALER'].iloc[0] == current_dealer]          
            if existing_truck_nos:
                done_data['Truck_no'] = max(existing_truck_nos) + 1       
            else:
                done_data['Truck_no'] = 1        
            results_list.append(done_data)
            min_inventory_group = done_data.groupby(['PLANT', 'SKU'])['remaining_inventory'].min().reset_index()
            for _, row in min_inventory_group.iterrows():
                plant, sku, min_inventory = row['PLANT'], row['SKU'], row['remaining_inventory']
                remaining_inventory_dict[(plant, sku)] = min_inventory - 1
            remaining_fund_dict[dealer] = remaining_fund_dict.get(dealer,0) - (len(done_data) * 72000)              
        if row_filled == 0:
            total_qty = group_inv_data['QTY'].sum()
            unique_skus = group_inv_data['SKU'].nunique()
            under_order_list.append({
                'PLANT': plant,
                'DEALER': dealer,
                'STATUS': "Under-Ordered",
                'NUNIQUE_SKU': unique_skus,
                'SUM_QTY': total_qty
            })
    final_done_data = pd.concat(results_list) if results_list else pd.DataFrame()
    under_order_df = pd.DataFrame(under_order_list)
    
    return final_done_data, under_order_df,nill_truck_remaining_data,incomplete_truck_remaining_data,remaining_fund_dict

def Truck_correction(MinQtyFullTrucks, final_done_data):
    """Adjust truck numbers by adding min-truck offsets."""
    offsets = MinQtyFullTrucks.groupby(['PLANT','DEALER'])['Truck_no'].max().reset_index()
    offsets.columns = ['PLANT','DEALER','MIN_QTY_LAST_TRUCK_NO']
    df = pd.merge(final_done_data, offsets, on=['PLANT','DEALER'], how='left')
    df['Truck_no'] = df['Truck_no'] + df['MIN_QTY_LAST_TRUCK_NO'].fillna(0)
    return df.drop(columns=['MIN_QTY_LAST_TRUCK_NO'])


def main(session, config):
    #FullTrucksMinQty = config['ReProcessedMinTrucks']
    ReProcessedMinTrucks = config['ReProcessedMinTrucks']
    DailyMaxQtyTrucks = config['DailyMaxQtyTrucks']
    FundBlockedPostReProcessedTrucks = config['FundBlockedPostReProcessedTrucks']
    DailyReProcessedTrucks = config['DailyReProcessedTrucks']
    # Fetch source data
    NoRowsIncrease =  config['NoRowsIncrease']
    AllProcessedMinTrucks= fetch_min_trucks_data(session)
    AllProcessedMinMaxTrucks= fetch_max_trucks_data(session)
 
    Inventory = fetch_inventory_data(session)
    DispatchInstances = fetch_dispatchable_data(session)

    
    
    #=== CHANGE 1: only keep plant/dealer combos with max row_id == 9 ===
    with_n_rows = (
            AllProcessedMinTrucks
              .groupby(['PLANT', 'DEALER', 'Truck_no'])['row_id']
              .max()
              .reset_index()
              .query(f"row_id < 10 - {NoRowsIncrease}")[['PLANT', 'DEALER']]
              .drop_duplicates() )
    # keep only those in your dispatchable instances
    DispatchInstances = DispatchInstances.merge(
        with_n_rows,
        on=['PLANT','DEALER'],
        how='inner'
    )
    

    fund_data = fetch_fund_data(session)
    score = DispatchInstances.groupby(['DEALER','SKU'])['B2G_URGENCY_SCORE'].mean().reset_index()

    DispatchInstances,AllProcessedTrucks, filtered_trucks = preprocess_dispatch_data(DispatchInstances, AllProcessedMinTrucks,AllProcessedMinMaxTrucks)
    Inventory = preprocess_inventory_data(Inventory, filtered_trucks)
    
    FundUsed = filtered_trucks.groupby('DEALER')['MIN_MAX_TRUCKS_QTY'].sum().reset_index().assign(QTY=lambda x: x['MIN_MAX_TRUCKS_QTY']*72000)
    FundUsed.drop('MIN_MAX_TRUCKS_QTY',axis=1,inplace=True)
    FundUsed.columns = ['DEALER','FUND_USED']

    # use uppercase FUND_AVAILABLE now
    fund_data = pd.merge(fund_data, FundUsed, on='DEALER', how='left')
    fund_data['FUND_REMAINING_POST_MIN_COMPLETE_TRUCKS'] = fund_data['FUND_AVAILABLE'] - fund_data['FUND_USED'].fillna(0)

    # drop_table(FullTrucksMinQty)
    # write_to_snowflake(MinQtyFullTrucks, FullTrucksMinQty)

    merged = preprocess_data(DispatchInstances, Inventory)
    dat = merged.sort_values('SCORE', ascending=False).reset_index(drop=True)
    dat['ORDER'] = dat.index
    inv_data = Inventory[['PLANT','SKU','AVAILABLE_QTY']].drop_duplicates()
    rem_inv = {(r['PLANT'],r['SKU']): r['AVAILABLE_QTY'] for _,r in inv_data.iterrows()}
    rem_fund = {r['DEALER']: r['FUND_REMAINING_POST_MIN_COMPLETE_TRUCKS'] for _,r in fund_data.iterrows()}

    df = split_and_rank_data(dat)
    df_min = df.copy()
    final_done_data, under_order_df, nill_truck_remaining_data, incomplete_truck_remaining_data,remaining_fund_dict = process_min_qty_rows(df_min, rem_inv, rem_fund)
    
    final_done_data = Truck_correction(AllProcessedTrucks, final_done_data)


    # compute, for each row, the min Truck_no for its PLANT/DEALER
    first_truck_no = final_done_data.groupby(
        ['PLANT','DEALER']
    )['Truck_no'].transform('min')
    
    # keep only those rows whose Truck_no equals that minimum
    final_done_data = final_done_data[first_truck_no == final_done_data['Truck_no']]

    row_id_max_data = final_done_data.groupby(['DEALER', 'PLANT', 'Truck_no'])['row_id'].max().reset_index()
    row_id_max_data.rename(columns={'row_id': 'row_id_max'}, inplace=True)
    
    final_done_data = final_done_data.merge(row_id_max_data, on=['DEALER', 'PLANT', 'Truck_no'], how='left')
    

    # build min_agg and write outputs
    min_agg = final_done_data.groupby(
        ['DEALER','PLANT','SKU','MULTIPLIER','Truck_no'],
        as_index=False
    ).agg(
        QTY=('QTY','sum'),
        row_id_max=('row_id','max')     # <-- compute max row_id directly
    )
    min_agg['IS_COMPLETE_TRUCK'] = (min_agg['row_id_max'] == 10).astype(int)
    min_agg['RUN_DATE'] = pd.Timestamp.now().date()
    min_agg['RUN_CYCLE_ID'] = cycle_id
    min_agg['RUN_CYCLE_TIME'] = current_time
    min_agg['KEY_IDENTIFIER'] = min_agg.PLANT + '_' + min_agg.DEALER + '_' + min_agg.Truck_no.astype(str)
    min_agg= min_agg[[ 'RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME','KEY_IDENTIFIER','DEALER','SKU','PLANT','MULTIPLIER','Truck_no','QTY','IS_COMPLETE_TRUCK']].copy()
    min_agg.columns = ['RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME','KEY_IDENTIFIER','DEALER','SKU','PLANT','MULTIPLIER','Truck_no','UNITS_TO_BE_DISPATCHED','IS_COMPLETE_TRUCK']
    min_agg=pd.merge(min_agg,fund_data.loc[fund_data.DEALER.isin(min_agg.DEALER),['DEALER','FUND_REMAINING_POST_MIN_COMPLETE_TRUCKS']],on='DEALER',how='left')

    min_agg['CUMMULATIVE_COST']=min_agg.groupby('DEALER')['UNITS_TO_BE_DISPATCHED'].cumsum() * 72000 
    min_agg = pd.merge(min_agg, score, on=['DEALER','SKU'], how='left')
    agg_vals = min_agg.groupby('KEY_IDENTIFIER')['B2G_URGENCY_SCORE'].agg(
        Score_max=lambda x: x.max(),
        Score_Median=lambda x: x.median()
    ).reset_index().set_index('KEY_IDENTIFIER')
    min_agg['Max_Score'] = min_agg['KEY_IDENTIFIER'].map(agg_vals['Score_max'])
    min_agg['Median_Score'] = min_agg['KEY_IDENTIFIER'].map(agg_vals['Score_Median'])
    min_agg['Max_Score_Rank'] = min_agg.groupby('PLANT')['Max_Score'].rank(method='dense', ascending=False)
    min_agg['Median_Score_Rank'] = min_agg.groupby('PLANT')['Median_Score'].rank(method='dense', ascending=False)
    min_agg = min_agg.sort_values(['PLANT','Max_Score_Rank','KEY_IDENTIFIER'])
    
    remaining_fund_df = pd.DataFrame([
        {'DEALER':d, 'FUND_AVAILABLE':f} for d,f in rem_fund.items()
    ])
    remaining_fund_df['RUN_DATE'] = pd.Timestamp.now().date()
    remaining_fund_df['RUN_CYCLE_ID'] = cycle_id
    remaining_fund_df['RUN_CYCLE_TIME'] = current_time

    
    
    drop_table(ReProcessedMinTrucks)
    write_to_snowflake(final_done_data, ReProcessedMinTrucks)
    #drop_table(FundBlockedPostReProcessedTrucks)
    delete_insert(FundBlockedPostReProcessedTrucks, cycle_id, current_date=datetime.now().date())
    write_to_snowflake(remaining_fund_df, FundBlockedPostReProcessedTrucks)
    #drop_table(DailyReProcessedTrucks)
    delete_insert(DailyMaxQtyTrucks, cycle_id, current_date=datetime.now().date())
    write_to_snowflake(min_agg, DailyReProcessedTrucks)

    return DispatchInstances, Inventory, final_done_data, min_agg

if __name__ == "__main__":
    DispatchInstances, Inventory, final_done_data, min_agg = main(session, config)
    time.sleep(20)
    max_trucks = final_done_data.copy()
# max_trucks
# min_agg

In [ ]:
# #################################### STEP7 ##################################################



# As was discussed in the meeting held on 14th Mar'24, please find attached latitude and longitude for all the HMLC dealers that can be used for clubbing load scenarios.

# There are a few things to take care of while calculating clubbing dealers-

# 1) Clubbing of dealers is allowed within a radius of 100kms between two dealers.

# 2) Clubbing of dealers is possible within the same Area Office only, not in different area offices although the state may be the same.
# For instance, Uttar Pradesh state is divided into 3 Area Offices, viz. Noida(AO Code-1601), Lucknow(AO Code- 1401) and Varanasi (AO Code- 1701).
# Faizabad (DC 11698) is located in Lucknow office(AO Code- 1401) whereas Basti (DC 11693) is located in Varanasi Office (AO Code- 1701). Although the distance between the dealers is only 78 kms, we won't be clubbing as they both are located in different Area offices.

# 3) As a rule, we do not club dealers within 100 kms radius from the source plant and no clubbing of dealers is allowed in Delhi Area office. (AO Code- 1301).


# 4) While I am providing latitude/longitude, request to please take distance via road only.

# Also, once calculated please share with us the output data in excel format so that we can verify the data and also update in our records.


import streamlit as st
import pandas as pd
import datetime
from datetime import datetime
import numpy as np
from snowflake.snowpark.context import get_active_session
import re
import yaml
import logging
import os
import time

pd.options.mode.chained_assignment = None

# Initialize logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
session = get_active_session()

config={
    "DailyAllTrucks":"MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyAllTrucks",
     "InventorySnapshotWithTruckQty" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.InventorySnapshotWithTruckQty",
}


def get_run_cycle():
    # Get the current time
    current_time = datetime.now().time()
    # Define time ranges
    if current_time >= datetime.strptime("00:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("11:00:00", "%H:%M:%S").time():
        return 1,current_time
    elif current_time >= datetime.strptime("11:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("14:00:00", "%H:%M:%S").time():
        return 2,current_time
    elif current_time >= datetime.strptime("14:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("16:00:00", "%H:%M:%S").time():
        return 3,current_time
    elif current_time >= datetime.strptime("16:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("18:00:00", "%H:%M:%S").time():
        return 4,current_time
    else:
        return 5,current_time

        
# Example usage
cycle_id,current_time = get_run_cycle()
print(f"Current run cycle: {cycle_id}")



def fetch_data(query, session):
    """
    Fetch data from the database using a SQL query.
    :param query: SQL query to execute
    :param session: Active database session
    :return: DataFrame with fetched data
    """
    return session.sql(query).to_pandas()

def drop_table(table_name):
    query = f'DROP TABLE IF EXISTS {table_name};'
    logger.info(f'Dropping table: {table_name}')
    session.sql(query).collect()
    

def delete_insert(table_name, cycle_id, current_date=pd.Timestamp.now().date()):
    # Construct the SQL query with both conditions
    query = f"""
        DELETE FROM {table_name} 
        WHERE RUN_DATE = '{current_date}' 
        AND RUN_CYCLE_ID = {cycle_id};
    """
    # Log the action
    logger.info(f"Deleting entries from {table_name} where RUN_DATE = {current_date} and RUN_CYCLE_ID = {cycle_id}")
    # Execute the query
    session.sql(query).collect()
    #session.sql.collect()

def write_to_snowflake(df, table_name, mode="append"):     
    logger.info(f'Writing to Snowflake table {table_name} with mode {mode}')
    agg_data_sp = session.create_dataframe(df)
    agg_data_sp.write.mode(mode).save_as_table(table_name)





def fetch_full_trucks_min_qty(session):
    query = """
        SELECT * 
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.FullTrucksMinQty;
    """
    return fetch_data( query,session)

def fetch_processed_trucks_min_max_qty(session):
    query = """
        SELECT * FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.AllProcessedTrucksMinMaxQty
        union 
        SELECT * FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.ReProcessedMinTrucks 
    """
    return fetch_data(query,session)

def fetch_dispatchable_data(session):
    """Fetch unblocked data."""
    DispatchInstancesQuery = """
               SELECT * 
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances t
        WHERE (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
            )
            GROUP BY run_date
        );
    """
    return fetch_data(DispatchInstancesQuery, session)

def fetch_inventory_snapshot(session):
    query = """SELECT * 
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.InventorySnapshot t
        WHERE (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.InventorySnapshot
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.InventorySnapshot
            )
            GROUP BY run_date
        );
        """
    return fetch_data(query,session)

def fetch_fund_data(session):
    fund_dealer = """ select DEALER, OBL_CR_LIMIT_AVAIL AS fund_available
    from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.FundBlocked t
     WHERE (t.run_date, t.run_cycle_id) IN (
        SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
        WHERE run_date = (
            SELECT MAX(run_date)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
        )
        GROUP BY run_date
        ) 
        AND FUND_BLOCK_STATUS = 'Not Blocked';
        """
       
    return fetch_data(fund_dealer, session)
 
def fetch_dealer_combine(session):
    query = """
        SELECT * FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DEALER_LAT_LNG_PLANT;
    """
    dealers_data = fetch_data(query,session)

    # Filter and process the data
    dealers_data = dealers_data.loc[~(dealers_data.DEALERAREACODE == 1301)]
    valid_dealers = dealers_data.loc[dealers_data.CAN_COMBINE_DEALER == 1].copy()

    melted_df = valid_dealers.melt(
        id_vars=["DEALERCODE", "DEALERCODE_2"],
        value_vars=["HM4N_DISTANCE", "HHHG_DISTANCE", "HM5V_DISTANCE", "HHHU_DISTANCE", "HM6C_DISTANCE", "HHHD_DISTANCE"],
        var_name="PLANT",
        value_name="DISTANCE"
    )

    melted_df["PLANT"] = melted_df["PLANT"].str.replace("_DISTANCE", "")
    melted_df = melted_df[melted_df.DISTANCE >= 100]
    grouped_df = melted_df.groupby(['DEALERCODE', 'PLANT']).apply(lambda x: list(x['DEALERCODE_2'])).reset_index()
    grouped_df.columns = ['DEALER', 'PLANT', "MATCH"]

    return grouped_df

def split_trucks_by_completion(truck_data):
    """Split trucks into complete and incomplete based on row_id."""
    max_row_ids = truck_data.groupby(['PLANT', 'DEALER', 'Truck_no'])['row_id'].transform('max')
    min_max_complete_trucks = truck_data[max_row_ids == 10].copy()
    min_max_incomplete_trucks = truck_data[max_row_ids < 10].copy()

    return min_max_complete_trucks, min_max_incomplete_trucks

def map_dealer_to_row_id(df):
    """Create mappings for plant-dealer combinations to row_id and dealer."""
    df['PLANTDEALER'] = df['PLANT'] + df['DEALER'].astype(str)
    return dict(zip(df['PLANTDEALER'], df['row_id'])), dict(zip(df['PLANTDEALER'], df['DEALER']))

def find_matched_row_ids(row, plant_dealer_to_row_id):
    """Find matched row_ids for a given row."""
    if not isinstance(row['MATCH'], list):
        return None
    return [plant_dealer_to_row_id.get(row['PLANT'] + str(dealer), None) for dealer in row['MATCH']]

def process_matching_rows(df, plant_dealer_to_row_id, plant_dealer_to_dealer):
    """Process rows to find matched pairs."""
    matched_pairs = {}
    while not df.empty:
        row = df.iloc[0]
        df = df.drop(df.index[0])

        if not isinstance(row['MATCH_ROWID'], list) or row['MATCH_ROWID'] is None:
            continue

        for match_row_id in row['MATCH_ROWID']:
            if match_row_id is not None and row['row_id'] + match_row_id >= 10:
                matched_dealer = plant_dealer_to_dealer.get(row['PLANT'] + str(row['MATCH'][row['MATCH_ROWID'].index(match_row_id)]))
                matched_plant_dealer = row['PLANT'] + str(matched_dealer)
                matched_pairs[row['PLANTDEALER']] = matched_plant_dealer
                df = df[df['PLANTDEALER'] != matched_plant_dealer]
                break

        df.reset_index(drop=True, inplace=True)

    return matched_pairs

def assign_group_ids(truck_data, matched_pairs):
    """Assign group IDs based on matched pairs."""
    group_mapping = {}
    group_id = 0

    for key, value in matched_pairs.items():
        if key not in group_mapping:
            group_mapping[key] = group_id
            group_id += 1
        if value not in group_mapping:
            group_mapping[value] = group_mapping[key]

    truck_data['PLANT_DEALER'] = truck_data['PLANT'] + truck_data['DEALER']
    truck_data['GROUP_ID'] = truck_data['PLANT_DEALER'].map(group_mapping).fillna(-1).astype(int)
    truck_data.drop(columns=['PLANT_DEALER'], inplace=True)

    return truck_data.sort_values(['GROUP_ID', 'PLANT', 'DEALER', 'row_id'], ascending=[False, True, True, True])

def process_grouped_data(df):
    """Process grouped data and update row_id calculations."""
    df['max_row_id'] = 0
    df['updated_row_id'] = 0

    non_clubbed = df[df['GROUP_ID'] == -1].copy()
    clubbed = df[df['GROUP_ID'] >= 0].copy()

    for group_id in clubbed['GROUP_ID'].unique():
        group_data = clubbed[clubbed['GROUP_ID'] == group_id].copy()
        first_row = group_data.iloc[0]

        max_row_id = group_data[
            (group_data['DEALER'] == first_row['DEALER']) & 
            (group_data['PLANT'] == first_row['PLANT'])
        ]['row_id'].max()
        if len(group_data['DEALER'].unique()) > 1 :
            group_data['Truck_no'] = 1
        
        group_data['max_row_id'] = group_data.apply(
            lambda row: 0 if (row['DEALER'] == first_row['DEALER'] and row['PLANT'] == first_row['PLANT']) else max_row_id,
            axis=1
        )

        clubbed.update(group_data)
    ClubPlusNonClub = pd.concat([clubbed, non_clubbed], axis=0)
    ClubPlusNonClub['updated_row_id'] = ClubPlusNonClub['row_id'] + ClubPlusNonClub['max_row_id']
  
    ClubPlusNonClub['Truck_no'] = ClubPlusNonClub['Truck_no'] + ((ClubPlusNonClub['updated_row_id'] - 1) // 10) 
    return ClubPlusNonClub


DailyAllTrucks = config['DailyAllTrucks'] 
InventorySnapshotWithTruckQty = config['InventorySnapshotWithTruckQty']
# Fetch Data
min_full_trucks = fetch_full_trucks_min_qty(session)
min_max_trucks = fetch_processed_trucks_min_max_qty(session)
min_max_full_trucks, min_max_incomplete_trucks = split_trucks_by_completion(min_max_trucks)
dealer_combine = fetch_dealer_combine(session)
inventory_snapshot=fetch_inventory_snapshot(session)
fund_data= fetch_fund_data(session)
DispatchInstances = fetch_dispatchable_data(session)

score=DispatchInstances.groupby(['DEALER','SKU'])['B2G_URGENCY_SCORE'].mean().reset_index()
# Prepare Data
df = min_max_incomplete_trucks.groupby(['PLANT', 'DEALER']).agg({'row_id': 'max'}).reset_index()
df = pd.merge(df, dealer_combine, on=['PLANT', 'DEALER'], how='left')
plant_dealer_to_row_id, plant_dealer_to_dealer = map_dealer_to_row_id(df)

df['MATCH_ROWID'] = df.apply(find_matched_row_ids, axis=1, args=(plant_dealer_to_row_id,))
matched_pairs = process_matching_rows(df, plant_dealer_to_row_id, plant_dealer_to_dealer)

# Process Trucks
incomplete_trucks = assign_group_ids(min_max_incomplete_trucks, matched_pairs)
incomplete_trucks = process_grouped_data(incomplete_trucks)

min_full_trucks['GROUP_ID'] = -1
min_full_trucks['max_row_id'] = 0
min_full_trucks['updated_row_id'] = min_full_trucks.max_row_id + min_full_trucks.row_id


min_max_full_trucks['GROUP_ID'] = -1
min_max_full_trucks['max_row_id'] = 0
min_max_full_trucks['updated_row_id'] = min_max_full_trucks.max_row_id + min_max_full_trucks.row_id


All_trucks = pd.concat([min_full_trucks,min_max_full_trucks,incomplete_trucks],axis=0 )
groups=All_trucks.groupby('GROUP_ID').apply(
    lambda x: (
        x.PLANT.unique()[0] + '_' + str(x.DEALER.unique()[0]) + '_' +
        str(x.DEALER.unique()[1] if len(x.DEALER.unique()) > 1 else None)
    ).replace("_None", "") if x['GROUP_ID'].iloc[0] >= 0 else None
).reset_index()

if groups is not None and not groups.empty:
    groups.columns = ['GROUP_ID','KEY_IDENTIFIER']
    All_trucks = pd.merge(All_trucks, groups, on='GROUP_ID', how='left')
else:
    All_trucks['KEY_IDENTIFIER'] = None 



#All_trucks = pd.merge(All_trucks,groups , on='GROUP_ID', how='left')





All_trucks['KEY_IDENTIFIER'] =  np.where(All_trucks.GROUP_ID>=0,
                               All_trucks.KEY_IDENTIFIER + '_' + All_trucks.Truck_no.astype(int).astype(str),
                                All_trucks.PLANT + '_' + All_trucks.DEALER + '_' + All_trucks.Truck_no.astype(int).astype(str))

row_id_max_data = All_trucks.groupby(['KEY_IDENTIFIER'])['updated_row_id'].max().reset_index()
row_id_max_data.rename(columns={'updated_row_id': 'row_id_maximum'}, inplace=True)
    
All_trucks = All_trucks.merge(row_id_max_data, on=['KEY_IDENTIFIER'], how='left')

All_trucks= All_trucks.groupby(['PLANT','DEALER','SKU','MULTIPLIER','Truck_no','KEY_IDENTIFIER']).agg({'row_id_maximum':'max','QTY':'sum'}).reset_index()




All_trucks.columns = ['PLANT','DEALER','SKU','MULTIPLIER','Truck_no','KEY_IDENTIFIER','Truck_last_row_filled','QTY']





#drop_table(DailyAllTrucks)

df=All_trucks.loc[All_trucks.Truck_last_row_filled==10,:].drop_duplicates(['PLANT','DEALER','Truck_no']).groupby(['PLANT','DEALER'])['Truck_no'].nunique().reset_index()

All_trucks['RUN_DATE'] =   pd.Timestamp.now().date()
All_trucks['RUN_CYCLE_ID'] = cycle_id
All_trucks['RUN_CYCLE_TIME'] = current_time
All_trucks['IS_COMPLETE_TRUCK'] = (All_trucks['Truck_last_row_filled'] == 10).astype(int)
All_trucks= All_trucks[[ 'RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME','KEY_IDENTIFIER','DEALER','SKU','PLANT','MULTIPLIER','Truck_no','QTY','IS_COMPLETE_TRUCK']].copy()
All_trucks.columns = ['RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME','KEY_IDENTIFIER','DEALER','SKU','PLANT','MULTIPLIER','Truck_no','UNITS_TO_BE_DISPATCHED','IS_COMPLETE_TRUCK']
All_trucks=pd.merge(All_trucks,fund_data.loc[fund_data.DEALER.isin(All_trucks.DEALER),['DEALER','FUND_AVAILABLE']],on='DEALER',how='left')
All_trucks = All_trucks.sort_values(['RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME','KEY_IDENTIFIER','DEALER'])
All_trucks['CUMMULATIVE_COST']=All_trucks.groupby('DEALER')['UNITS_TO_BE_DISPATCHED'].cumsum() * 72000 

All_trucks=pd.merge(All_trucks,score, on =['DEALER','SKU'], how='left')

agg_values = All_trucks.groupby('KEY_IDENTIFIER')['B2G_URGENCY_SCORE'].agg(
                Score_max=lambda x: x.dropna().max(),
                Score_Median=lambda x: x.dropna().median()).fillna(0)

All_trucks['Max_Score'] = All_trucks['KEY_IDENTIFIER'].map(agg_values['Score_max'])
All_trucks['Median_Score'] = All_trucks['KEY_IDENTIFIER'].map(agg_values['Score_Median'])

All_trucks['Max_Score_Rank'] = All_trucks.groupby(['PLANT'])['Max_Score'].rank(method='dense',ascending=False)
All_trucks['Median_Score_Rank'] = All_trucks.groupby(['PLANT'])['Median_Score'].rank(method='dense',ascending=False)

All_trucks = All_trucks.sort_values(['PLANT','Max_Score_Rank','KEY_IDENTIFIER'])


All_trucks['CUMMULATIVE_COST']=All_trucks.groupby('DEALER')['UNITS_TO_BE_DISPATCHED'].cumsum() * 72000 


#drop_table(DailyAllTrucks)
delete_insert(DailyAllTrucks, cycle_id,current_date=datetime.now().date())
write_to_snowflake(All_trucks, DailyAllTrucks)

inventory_snapshot=pd.merge(inventory_snapshot,All_trucks.loc[All_trucks.IS_COMPLETE_TRUCK==1,:].groupby(['PLANT','SKU'])['UNITS_TO_BE_DISPATCHED'].sum().reset_index(), on =['PLANT','SKU'], how='left')
inventory_snapshot.UNITS_TO_BE_DISPATCHED.fillna(0, inplace=True)
inventory_snapshot.rename(columns={'UNITS_TO_BE_DISPATCHED': 'COMPLETE_TRUCK_UNITS_TO_BE_DISPATCHED'}, inplace=True)

inventory_snapshot=pd.merge(inventory_snapshot,All_trucks.loc[All_trucks.IS_COMPLETE_TRUCK==0,:].groupby(['PLANT','SKU'])['UNITS_TO_BE_DISPATCHED'].sum().reset_index(), on =['PLANT','SKU'], how='left')
inventory_snapshot.UNITS_TO_BE_DISPATCHED.fillna(0, inplace=True)
inventory_snapshot.rename(columns={'UNITS_TO_BE_DISPATCHED': 'INCOMPLETE_TRUCK_UNITS_TO_BE_DISPATCHED'}, inplace=True)

inventory_snapshot['RUN_DATE'] =  pd.Timestamp.now().date()
inventory_snapshot['RUN_CYCLE_ID'] = cycle_id
inventory_snapshot['RUN_CYCLE_TIME'] = current_time
#drop_table(InventorySnapshotWithTruckQty)
delete_insert(InventorySnapshotWithTruckQty,cycle_id, current_date=datetime.now().date())
write_to_snowflake(inventory_snapshot, InventorySnapshotWithTruckQty)
time.sleep(20)

In [ ]:
# #################################### STEP8 ##################################################
import streamlit as st
import pandas as pd
import datetime
from datetime import datetime, timedelta
import numpy as np
from snowflake.snowpark.context import get_active_session
import re
import yaml
import logging
import os
import time

pd.options.mode.chained_assignment = None

# Initialize logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
session = get_active_session()

holiday_data = {
    "HHHD": ["2/2/2025", "2/9/2025", "2/16/2025", "2/23/2025"],
    "HHHG": ["2/2/2025", "2/9/2025", "2/16/2025", "2/23/2025"],
    "HHHU": ["2/2/2025", "2/9/2025", "2/16/2025", "2/23/2025"],
    "HM4N": ["2/2/2025", "2/9/2025", "2/16/2025", "2/23/2025"],
    "HM5V": ["2/2/2025", "2/9/2025", "2/16/2025", "2/23/2025"],
    "HM6C": ["2/2/2025", "2/9/2025", "2/16/2025", "2/23/2025"]
}

holiday_data = {
    "HHHD": ["3/2/2025", "3/9/2025", "3/14/2025", "3/16/2025", "3/23/2025", "3/30/2025"],
    "HHHG": ["3/2/2025", "3/9/2025", "3/14/2025", "3/16/2025", "3/23/2025", "3/30/2025"],
    "HHHU": ["3/2/2025", "3/9/2025", "3/14/2025", "3/16/2025", "3/23/2025", "3/30/2025"],
    "HM4N": ["3/2/2025", "3/9/2025", "3/14/2025", "3/16/2025", "3/23/2025", "3/30/2025"],
    "HM5V": ["3/2/2025", "3/9/2025", "3/14/2025", "3/16/2025", "3/23/2025", "3/30/2025"],
    "HM6C": ["3/2/2025", "3/9/2025", "3/14/2025", "3/16/2025", "3/23/2025", "3/30/2025"]
}

holiday_data = {
    "HHHD": ["4/1/2025", "4/6/2025", "4/13/2025", "4/20/2025", "4/27/2025"],
    "HHHG": ["4/1/2025", "4/6/2025", "4/13/2025", "4/20/2025", "4/27/2025"],
    "HHHU": ["4/1/2025", "4/6/2025", "4/13/2025", "4/20/2025", "4/27/2025"],
    "HM4N": ["4/1/2025", "4/6/2025", "4/13/2025", "4/20/2025", "4/27/2025"],
    "HM5V": ["4/1/2025", "4/6/2025", "4/13/2025", "4/20/2025", "4/27/2025"],
    "HM6C": ["4/6/2025", "4/13/2025", "4/20/2025", "4/27/2025"]
}

holiday_data = {
    "HHHD": [ "5/4/2025", "5/11/2025", "5/18/2025", "5/25/2025"],
    "HHHG": [ "5/4/2025", "5/11/2025", "5/18/2025", "5/25/2025"],
    "HHHU": ["5/4/2025", "5/11/2025", "5/18/2025", "5/25/2025"],
    "HM4N": [ "5/4/2025", "5/11/2025", "5/18/2025", "5/25/2025"],
    "HM5V": [ "5/4/2025", "5/11/2025", "5/18/2025", "5/25/2025"],
    "HM6C": ["5/1/2025", "5/4/2025", "5/11/2025", "5/18/2025", "5/25/2025"]
}

holiday_data = {
    "HHHD": ["7/2/2025", "7/6/2025", "7/13/2025", "7/20/2025", "7/27/2025"],
    "HHHG": ["7/2/2025", "7/6/2025", "7/13/2025", "7/20/2025", "7/27/2025"],
    "HHHU": ["7/2/2025", "7/6/2025", "7/13/2025",
             "7/19/2025",             
             "7/20/2025",               
             "7/21/2025", "7/22/2025", "7/23/2025", "7/24/2025"],
    "HM4N": ["7/2/2025", "7/6/2025", "7/13/2025", "7/20/2025", "7/27/2025"],
    "HM5V": ["7/2/2025", "7/6/2025", "7/13/2025", "7/20/2025", "7/27/2025"],
    "HM6C": ["7/2/2025", "7/6/2025", "7/13/2025", "7/20/2025", "7/27/2025"],
}

holiday_data = {
    "HHHD": ["8/3/2025", "8/9/2025", "8/15/2025", "8/16/2025", "8/24/2025", "8/31/2025"],
    "HHHG": ["8/3/2025", "8/9/2025", "8/15/2025", "8/16/2025", "8/24/2025", "8/31/2025"],
    "HHHU": ["8/3/2025", "8/9/2025", "8/10/2025", "8/15/2025", "8/24/2025", "8/31/2025"],
    "HM4N": ["8/3/2025", "8/9/2025", "8/15/2025", "8/16/2025", "8/24/2025", "8/31/2025"],
    "HM5V": ["8/3/2025", "8/9/2025", "8/15/2025", "8/16/2025", "8/17/2025", "8/24/2025","8/31/2025"],
    "HM6C": ["8/3/2025", "8/10/2025", "8/15/2025", "8/17/2025", "8/27/2025", "8/31/2025"],
}

holiday_data = {
    "HHHD": ["9/7/2025", "9/14/2025", "9/21/2025", "9/28/2025"],
    "HHHG": ["9/7/2025", "9/14/2025", "9/21/2025", "9/28/2025"],
    "HHHU": ["9/7/2025", "9/14/2025", "9/21/2025", "9/28/2025"],
    "HM4N": ["9/7/2025", "9/14/2025", "9/21/2025", "9/28/2025"],
    "HM5V": ["9/7/2025", "9/14/2025", "9/21/2025", "9/28/2025"],
    "HM6C": ["9/7/2025", "9/14/2025", "9/21/2025", "9/28/2025"]
}

holiday_data = {

    "HHHD": ["10/2/2025", "10/10/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HHHG": ["10/2/2025", "10/10/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HHHU": ["10/2/2025", "10/10/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HM4N": ["10/2/2025", "10/10/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HM5V": ["10/2/2025", "10/12/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
    "HM6C": ["10/2/2025", "10/12/2025", "10/19/2025", "10/20/2025", 
             "10/21/2025", "10/22/2025", "10/23/2025", "10/24/2025"],
}

holiday_data = {
    "HHHD": ["11/2/2025", "11/5/2025", "11/9/2025", "11/16/2025", "11/23/2025"],
    "HHHG": ["11/2/2025", "11/5/2025", "11/9/2025", "11/16/2025", "11/23/2025"],
    "HHHU": ["11/2/2025", "11/9/2025", "11/16/2025", "11/23/2025"],
    "HM4N": ["11/2/2025", "11/9/2025", "11/16/2025", "11/23/2025"],
    "HM5V": ["11/2/2025", "11/9/2025", "11/16/2025", "11/23/2025"],
    "HM6C": ["11/2/2025", "11/9/2025", "11/16/2025", "11/23/2025"],
}

holiday_data = {
    "HHHD": ["12/1/2025", "12/7/2025", "12/14/2025", "12/21/2025", "12/28/2025", "12/29/2025", "12/30/2025", "12/31/2025"],
    "HHHG": ["12/1/2025", "12/7/2025", "12/14/2025", "12/21/2025", "12/28/2025", "12/29/2025", "12/30/2025", "12/31/2025"],
    "HHHU": ["12/1/2025", "12/7/2025", "12/14/2025", "12/21/2025", "12/28/2025", "12/29/2025", "12/30/2025", "12/31/2025"],
    "HM4N": ["12/1/2025", "12/7/2025", "12/14/2025", "12/21/2025", "12/28/2025", "12/29/2025", "12/30/2025", "12/31/2025"],
    "HM5V": ["12/1/2025", "12/7/2025", "12/14/2025", "12/21/2025", "12/28/2025", "12/29/2025", "12/30/2025", "12/31/2025"],
    "HM6C": ["12/1/2025", "12/7/2025", "12/14/2025", "12/21/2025", "12/28/2025", "12/29/2025", "12/30/2025", "12/31/2025"],
}

holiday_data = {
    "HHHD": ["1/4/2026", "1/11/2026", "1/18/2026", "1/25/2026"],
    "HHHG": ["1/4/2026", "1/11/2026", "1/18/2026", "1/25/2026"],
    "HHHU": ["1/4/2026", "1/11/2026", "1/18/2026", "1/25/2026"],
    "HM4N": ["1/4/2026", "1/11/2026", "1/18/2026", "1/25/2026"],
    "HM5V": ["1/4/2026", "1/11/2026", "1/18/2026", "1/25/2026"],
    "HM6C": ["1/4/2026", "1/11/2026", "1/18/2026", "1/25/2026"],
}

holiday_data = {
    "HHHD": ["1/1/2026", "1/2/2026", "1/3/2026", "1/11/2026", "1/18/2026", "1/26/2026"],
    "HHHG": ["1/1/2026", "1/2/2026", "1/3/2026", "1/11/2026", "1/18/2026", "1/26/2026"],
    "HHHU": ["1/1/2026", "1/2/2026", "1/3/2026", "1/11/2026", "1/18/2026", "1/26/2026"],
    "HM4N": ["1/1/2026", "1/2/2026", "1/3/2026", "1/11/2026", "1/18/2026", "1/26/2026"],
    "HM5V": ["1/1/2026", "1/2/2026", "1/11/2026", "1/14/2026", "1/15/2026", "1/18/2026", "1/26/2026"],
    "HM6C": ["1/1/2026", "1/2/2026", "1/3/2026", "1/11/2026", "1/14/2026", "1/15/2026", "1/16/2026", "1/26/2026"]
}

holiday_data = {
    "HHHD": ["2/1/2026", "2/8/2026", "2/15/2026", "2/22/2026"],
    "HHHG": ["2/1/2026", "2/8/2026", "2/15/2026", "2/22/2026"],
    "HHHU": ["2/1/2026", "2/8/2026", "2/15/2026", "2/22/2026"],
    "HM4N": ["2/1/2026", "2/8/2026", "2/15/2026", "2/22/2026"],
    "HM5V": ["2/1/2026", "2/8/2026", "2/15/2026", "2/22/2026"],
    "HM6C": ["2/1/2026", "2/8/2026", "2/15/2026", "2/22/2026"],
}


holiday_data = {
    "HHHD": ["3/1/2026", "3/4/2026", "3/8/2026", "3/15/2026", "3/22/2026", "3/29/2026"],
    "HHHG": ["3/1/2026", "3/4/2026", "3/8/2026", "3/15/2026", "3/22/2026", "3/29/2026"],
    "HHHU": ["3/1/2026", "3/4/2026", "3/8/2026", "3/15/2026", "3/22/2026", "3/29/2026"],
    "HM4N": ["3/1/2026", "3/4/2026", "3/8/2026", "3/15/2026", "3/22/2026", "3/29/2026"],
    "HM5V": ["3/1/2026", "3/4/2026", "3/8/2026", "3/15/2026", "3/22/2026", "3/29/2026"],
    "HM6C": ["3/1/2026", "3/8/2026", "3/15/2026", "3/19/2026", "3/22/2026", "3/29/2026"],
}

holiday_data = {
    "HHHD": ["4/1/2026", "4/5/2026", "4/12/2026", "4/19/2026", "4/26/2026"],
    "HHHG": ["4/1/2026", "4/5/2026", "4/12/2026", "4/19/2026", "4/26/2026"],
    "HHHU": ["4/1/2026", "4/5/2026", "4/12/2026", "4/19/2026", "4/26/2026"],
    "HM4N": ["4/1/2026", "4/5/2026", "4/12/2026", "4/19/2026", "4/26/2026"],
    "HM5V": ["4/1/2026", "4/5/2026", "4/12/2026", "4/19/2026", "4/26/2026"],
    "HM6C": ["4/1/2026", "4/5/2026", "4/12/2026", "4/19/2026", "4/26/2026"],
}

holiday_data = {
    "HHHD": ["5/3/2026", "5/10/2026", "5/17/2026", "5/24/2026", "5/31/2026"],
    "HHHG": ["5/3/2026", "5/10/2026", "5/17/2026", "5/24/2026", "5/31/2026"],
    "HHHU": ["5/3/2026", "5/10/2026", "5/17/2026", "5/24/2026", "5/31/2026"],
    "HM4N": ["5/3/2026", "5/10/2026", "5/17/2026", "5/24/2026", "5/31/2026"],
    "HM5V": ["5/3/2026", "5/10/2026", "5/17/2026", "5/24/2026", "5/31/2026"],
    "HM6C": ["5/1/2026", "5/3/2026", "5/10/2026", "5/17/2026", "5/24/2026", "5/31/2026"],
}

holiday_data = {
    "HHHD": ["7/5/2026", "7/12/2026", "7/19/2026", "7/26/2026"],
    "HHHG": ["7/5/2026", "7/12/2026", "7/19/2026", "7/26/2026"],
    "HHHU": ["7/5/2026", "7/12/2026", "7/19/2026", "7/26/2026"],
    "HM4N": ["7/5/2026", "7/12/2026", "7/19/2026", "7/26/2026"],
    "HM5V": ["7/5/2026", "7/12/2026", "7/19/2026", "7/26/2026"],
    "HM6C": ["7/5/2026", "7/12/2026", "7/19/2026", "7/26/2026"]
}

holiday_data = {
    "HHHD": ["8/2/2026", "8/9/2026", "8/16/2026", "8/23/2026","8/30/2026"],
    "HHHG": ["8/2/2026", "8/9/2026", "8/16/2026", "8/23/2026","8/30/2026"],
    "HHHU": ["8/2/2026", "8/9/2026", "8/16/2026", "8/23/2026","8/30/2026"],
    "HM4N": ["8/2/2026", "8/9/2026", "8/16/2026", "8/23/2026","8/30/2026"],
    "HM5V": ["8/2/2026", "8/9/2026", "8/16/2026", "8/23/2026","8/30/2026"],
    "HM6C": ["8/2/2026", "8/9/2026", "8/16/2026", "8/23/2026","8/30/2026"]
}

holiday_data = {
    "HHHD": ["9/6/2026", "9/13/2026", "9/20/2026", "9/27/2026"],
    "HHHG": ["9/6/2026", "9/13/2026", "9/20/2026", "9/27/2026"],
    "HHHU": ["9/6/2026", "9/13/2026", "9/20/2026", "9/27/2026"],
    "HM4N": ["9/6/2026", "9/13/2026", "9/20/2026", "9/27/2026"],
    "HM5V": ["9/6/2026", "9/13/2026", "9/20/2026", "9/27/2026"],
    "HM6C": ["9/6/2026", "9/13/2026", "9/20/2026", "9/27/2026"]
}

holiday_data = {
    "HHHD": ["10/2/2026","10/4/2026","10/11/2026", "10/18/2026","10/25/2026"],
    "HHHG": ["10/2/2026","10/4/2026","10/11/2026", "10/18/2026","10/25/2026"],
    "HHHU": ["10/2/2026","10/4/2026","10/11/2026", "10/18/2026","10/25/2026"],
    "HM4N": ["10/2/2026","10/4/2026","10/11/2026", "10/18/2026","10/25/2026"],
    "HM5V": ["10/2/2026","10/4/2026","10/11/2026", "10/18/2026","10/25/2026"],
    "HM6C": ["10/2/2026","10/4/2026","10/11/2026", "10/18/2026","10/25/2026"]
}

config={
    "DailyDealerSKUTable":"MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDealerSKUTable",
    "DailyDealerTrucksTable" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDealerTrucksTable",
    "DailyPlantTrucksTable" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyPlantTrucksTable"
}
  

def get_run_cycle():
    # Get the current time
    current_time = datetime.now().time()
    # Define time ranges
    if current_time >= datetime.strptime("00:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("11:00:00", "%H:%M:%S").time():
        return 1,current_time
    elif current_time >= datetime.strptime("11:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("14:00:00", "%H:%M:%S").time():
        return 2,current_time
    elif current_time >= datetime.strptime("14:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("16:00:00", "%H:%M:%S").time():
        return 3,current_time
    elif current_time >= datetime.strptime("16:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("18:00:00", "%H:%M:%S").time():
        return 4,current_time
    else:
        return 5,current_time

# Example usage
cycle_id,current_time = get_run_cycle()
print(f"Current run cycle: {cycle_id}")



def fetch_data(query, session):
    """
    Fetch data from the database using a SQL query.
    :param query: SQL query to execute
    :param session: Active database session
    :return: DataFrame with fetched data
    """
    return session.sql(query).to_pandas()

def drop_table(table_name):
    query = f'DROP TABLE IF EXISTS {table_name};'
    logger.info(f'Dropping table: {table_name}')
    session.sql(query).collect()
    

def delete_insert(table_name, cycle_id, current_date=pd.Timestamp.now().date()):
    # Construct the SQL query with both conditions
    query = f"""
        DELETE FROM {table_name} 
        WHERE RUN_DATE = '{current_date}' 
        AND RUN_CYCLE_ID = {cycle_id};
    """
    # Log the action
    logger.info(f"Deleting entries from {table_name} where RUN_DATE = {current_date} and RUN_CYCLE_ID = {cycle_id}")
    # Execute the query
    session.sql(query).collect()
    #session.sql.collect()

def write_to_snowflake(df, table_name, mode="append"):     
    logger.info(f'Writing to Snowflake table {table_name} with mode {mode}')
    agg_data_sp = session.create_dataframe(df)
    agg_data_sp.write.mode(mode).save_as_table(table_name)




def fetch_All_Trucks(session):
    query = """select * from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DAILYALLTRUCKS t
                 WHERE (t.run_date, t.run_cycle_id) IN (
                SELECT run_date, MAX(run_cycle_id)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
                WHERE run_date = (SELECT MAX(run_date) FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file)
                GROUP BY run_date
            );"""
    return fetch_data( query,session)

def fetch_prioritisation_table(session):
    query = """
                SELECT 
                t.RUN_DATE, 
                t.RUN_CYCLE_ID, 
                t.RUN_CYCLE_TIME, 
                t.DEALER, 
                t.SKU, 
                ROUND( AVG(t.LAST_N_DAY_SALES),3) AS LAST_N_DAY_SALES,
                ROUND(AVG(t.SAFETY_STOCK),3) AS SAFETY_STOCK,
                ROUND(AVG(t.PREDICTED_SALES_SKU),3) AS PREDICTED_SALES_SKU,
                ROUND(AVG(t.SALES_PROPORTION),3) AS SALES_PROPORTION,
                ROUND(AVG(t.AVG_LEAD_TIME_STOCK),3) AS LEAD_TIME_STOCK,
                ROUND(AVG(t.SLOPE),3) AS SLOPE,
                ROUND(AVG("expected_demand_in_day"),3) AS EXPECTED_DEMAND_REMAINDER_WEEK,
                ROUND(AVG(t.QUANTITY_DISPATCHED_TODAY),3) AS QUANTITY_DISPATCHED_TODAY,
                ROUND(AVG(t.CURRENT_AVAILABLE_STOCK),3) AS CURRENT_AVAILABLE_STOCK,
                ROUND(AVG(t.REORDER_LEVEL)) AS REORDER_LEVEL, 
                ROUND(AVG("Daily_Reorder_level_urgency"), 3) AS REORDER_URGENCY_SCORE,
                SUM(QUANTITY) as MOP_PLANNED_QUANTITY,
                
                -- SUM(B2G) as TOTAL_B2G 
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DAILY_DISPATCH_PRIORITISATION t
            WHERE (t.run_date, t.run_cycle_id) IN (
                SELECT run_date, MAX(run_cycle_id)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
                WHERE run_date = (SELECT MAX(run_date) FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file)
                GROUP BY run_date
            )
            GROUP BY t.RUN_DATE, t.RUN_CYCLE_ID, t.RUN_CYCLE_TIME, t.DEALER, t.SKU"""
    return fetch_data( query,session)


def fetch_open_order_data(session):
    query = """select DEALER, SKU, sum(TOTAL_OPEN_ORDER_QTY)  as TOTAL_OPEN_ORDER_QTY from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.OpenOrdersBlocked t
        WHERE (t.run_date, t.run_cycle_id) IN (
         SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
        WHERE run_date = (SELECT MAX(run_date) FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file)
        GROUP BY run_date) Group by 1,2;
        """
    try:
        df= fetch_data(query,session)
    except:
        df= pd.DataFrame()
    return df

def fetch_dispatch_data(session):
    query = """   select DEALER, SKU, sum(QUANTITY_DISPATCHED) as TOTAL_DISPATCH from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchTable t
             WHERE (t.run_date, t.run_cycle_id) IN (
         SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
        WHERE run_date = (SELECT MAX(run_date) FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file)
        GROUP BY run_date
            )Group by 1,2;
         """
    try:
        df= fetch_data(query,session)
        
    except:
        df= pd.DataFrame()
  
    return df

def fetch_fund_block_data(session):
    query = """  select DEALER, OBL_CR_LIMIT_AVAIL from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.FundBlocked t
             WHERE (t.run_date, t.run_cycle_id) IN (
         SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
        WHERE run_date = (SELECT MAX(run_date) FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file)
        GROUP BY run_date
            )Group by 1,2;
         """
    try:
        df= fetch_data(query,session)
        
    except:
        df= pd.DataFrame()
  
    return df

def fetch_demand_block_data(session):
    query = """  select DEALER, PLANT, DEMAND_BLOCK_STATUS from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DemandBlocked t
             WHERE (t.run_date, t.run_cycle_id) IN (
         SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
        WHERE run_date = (SELECT MAX(run_date) FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file)
        GROUP BY run_date)
         """
    try:
        df= fetch_data(query,session)
        
    except:
        df= pd.DataFrame()
  
    return df
    

    
    
def calculate_remaining_working_days_for_plants(current_date, holiday_data):
    next_month = 1 if current_date.month == 12 else current_date.month + 1
    next_year = current_date.year + 1 if current_date.month == 12 else current_date.year
    total_days_in_month = (datetime(next_year, next_month, 1) - timedelta(days=1)).day
    total_working_days_in_month = {}
    remaining_working_days = {}
    working_days_past = {}

    for plant, holidays in holiday_data.items():
        # Convert holiday strings to datetime.date objects
        holidays = [datetime.strptime(date, '%m/%d/%Y').date() for date in holidays]
        
        # Calculate total working days in the month
        total_working_days = total_days_in_month - len(
            [day for day in range(1, total_days_in_month + 1)
             if datetime(current_date.year, current_date.month, day).date() in holidays]
        )
        
        # Calculate days past as working days
        days_past = [
            day for day in range(1, current_date.day  )
            if datetime(current_date.year, current_date.month, day).date() not in holidays
        ]
        
        # Calculate remaining working days
        remaining_days_in_month = [
            day for day in range(current_date.day, total_days_in_month + 1)
            if datetime(current_date.year, current_date.month, day).date() not in holidays
        ]
        
        # Store results
        total_working_days_in_month[plant] = total_working_days
        working_days_past[plant] = len(days_past) + 1
        remaining_working_days[plant] = len(remaining_days_in_month)

    return (max(total_working_days_in_month.values(), default=0),max(working_days_past.values(), default=0),max(remaining_working_days.values(), default=0))

def calculate_prorated_quantity(row, remaining_days_map, days_gone):
 # Get the plant for the current row
    total_days_in_month = remaining_days_map  # Get remaining days for the plant
    
    if total_days_in_month == 0:  # Avoid division by zero
        return 0
    
    prorated_quantity = (row['MOP_PLANNED_QUANTITY'] / total_days_in_month) * days_gone # Calculate prorated quantity
    return prorated_quantity


def rank_with_fallback(df, urgency_col, quantity_col):
    # Separate positive and non-positive quantity values
    positive_df = df[df[quantity_col] > 0].copy()
   
    non_positive_df = df[df[quantity_col] <= 0].copy()

    # Rank the positive quantity values first
    positive_df = positive_df.sort_values(by=[urgency_col, quantity_col], ascending=[False, False])
    positive_df['MODIFIED_URGENCY_RANK'] = range(1, len(positive_df) + 1)

    # Rank the non-positive quantity values next
    non_positive_df = non_positive_df.sort_values(by=[urgency_col], ascending=[False])
    non_positive_df['MODIFIED_URGENCY_RANK'] = range(len(positive_df) + 1, len(positive_df) + len(non_positive_df) + 1)

    # Combine both dataframes
    ranked_df = pd.concat([positive_df, non_positive_df])

    return ranked_df



DailyDealerSKUTable = config['DailyDealerSKUTable'] 
DailyDealerTrucksTable = config['DailyDealerTrucksTable']
DailyPlantTrucksTable = config['DailyPlantTrucksTable']

prioritisation = fetch_prioritisation_table(session)


dispatch_data = fetch_dispatch_data(session)

OpenOrders = fetch_open_order_data(session)


if len(OpenOrders) >0 :
    prioritisation= pd.merge(prioritisation,OpenOrders, on =['DEALER','SKU'], how='left')
else:
    prioritisation['TOTAL_OPEN_ORDER_QTY'] = 0

if len(dispatch_data) >0 :
    prioritisation= pd.merge(prioritisation,dispatch_data, on =['DEALER','SKU'], how='left')
else:
    prioritisation['TOTAL_DISPATCH'] = 0


prioritisation['TOTAL_B2G'] = prioritisation.MOP_PLANNED_QUANTITY.fillna(0) -  prioritisation.TOTAL_DISPATCH.fillna(0) - prioritisation.TOTAL_OPEN_ORDER_QTY.fillna(0)
#prioritisation['TOTAL_B2G'] = prioritisation['TOTAL_B2G'] -  prioritisation['TOTAL_OPEN_ORDER_QTY'].fillna(0)
#prioritisation['TOTAL_B2G'] = np.where(prioritisation['TOTAL_B2G']  > 0, prioritisation['TOTAL_B2G'] ,0)
prioritisation['SALES_VELOCITY_NDAYS'] = np.round(
        prioritisation['LAST_N_DAY_SALES'].fillna(0) / 15, 5
    )
prioritisation['SKU_SALES_PROPORTION_OVERALL'] = np.round(
        prioritisation['LAST_N_DAY_SALES'].fillna(0) /
        prioritisation.groupby('SKU')['LAST_N_DAY_SALES'].transform('sum').fillna(1), 5
    )

beta, gamma =0.5,0.5

prioritisation['RECENCY_FACTOR'] = (
        beta * prioritisation['SALES_VELOCITY_NDAYS'] +
        (1 - beta) * prioritisation['SKU_SALES_PROPORTION_OVERALL']
    )
prioritisation['MODIFIED_REORDER_URGENCY_SCORE'] = (
        gamma * prioritisation['REORDER_URGENCY_SCORE'] +
        (1 - gamma) * prioritisation['RECENCY_FACTOR']
    )
current_date = datetime.now()
remaining_days_map, days_gone, remaining_working_days = calculate_remaining_working_days_for_plants(current_date, holiday_data)
remaining_days_map,days_gone,remaining_working_days
prioritisation['Prorated_Quantity'] = prioritisation.apply(lambda row: np.ceil(calculate_prorated_quantity(row, remaining_days_map, days_gone)), axis=1)
prioritisation['Prorated_B2G'] = prioritisation['Prorated_Quantity'].fillna(0) - prioritisation['TOTAL_DISPATCH'].fillna(0)


prioritisation= rank_with_fallback(prioritisation, 'MODIFIED_REORDER_URGENCY_SCORE', 'Prorated_B2G')

prioritisation = prioritisation.rename(columns={"MODIFIED_URGENCY_RANK" : "PRIORITY_RANK_OVERALL"})

prioritisation = prioritisation[['RUN_DATE','RUN_CYCLE_ID','RUN_CYCLE_TIME','DEALER','SKU','LAST_N_DAY_SALES','SAFETY_STOCK','PREDICTED_SALES_SKU',
                                'SALES_PROPORTION','LEAD_TIME_STOCK','SLOPE','EXPECTED_DEMAND_REMAINDER_WEEK','QUANTITY_DISPATCHED_TODAY','CURRENT_AVAILABLE_STOCK','REORDER_LEVEL',
                                'REORDER_URGENCY_SCORE','MOP_PLANNED_QUANTITY','TOTAL_DISPATCH','TOTAL_B2G','TOTAL_OPEN_ORDER_QTY','SALES_VELOCITY_NDAYS','SKU_SALES_PROPORTION_OVERALL',
                                'RECENCY_FACTOR','MODIFIED_REORDER_URGENCY_SCORE','Prorated_Quantity','Prorated_B2G','PRIORITY_RANK_OVERALL']].copy()

prioritisation.TOTAL_OPEN_ORDER_QTY.fillna(0, inplace=True)
prioritisation.TOTAL_DISPATCH.fillna(0, inplace=True)

# First, fetch the necessary data (this part is already in your code)
fund_block_data = fetch_fund_block_data(session)
demand_block_data = fetch_demand_block_data(session)


# Join fund blocked data to prioritisation dataframe on DEALER
if len(fund_block_data) > 0:
    prioritisation = pd.merge(prioritisation, fund_block_data, on='DEALER', how='left')
else:
    prioritisation['OBL_CR_LIMIT_AVAIL'] = 0

# For demand blockers, we need to pivot the data to have plants as columns
if len(demand_block_data) > 0:
    # Pivot the demand block data to have plants as columns
    sorted_plants = sorted(demand_block_data['PLANT'].unique())
    demand_block_pivoted = demand_block_data.pivot(
        index='DEALER', 
        columns='PLANT', 
        values='DEMAND_BLOCK_STATUS'
    ).reset_index()
    
    # Rename columns to indicate they are block statuses
    new_columns = {col: f'BLOCK_STATUS_{col}' for col in demand_block_pivoted.columns if col != 'DEALER'}
    demand_block_pivoted.rename(columns=new_columns, inplace=True)
    
    # Join the pivoted demand block data to the prioritisation dataframe
    prioritisation = pd.merge(prioritisation, demand_block_pivoted, on='DEALER', how='left')
    
    # Fill NaN values with 'NOT_BLOCKED' or appropriate default
    block_status_cols = [col for col in prioritisation.columns if col.startswith('BLOCK_STATUS_')]
    for col in block_status_cols:
        prioritisation[col].fillna('Not Blocked', inplace=True)
else:
    # If there's no demand block data, we don't need to add any columns
    pass


# Ensure final data is properly organized before writing to Snowflake
prioritisation['OBL_CR_LIMIT_AVAIL'].fillna(0, inplace=True)

# Update the final column list to include the new columns
final_columns = ['RUN_DATE', 'RUN_CYCLE_ID', 'RUN_CYCLE_TIME', 'DEALER', 'SKU', 
                 'LAST_N_DAY_SALES', 'SAFETY_STOCK', 'PREDICTED_SALES_SKU',
                 'SALES_PROPORTION', 'LEAD_TIME_STOCK', 'SLOPE', 'EXPECTED_DEMAND_REMAINDER_WEEK', 
                 'QUANTITY_DISPATCHED_TODAY','CURRENT_AVAILABLE_STOCK', 'REORDER_LEVEL', 'REORDER_URGENCY_SCORE', 
                 'MOP_PLANNED_QUANTITY', 'TOTAL_DISPATCH', 'TOTAL_B2G', 'TOTAL_OPEN_ORDER_QTY', 
                 'SALES_VELOCITY_NDAYS', 'SKU_SALES_PROPORTION_OVERALL', 'RECENCY_FACTOR', 
                 'MODIFIED_REORDER_URGENCY_SCORE', 'Prorated_Quantity', 'Prorated_B2G', 
                 'PRIORITY_RANK_OVERALL', 'OBL_CR_LIMIT_AVAIL']

# Add block status columns to the final columns list
if len(demand_block_data) > 0:
    block_status_cols = [f'BLOCK_STATUS_{plant}' for plant in sorted_plants]
    final_columns.extend(block_status_cols)

# Select only the columns we need for the final output
prioritisation = prioritisation[final_columns].copy()


# Now write to Snowflake as before
#drop_table(DailyDealerSKUTable)
delete_insert(DailyDealerSKUTable, cycle_id, current_date=datetime.now().date())
write_to_snowflake(prioritisation, DailyDealerSKUTable)

#drop_table(DailyDealerSKUTable)
delete_insert(DailyDealerSKUTable,cycle_id, current_date=datetime.now().date())
write_to_snowflake(prioritisation, DailyDealerSKUTable)

All_trucks=fetch_All_Trucks(session)

result = (
    All_trucks.groupby("DEALER")
    .agg(
        NO_OF_TRUCKS=("KEY_IDENTIFIER", "nunique"),
        NO_OF_COMPLETE_TRUCKS=("KEY_IDENTIFIER", lambda x: x[All_trucks["IS_COMPLETE_TRUCK"] == 1].nunique()),
        NO_OF_INCOMPLETE_TRUCKS=("KEY_IDENTIFIER", lambda x: x[All_trucks["IS_COMPLETE_TRUCK"] == 0].nunique()),
        INITIAL_FUND_AVAILABLE=("FUND_AVAILABLE", "first"),
        COMPLETE_TRUCKS_TOTAL_DISPATCHABLE_QUANTITY=("UNITS_TO_BE_DISPATCHED", lambda x: x[All_trucks["IS_COMPLETE_TRUCK"] == 1].sum() ),
        INCOMPLETE_TRUCKS_TOTAL_DISPATCHABLE_QUANTITY=("UNITS_TO_BE_DISPATCHED", lambda x: x[All_trucks["IS_COMPLETE_TRUCK"] == 0].sum()),
        COMPLETE_TRUCKS_FUND_USED=("UNITS_TO_BE_DISPATCHED", lambda x: x[All_trucks["IS_COMPLETE_TRUCK"] == 1].sum() * 72000 ),
        INCOMPLETE_TRUCKS_FUND_USED=("UNITS_TO_BE_DISPATCHED", lambda x: x[All_trucks["IS_COMPLETE_TRUCK"] == 0].sum() * 72000)
         )
).reset_index()

result2 = (
    All_trucks.groupby("PLANT")
    .agg(
        NO_OF_TRUCKS=("KEY_IDENTIFIER", "nunique"),
        NO_OF_COMPLETE_TRUCKS=("KEY_IDENTIFIER", lambda x: x[All_trucks["IS_COMPLETE_TRUCK"] == 1].nunique()),
        NO_OF_INCOMPLETE_TRUCKS=("KEY_IDENTIFIER", lambda x: x[All_trucks["IS_COMPLETE_TRUCK"] == 0].nunique()),
        INITIAL_FUND_AVAILABLE=("FUND_AVAILABLE", "first"),
        COMPLETE_TRUCKS_TOTAL_DISPATCHABLE_QUANTITY=("UNITS_TO_BE_DISPATCHED", lambda x: x[All_trucks["IS_COMPLETE_TRUCK"] == 1].sum() ),
        INCOMPLETE_TRUCKS_TOTAL_DISPATCHABLE_QUANTITY=("UNITS_TO_BE_DISPATCHED", lambda x: x[All_trucks["IS_COMPLETE_TRUCK"] == 0].sum())   )
).reset_index()

result['FUND_LEFT'] = result['INITIAL_FUND_AVAILABLE'].fillna(0) - result['COMPLETE_TRUCKS_FUND_USED'].fillna(0) - result['INCOMPLETE_TRUCKS_FUND_USED'].fillna(0)
result['RUN_DATE'] =   pd.Timestamp.now().date()
result['RUN_CYCLE_ID'] = cycle_id
result['RUN_CYCLE_TIME'] = current_time


result2['RUN_DATE'] =   pd.Timestamp.now().date()
result2['RUN_CYCLE_ID'] = cycle_id
result2['RUN_CYCLE_TIME'] = current_time

#drop_table(DailyDealerTrucksTable)
delete_insert(DailyDealerTrucksTable,cycle_id, current_date=datetime.now().date())
write_to_snowflake(result, DailyDealerTrucksTable)

#drop_table(DailyPlantTrucksTable)
delete_insert(DailyPlantTrucksTable,cycle_id, current_date=datetime.now().date())
write_to_snowflake(result2, DailyPlantTrucksTable)
time.sleep(20)


In [ ]:
# #################################### STEP9 ##################################################

import streamlit as st
import pandas as pd
import datetime
from datetime import datetime, timedelta
import numpy as np
from snowflake.snowpark.context import get_active_session
import re
import yaml
import logging
import os
import time
pd.options.mode.chained_assignment = None

# Initialize logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
session = get_active_session()

config={
    "DailyAllTrucks":"MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyAllTrucks",
     "InventorySnapshotWithTruckQty" : "MOP_DATABASE.DISPATCH_AUTOMATION_OBL.InventorySnapshotWithTruckQty",
}


def get_run_cycle():
    # Get the current time
    current_time = datetime.now().time()
    # Define time ranges
    if current_time >= datetime.strptime("00:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("11:00:00", "%H:%M:%S").time():
        return 1,current_time
    elif current_time >= datetime.strptime("11:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("14:00:00", "%H:%M:%S").time():
        return 2,current_time
    elif current_time >= datetime.strptime("14:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("16:00:00", "%H:%M:%S").time():
        return 3,current_time
    elif current_time >= datetime.strptime("16:00:00", "%H:%M:%S").time() and current_time < datetime.strptime("18:00:00", "%H:%M:%S").time():
        return 4,current_time
    else:
        return 5,current_time

# Example usage
cycle_id,current_time = get_run_cycle()
print(f"Current run cycle: {cycle_id}")

def fetch_data(query, session):
    """
    Fetch data from the database using a SQL query.
    :param query: SQL query to execute
    :param session: Active database session
    :return: DataFrame with fetched data
    """
    return session.sql(query).to_pandas()

def drop_table(table_name):
    query = f'DROP TABLE IF EXISTS {table_name};'
    logger.info(f'Dropping table: {table_name}')
    session.sql(query).collect()
    
def delete_insert(table_name, cycle_id, current_date=pd.Timestamp.now().date()):
    # Construct the SQL query with both conditions
    query = f"""
        DELETE FROM {table_name} 
        WHERE RUN_DATE = '{current_date}' 
        AND RUN_CYCLE_ID = {cycle_id};
    """
    # Log the action
    logger.info(f"Deleting entries from {table_name} where RUN_DATE = {current_date} and RUN_CYCLE_ID = {cycle_id}")
    # Execute the query
    session.sql(query).collect()
    #session.sql.collect()

def write_to_snowflake(df, table_name, mode="append"):     
    logger.info(f'Writing to Snowflake table {table_name} with mode {mode}')
    agg_data_sp = session.create_dataframe(df)
    agg_data_sp.write.mode(mode).save_as_table(table_name)


# Fetch Parent Dealer Mapping
def getParentDealerMapping():
    query = '''
    SELECT DISTINCT X_DEALER_CODE_HIER AS DEALER_CODE, PAR_ORG_NAME 
    FROM FIVETRAN_DATABASE.ORACLE_LDP_OLAP_SCHEMA.WC_INT_ORG_DH 
    WHERE X_DEALER_CODE_HIER IS NOT NULL
    '''
    parent_dealer_mapping = session.sql(query).to_pandas()
    parent_dealer_mapping['PARENT_DEALER_CODE'] = parent_dealer_mapping['PAR_ORG_NAME'].apply(lambda x: str(x).split("-")[0].strip())
    logger.info(f'Retrieved {len(parent_dealer_mapping)} parent dealer mappings')
    return parent_dealer_mapping
    
def fetchMOPData(config,parent_dealer_mapping):
    mop_table = "ANALYTICS_DATABASE.ANALYTICS_SALES.Dispatch_Plan_F_monthwise"
    CAL_DATE  = datetime.today().replace(day=1).strftime('%Y-%m-%d')
    logger.info(f'Fetching MOP Data from {mop_table}')
    query = f"""select DISTINCT DEALERCODE as DEALER_CODE, MODEL,PLANT, SKU, PLAN_QTY AS QUANTITY, VERSION    
                from {mop_table}   
                where CAL_DATE = '{CAL_DATE}' 
                and  VERSION = (
                select distinct version from {mop_table} where  CAL_DATE = '{CAL_DATE}' order by version desc limit 1);"""
    
    mop_data = session.sql(query).to_pandas()
    print(f"list of DEALER WITH NO PARENT DEALERS :\n", mop_data.loc[~mop_data.DEALER_CODE.isin(parent_dealer_mapping.DEALER_CODE),'DEALER_CODE'] )
    print(os.getcwd())
    #mop_data.loc[~(mop_data.DEALER_CODE.isin(parent_dealer_mapping.DEALER_CODE)),'DEALER_CODE'].to_csv('snow://notebook/MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DISPATCH_DEMAND_20240923/missing_delaers.csv', index=False)
    logger.info(f'Retrieved missing dealers with parent dealers to missing_dealers.csv file')
    
    #print(mop_data.loc[~(mop_data.DEALER_CODE.isin(parent_dealer_mapping.DEALER_CODE)),'DEALER_CODE'].shape)
   # --- CODE EDIT STARTS ---
    mop_data = pd.merge(mop_data, parent_dealer_mapping[['DEALER_CODE', 'PARENT_DEALER_CODE']], on="DEALER_CODE", how="left")
    
    null_before = mop_data['PARENT_DEALER_CODE'].isna().sum()
    print(f"Number of rows with NULL PARENT_DEALER_CODE before fallback: {null_before}")
    
    dealer_code_str = mop_data['DEALER_CODE'].astype(str)
    condition = dealer_code_str.str.startswith('1') & mop_data['PARENT_DEALER_CODE'].isna()
    mop_data.loc[condition, 'PARENT_DEALER_CODE'] = mop_data.loc[condition, 'DEALER_CODE']
    
    null_after = mop_data['PARENT_DEALER_CODE'].isna().sum()
    print(f"Number of rows with NULL PARENT_DEALER_CODE after fallback: {null_after}")

    if null_after>0:
        # mop_data = mop_data.dropna(subset=['PARENT_DEALER_CODE']).reset_index(drop=True)
        print(f"MOP_DATA table still has null values in parent dealer code")
   
    mop_data = mop_data.groupby(['DEALER_CODE', 'SKU', 'PLANT','VERSION', 'MODEL'])['QUANTITY'].sum().reset_index()
    mop_data.columns = ['DEALER', 'SKU','PLANT','VERSION','MODEL', 'QUANTITY']
    mop_data.DEALER = mop_data.DEALER.astype(int).astype(str)
    logger.info(f'Retrieved {len(mop_data)} rows of MOP data')
    
    return mop_data


def fetch_DealerSKU(session):
    query = """
          select * from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDealerSKUTable t
    WHERE (t.run_date, t.run_cycle_id) IN (
        SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
        WHERE run_date = (
            SELECT MAX(run_date)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
        )
        GROUP BY run_date
        ) ;
    """
    return fetch_data( query,session)

def fetch_DealerSKU_prev(session):
    query = """
          select * from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDealerSKUTable t
    WHERE (t.run_date, t.run_cycle_id) IN (
        SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
        WHERE run_date = (
            SELECT MAX(run_date) -1
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
        )
        GROUP BY run_date
        ) ;
    """
    return fetch_data( query,session)

def fetch_Dispatchable(session):
    query = """
          select RUN_DATE, DEALER, SKU, PLANT, QUANTITY, MODEL, VERSION, PREDICTED_SALES_SKU, AVG_LEAD_TIME_STOCK AS LEAD_TIME_STOCK,
         SAFETY_STOCK, REORDER_LEVEL, CURRENT_AVAILABLE_STOCK, REORDER_URGENCY_SCORE, 
         PER_DAY_PREDICTED_SALES,last_n_day_per_day_sales, SLOPE, DEMAND_FACTOR_DAILY, 
        "expected_demand_per_day", EXPECTED_DEMAND_REMAINDER_WEEK, ROUND("available_stock_daily") as available_stock_daily, "Daily_Reorder_level_urgency",
        sales_proportion, current_month_sales, last_n_day_sales, Quantity_dispatched, Total_open_order_qty, B2g, Urgency_rank, run_cycle_id, load_size_multiples, available_plant_stock, today_production_stock, eod_available_quantity,  
        sku_sales_proportion_overall, recency_factor, modified_reorder_urgency_score, modified_reorder_urgency_score_plant_proportion, b2g_proportion, b2g_urgency_score, normalised_b2g_urgency_score, final_b2g,
         "Prorated_Quantity", max_qty, priority_rank_overall, n, prority_rank_plant_wise from  MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances t
                where (t.run_date, t.run_cycle_id) IN (
                    SELECT run_date, MAX(run_cycle_id)
                    FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
                    WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        ) ;
    """
    return fetch_data( query,session)


def fetch_propotionedb2glessthanzero(session):
    query = """
          select RUN_DATE, DEALER, SKU, PLANT, QUANTITY, MODEL, VERSION, PREDICTED_SALES_SKU, AVG_LEAD_TIME_STOCK AS LEAD_TIME_STOCK,
             SAFETY_STOCK, REORDER_LEVEL, CURRENT_AVAILABLE_STOCK, Quantity_dispatched, Total_open_order_qty, B2g, Urgency_rank, run_cycle_id, load_size_multiples, available_plant_stock, today_production_stock, open_order_qty, eod_available_quantity,  
            sku_sales_proportion_overall, recency_factor, modified_reorder_urgency_score, modified_reorder_urgency_score_plant_proportion, b2g_proportion, b2g_urgency_score, normalised_b2g_urgency_score, final_b2g
             from  MOP_DATABASE.DISPATCH_AUTOMATION_OBL.propotionedb2glessthanzero t
                    where (t.run_date, t.run_cycle_id) IN (
                        SELECT run_date, MAX(run_cycle_id)
                        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        ) ;
    """
    return fetch_data( query,session)


def fetch_MinTrucks(session):
    query = """
          select * from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DAILYMINQTYTRUCKS t
     --and  IS_COMPLETE_TRUCK = 1
     where (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        ) ;
    """
    return fetch_data( query,session)


def fetch_MaxTrucks(session):
    query = """
          select * from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DAILYMAXQTYTRUCKS t
     --and  IS_COMPLETE_TRUCK = 1
     where (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        ) ;

    """
    return fetch_data( query,session)


def fetch_AllTrucks(session):
    query = """
          select * from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DAILYALLTRUCKS t
     --and  IS_COMPLETE_TRUCK = 1
     where (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        ) ;
    """
    return fetch_data( query,session)


def fetch_DealerTruckSummary(session):
    query = """
           select * from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDealerTrucksTable t
          WHERE (t.run_date, t.run_cycle_id) IN (
        SELECT run_date, MAX(run_cycle_id)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
        WHERE run_date = (
            SELECT MAX(run_date)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
        )
        GROUP BY run_date
        ) ;
    """
    return fetch_data( query,session)


def fetch_InvSnapshot(session):
    query = """
          select * from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.INVENTORYSNAPSHOTWITHTRUCKQTY t
        where  (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id), 
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        ) ;
    """
    return fetch_data( query,session)

def fetch_Fund(session):
    query = """
            select * from MOP_DATABASE.DISPATCH_AUTOMATION_OBL.FundBlocked t
        --where FUND_BLOCK_STATUS = 'Blocked'
        where  (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        );
    """
    return fetch_data( query,session)


def fetch_Demandblock(session):
    query = """
           select * from  MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DemandBlocked t
       -- where DEMAND_BLOCK_STATUS = 'Blocked'
        where  (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        );
    """
    return fetch_data( query,session)

def fetch_OpenOrder(session):
    query = """
          
          select * from  MOP_DATABASE.DISPATCH_AUTOMATION_OBL.openordersblocked t
       -- where OPEN_ORDER_BLOCK_STATUS = 'Blocked'
        where  (t.run_date, t.run_cycle_id) IN (
            SELECT run_date, MAX(run_cycle_id)
            FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            WHERE run_date = (
                SELECT MAX(run_date)
                FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
            )
            GROUP BY run_date
        );
    """
    return fetch_data( query,session)

def fetchDispatchData(config, parent_dealer_mapping):
    CALDAY = datetime.today().replace(day=1).strftime('%Y%m%d')
    yday = (datetime.today() - timedelta(days=1)).strftime('%Y%m%d')

   
    
    query = f"""select SOLD_TO as DEALER_CODE,  SKU , PLANT, sum(QUANT_B) as Quantity_dispatched 
        from ANALYTICS_DATABASE.ANALYTICS_SALES.FINAL_DISPATCH_VIEW where CALDAY >= {CALDAY}  group by 1,2,3;"""
    
    try :
        Dispatch_data = session.sql(query).to_pandas()
        parent_dealer_mapping=getParentDealerMapping()
        if Dispatch_data.shape[0] > 0:
            Dispatch_data.DEALER_CODE = Dispatch_data.DEALER_CODE.astype(int).astype(str)
            Dispatch_data = pd.merge(Dispatch_data, parent_dealer_mapping[['DEALER_CODE', 'PARENT_DEALER_CODE']], on="DEALER_CODE", how="inner")
            Dispatch_data = Dispatch_data.groupby(['DEALER_CODE', 'SKU','PLANT'])['QUANTITY_DISPATCHED'].sum().reset_index()
            Dispatch_data.columns = ['DEALER', 'SKU', 'PLANT', 'QUANTITY_DISPATCHED']
        #Dispatch_data.DEALER = Dispatch_data.DEALER.astype(int).astype(str)
            logger.info(f'Retrieved {len(Dispatch_data)} rows of Dispatch data')
            return Dispatch_data
        else:
            Dispatch_data = pd.DataFrame(data= None, columns = ['DEALER', 'SKU', 'PLANT' ,'QUANTITY_DISPATCHED'])
            return Dispatch_data  
    except:
        print(" Dispatch  data not present")
        Dispatch_data = pd.DataFrame(data= None, columns = ['DEALER', 'SKU', 'PLANT' ,'QUANTITY_DISPATCHED'])
        return Dispatch_data      


def fetch_plant_inventory(session):
    """Fetch plant inventory data."""
    plant_inventory_query = """
        SELECT a.PLANT AS PLANT, a.SKU, SUM(a.QTY) AS AVAILABLE_PLANT_STOCK 
    FROM SAP_DATA_DATABASE.SAP_DATA.VW_PLANT_INVENTORY a
    group by 1,2;
    """
    return fetch_data(plant_inventory_query, session)

def fetch_today_production_stock(session):
    """Fetch today's production stock data."""
    plant_plan_today_query = """
        SELECT WERKS AS PLANT, MATNR AS SKU, SUM(ZQTY) AS TODAY_PRODUCTION_STOCK
        FROM SAP_DATA_DATABASE.SAP_DATA.ZEMFT025 main
        WHERE ZDATE = TO_CHAR(CURRENT_DATE, 'yyyymmdd')
          AND ZVERSION = (
            SELECT MAX(ZVERSION)
            FROM SAP_DATA_DATABASE.SAP_DATA.ZEMFT025 sub
            WHERE sub.ZDATE = main.ZDATE
              AND sub.WERKS = main.WERKS
              AND sub.MATNR = main.MATNR
        )
        GROUP BY 1, 2;
    """
    return fetch_data(plant_plan_today_query, session)    

def open_order_block(session):

    # Filter rows where B2G > 5 and select unique combinations of SKU, PLANT, and DEALER
    yday = (datetime.today() - timedelta(days=1)).strftime('%Y%m%d')
    CALDAY = datetime.today().replace(day=1).strftime('%Y%m%d')
    
    query = f"""
        SELECT PLANT, DEALER_CODE AS DEALER, SKU, SUM(TOTAL_OPEN_ORDER_QTY) AS TOTAL_OPEN_ORDER_QTY
        FROM SAP_DATA_DATABASE.SAP_DATA.VW_OPEN_ORDERS_LIVE
        WHERE total_open_order_qty > 0 
        AND open_order_date >= {CALDAY} 
        --and len(DEALER_CODE) == 5
        GROUP BY 1, 2, 3;
    """
    print(query)
    try :
        open_order_block = session.sql(query).to_pandas()
        open_order_block.DEALER =  open_order_block.DEALER .astype(str)
        if open_order_block.shape[0] > 0 :
            open_order_block['OPEN_ORDER_BLOCK_STATUS'] = 'Blocked'
        else :
            open_order_block['OPEN_ORDER_BLOCK_STATUS'] = None   
        return open_order_block
       
    except:
        print(" Open orders data not present")
        open_order_block = pd.DataFrame(data= None, columns = ['PLANT', 'DEALER', 'SKU' ,'TOTAL_OPEN_ORDER_QTY','OPEN_ORDER_BLOCK_STATUS'])
        
        return open_order_block


def fetch_all_data(session):
    parent_dealer_mapping= getParentDealerMapping()
    return {
        
        "DealerSKU": fetch_DealerSKU(session),
        "DealerSKU_prev": fetch_DealerSKU_prev(session),
        "Dispatchable": fetch_Dispatchable(session),
        "PropotionedB2GLessThanZero": fetch_propotionedb2glessthanzero(session),
        "MinTrucks": fetch_MinTrucks(session),
        "MaxTrucks": fetch_MaxTrucks(session),
        "AllTrucks": fetch_AllTrucks(session),
        "DealerTruckSummary": fetch_DealerTruckSummary(session),
        "InvSnapshot": fetch_InvSnapshot(session),
        "FundBlocked": fetch_Fund(session),
        "DemandBlocked": fetch_Demandblock(session),
        "OpenOrder": fetch_OpenOrder(session),
         "SourceDispatch" : fetchDispatchData(session,parent_dealer_mapping),
         "SourceMop" : fetchMOPData(config,parent_dealer_mapping),
        "SourcePlantInv" : fetch_plant_inventory(session),
        "SourcePlantProd" :  fetch_today_production_stock(session),
        "SourceOpenOrders": open_order_block(session)
    }


def check_dealer_sku_validity(dataframes):
    """
    Ensures that all (DEALER, SKU) combinations in Dispatchable are present in DealerSKU.
    """
    dispatchable_df = dataframes["Dispatchable"]
    dealer_sku_df = dataframes["DealerSKU"]

    # Extract (DEALER, SKU) combinations
    dispatchable_combinations = set(dispatchable_df[["DEALER", "SKU"]].itertuples(index=False, name=None))
    dealer_sku_combinations = set(dealer_sku_df[["DEALER", "SKU"]].itertuples(index=False, name=None))

    # Find invalid entries
    invalid_combinations = dispatchable_combinations - dealer_sku_combinations

    if invalid_combinations:
        print(f"Error: The following DEALER-SKU combinations in Dispatchable are not in DealerSKU:\n{invalid_combinations}")
        return False
    else:
        print("Check passed: All DEALER-SKU combinations in Dispatchable exist in DealerSKU.")
        return True

def check_trucks_dealer_sku_validity(dataframes, truck_type):
    """
    Ensures that all (DEALER, SKU) combinations in the given truck dataframe
    (MinTrucks, MaxTrucks, or AllTrucks) are present in DealerSKU.
    """
    truck_df = dataframes[truck_type]
    dealer_sku_df = dataframes["DealerSKU"]

    # Extract (DEALER, SKU) combinations
    truck_combinations = set(truck_df[["DEALER", "SKU"]].itertuples(index=False, name=None))
    dealer_sku_combinations = set(dealer_sku_df[["DEALER", "SKU"]].itertuples(index=False, name=None))

    # Find invalid entries
    invalid_combinations = truck_combinations - dealer_sku_combinations

    if invalid_combinations:
        print(f"Error: The following DEALER-SKU combinations in {truck_type} are not in DealerSKU:\n{invalid_combinations}")
        return False
    else:
        print(f"Check passed: All DEALER-SKU combinations in {truck_type} exist in DealerSKU.")
        return True

def check_trucks_dispatchable_validity(dataframes, truck_type):
    """
    Ensures that all (DEALER, SKU) combinations in the given truck dataframe
    (MinTrucks, MaxTrucks, or AllTrucks) are present in DealerSKU.
    """
    truck_df = dataframes[truck_type]
    dispatchable = dataframes["Dispatchable"]

    # Extract (DEALER, SKU) combinations
    truck_combinations = set(truck_df[["DEALER", "SKU"]].itertuples(index=False, name=None))
    dealer_sku_combinations = set(dispatchable[["DEALER", "SKU"]].itertuples(index=False, name=None))

    # Find invalid entries
    invalid_combinations = truck_combinations - dealer_sku_combinations

    if invalid_combinations:
        print(f"Error: The following DEALER-SKU combinations in {truck_type} are not in DispatchableInstances:\n{invalid_combinations}")
        return False
    else:
        print(f"Check passed: All DEALER-SKU combinations in {truck_type} exist in DispatchableInstances.")
        return True


def validate_data_relationships(dataframes):
    """
    Validates relationships between data from different sources to ensure consistency.
    Checks that derived values don't exceed source values where applicable.
    """
    validation_results = []
    
    # Run each validation check and collect results
    validation_results.append(validate_dispatch_quantities(dataframes))
    validation_results.append(validate_open_order_quantities(dataframes))
    validation_results.append(validate_available_plant_stock(dataframes))
    validation_results.append(validate_production_stock(dataframes))
    validation_results.append(validate_dealer_sku_open_orders(dataframes))
    validation_results.append(validate_dealer_sku_dispatch(dataframes))
    validation_results.append(validate_dispatch_vs_inventory(dataframes))
    validation_results.append(validate_mop_vs_dispatchable(dataframes))
    validation_results.append(validate_alltrucks_dispatch_range(dataframes))
    validation_results.append(validate_mintrucks_dispatch_range(dataframes))
    
    # Create summary dataframe
    validation_df = pd.DataFrame(validation_results)
    
    # Print summary report
    print_validation_summary(validation_df)
    
    # Determine overall pass/fail status
    overall_status = "PASS" if all(result['status'] == "PASS" for result in validation_results) else "FAIL"
    print(f"Overall Validation Status: {overall_status}")
    
    return validation_df, overall_status == "PASS"


def validate_dispatch_quantities(dataframes):
    """Validates that Dispatchable.QUANTITY_DISPATCHED matches SourceDispatch.QUANTITY_DISPATCHED"""
    try:
        # Aggregate and compare at DEALER+SKU+PLANT level
        dispatchable_dispatch = dataframes['Dispatchable'].groupby(['DEALER', 'SKU', 'PLANT'])['QUANTITY_DISPATCHED'].sum().reset_index()
        source_dispatch = dataframes['SourceDispatch'].groupby(['DEALER', 'SKU', 'PLANT'])['QUANTITY_DISPATCHED'].sum().reset_index()
        
        merged_dispatch = dispatchable_dispatch.merge(source_dispatch, on=['DEALER', 'SKU', 'PLANT'], how='left', 
                                                      suffixes=('_dispatchable', '_source'))
        merged_dispatch['_source'] = merged_dispatch['QUANTITY_DISPATCHED_source'].fillna(0)
        merged_dispatch['is_valid'] = merged_dispatch['QUANTITY_DISPATCHED_dispatchable'] == merged_dispatch['_source']
        merged_dispatch.loc[ merged_dispatch['is_valid']==False]
        dispatchable_dispatch_valid = merged_dispatch['is_valid'].all()
        invalid_count = (~merged_dispatch['is_valid']).sum()
        
        return {
            'test_case': 'Dispatchable.QUANTITY_DISPATCHED == SourceDispatch.QUANTITY_DISPATCHED',
            'status': 'PASS' if dispatchable_dispatch_valid else 'FAIL',
            'notes': f"All records valid" if dispatchable_dispatch_valid else f"{invalid_count} records invalid"
        }
    except Exception as e:
        return {
            'test_case': 'Dispatchable.QUANTITY_DISPATCHED == SourceDispatch.QUANTITY_DISPATCHED',
            'status': 'ERROR',
            'notes': f"Error during validation: {str(e)}"
        }


def validate_open_order_quantities(dataframes):
    """Validates that Dispatchable.TOTAL_OPEN_ORDER_QTY matches SourceOpenOrder.TOTAL_OPEN_ORDER_QTY"""
    try:
        dispatchable_open_orders = dataframes['Dispatchable'].groupby(['DEALER', 'SKU', 'PLANT'])['TOTAL_OPEN_ORDER_QTY'].sum().reset_index()
        source_open_orders = dataframes['SourceOpenOrders'].groupby(['DEALER', 'SKU', 'PLANT'])['TOTAL_OPEN_ORDER_QTY'].sum().reset_index()
        
        merged_orders = dispatchable_open_orders.merge(source_open_orders, on=['DEALER', 'SKU', 'PLANT'], how='left', 
                                                      suffixes=('_dispatchable', '_source'))
        merged_orders['_source'] = merged_orders['TOTAL_OPEN_ORDER_QTY_source'].fillna(0)
        merged_orders['is_valid'] = merged_orders['TOTAL_OPEN_ORDER_QTY_dispatchable'] == merged_orders['_source']
        
        open_orders_valid = merged_orders['is_valid'].all()
        invalid_count = (~merged_orders['is_valid']).sum()
        
        return {
            'test_case': 'Dispatchable.TOTAL_OPEN_ORDER_QTY == SourceOpenOrder.TOTAL_OPEN_ORDER_QTY',
            'status': 'PASS' if open_orders_valid else 'FAIL',
            'notes': f"All records valid" if open_orders_valid else f"{invalid_count} records invalid"
        }
    except Exception as e:
        return {
            'test_case': 'Dispatchable.TOTAL_OPEN_ORDER_QTY == SourceOpenOrder.TOTAL_OPEN_ORDER_QTY',
            'status': 'ERROR',
            'notes': f"Error during validation: {str(e)}"
        }


def validate_available_plant_stock(dataframes):
    """Validates that Dispatchable.AVAILABLE_PLANT_STOCK matches SourcePlantInv.AVAILABLE_PLANT_STOCK"""
    try:
        dispatchable_plant_inv = dataframes['Dispatchable'].groupby(['SKU', 'PLANT'])['AVAILABLE_PLANT_STOCK'].mean().reset_index()
        source_plant_inv = dataframes['SourcePlantInv'].groupby(['SKU', 'PLANT'])['AVAILABLE_PLANT_STOCK'].sum().reset_index()
        
        merged_inv = dispatchable_plant_inv.merge(source_plant_inv, on=['SKU', 'PLANT'], how='left', 
                                                 suffixes=('_dispatchable', '_source'))
        merged_inv['_source'] = merged_inv['AVAILABLE_PLANT_STOCK_source'].fillna(0)
        merged_inv['is_valid'] = merged_inv['AVAILABLE_PLANT_STOCK_dispatchable'] == merged_inv['_source']
        
        plant_inv_valid = merged_inv['is_valid'].all()
        invalid_count = (~merged_inv['is_valid']).sum()
        
        return {
            'test_case': 'Dispatchable.AVAILABLE_PLANT_STOCK == SourcePlantInv.AVAILABLE_PLANT_STOCK',
            'status': 'PASS' if plant_inv_valid else 'FAIL',
            'notes': f"All records valid" if plant_inv_valid else f"{invalid_count} records invalid"
        }
    except Exception as e:
        return {
            'test_case': 'Dispatchable.AVAILABLE_PLANT_QUANTITY == SourcePlantInv.AVAILABLE_PLANT_STOCK',
            'status': 'ERROR',
            'notes': f"Error during validation: {str(e)}"
        }


def validate_production_stock(dataframes):
    """Validates that Dispatchable.TODAY_PRODUCTION_STOCK matches SourcePlantProd.TODAY_PRODUCTION_STOCK"""
    try:
        dispatchable_plant_inv = dataframes['Dispatchable'].groupby(['SKU', 'PLANT'])['TODAY_PRODUCTION_STOCK'].mean().reset_index()
        source_plant_prod = dataframes['SourcePlantProd'].groupby(['SKU', 'PLANT'])['TODAY_PRODUCTION_STOCK'].sum().reset_index()
        
        merged_inv = dispatchable_plant_inv.merge(source_plant_prod, on=['SKU', 'PLANT'], how='left', 
                                                 suffixes=('_dispatchable', '_source'))
        merged_inv['_source'] = merged_inv['TODAY_PRODUCTION_STOCK_source'].fillna(0)
        merged_inv['is_valid'] = merged_inv['TODAY_PRODUCTION_STOCK_dispatchable'] == merged_inv['_source']
        
        plant_inv_valid = merged_inv['is_valid'].all()
        invalid_count = (~merged_inv['is_valid']).sum()
        
        return {
            'test_case': 'Dispatchable.TODAY_PRODUCTION_STOCK == SourcePlantInv.TODAY_PRODUCTION_STOCK',
            'status': 'PASS' if plant_inv_valid else 'FAIL',
            'notes': f"All records valid" if plant_inv_valid else f"{invalid_count} records invalid"
        }
    except Exception as e:
        return {
            'test_case': 'Dispatchable.TODAY_PRODUCTION_STOCK == SourcePlantInv.TODAY_PRODUCTION_STOCK',
            'status': 'ERROR',
            'notes': f"Error during validation: {str(e)}"
        }


def validate_dealer_sku_open_orders(dataframes):
    """Validates that DealerSKU.TOTAL_OPEN_ORDER_QTY matches SourceOpenOrder.TOTAL_OPEN_ORDER_QTY"""
    try:
        dealer_sku_orders = dataframes['DealerSKU'].groupby(['DEALER', 'SKU'])['TOTAL_OPEN_ORDER_QTY'].sum().reset_index()
        source_open_orders_agg = dataframes['SourceOpenOrders'].groupby(['DEALER', 'SKU'])['TOTAL_OPEN_ORDER_QTY'].sum().reset_index()
        
        source_open_orders_agg.rename(columns={'TOTAL_OPEN_ORDER_QTY': 'SOURCE_TOTAL_OPEN_ORDER_QTY'}, inplace=True)
        
        merged_dealer_orders = dealer_sku_orders.merge(source_open_orders_agg, on=['DEALER', 'SKU'], how='left')
        merged_dealer_orders['SOURCE_TOTAL_OPEN_ORDER_QTY'] = merged_dealer_orders['SOURCE_TOTAL_OPEN_ORDER_QTY'].fillna(0)
        merged_dealer_orders['is_valid'] = merged_dealer_orders['TOTAL_OPEN_ORDER_QTY'] == merged_dealer_orders['SOURCE_TOTAL_OPEN_ORDER_QTY']
        merged_dealer_orders.loc[ merged_dealer_orders['is_valid']==False]
        dealer_orders_valid = merged_dealer_orders['is_valid'].all()
        invalid_count = (~merged_dealer_orders['is_valid']).sum()
        
        return {
            'test_case': 'DealerSKU.TOTAL_OPEN_ORDER_QTY == SourceOpenOrder.TOTAL_OPEN_ORDER_QTY',
            'status': 'PASS' if dealer_orders_valid else 'FAIL',
            'notes': f"All records valid" if dealer_orders_valid else f"{invalid_count} records invalid"
        }
    except Exception as e:
        return {
            'test_case': 'DealerSKU.TOTAL_OPEN_ORDER_QTY == SourceOpenOrder.TOTAL_OPEN_ORDER_QTY',
            'status': 'ERROR',
            'notes': f"Error during validation: {str(e)}"
        }


def validate_dealer_sku_dispatch(dataframes):
    """Validates that DealerSKU.TOTAL_DISPATCH matches SourceDispatch.QUANTITY_DISPATCHED"""
    try:
        dealer_sku_dispatch = dataframes['DealerSKU'].groupby(['DEALER', 'SKU'])['TOTAL_DISPATCH'].sum().reset_index()
        source_dispatch_agg = dataframes['SourceDispatch'].groupby(['DEALER', 'SKU'])['QUANTITY_DISPATCHED'].sum().reset_index()
        
        merged_dealer_dispatch = dealer_sku_dispatch.merge(source_dispatch_agg, on=['DEALER', 'SKU'], how='left')
        
        merged_dealer_dispatch['SOURCE_DISPATCH'] = merged_dealer_dispatch['QUANTITY_DISPATCHED'].fillna(0)
        merged_dealer_dispatch['is_valid'] = merged_dealer_dispatch['TOTAL_DISPATCH'] == merged_dealer_dispatch['SOURCE_DISPATCH']
        merged_dealer_dispatch.loc[ merged_dealer_dispatch['is_valid']==False]
        dealer_dispatch_valid = merged_dealer_dispatch['is_valid'].all()
        invalid_count = (~merged_dealer_dispatch['is_valid']).sum()
        
        return {
            'test_case': 'DealerSKU.TOTAL_DISPATCH == SourceDispatch.QUANTITY_DISPATCHED',
            'status': 'PASS' if dealer_dispatch_valid else 'FAIL',
            'notes': f"All records valid" if dealer_dispatch_valid else f"{invalid_count} records invalid"
        }
    except Exception as e:
        return {
            'test_case': 'DealerSKU.TOTAL_DISPATCH == SourceDispatch.QUANTITY_DISPATCHED',
            'status': 'ERROR',
            'notes': f"Error during validation: {str(e)}"
        }
        
def validate_dispatch_vs_inventory(dataframes):
    """
    Validates that total_units_tobe_dispatched at SKU, PLANT level from AllTrucks
    is less than or equal to the available inventory at plant, SKU level.
    """
    try:
        # Aggregate total units to be dispatched from AllTrucks by SKU and PLANT
        all_trucks_dispatch = dataframes['AllTrucks'].groupby(['SKU', 'PLANT'])['UNITS_TO_BE_DISPATCHED'].sum().reset_index()
        
        # Get available inventory from SourcePlantInv by SKU and PLANT
        plant_inventory = dataframes['InvSnapshot'].groupby(['SKU', 'PLANT'])['EOD_AVAILABLE_QUANTITY'].sum().reset_index()
        
        # Merge the datasets for comparison
        merged_data = all_trucks_dispatch.merge(plant_inventory, on=['SKU', 'PLANT'], how='left')
        
        # Fill any missing inventory values with 0
        merged_data['EOD_AVAILABLE_QUANTITY'] = merged_data['EOD_AVAILABLE_QUANTITY'].fillna(0)
        
        # Check if dispatch quantity is less than or equal to available inventory
        merged_data['is_valid'] = merged_data['UNITS_TO_BE_DISPATCHED'] <= merged_data['EOD_AVAILABLE_QUANTITY']
        
        validation_valid = merged_data['is_valid'].all()
        invalid_count = (~merged_data['is_valid']).sum()
        
        return {
            'test_case': 'AllTrucks.UNITS_TO_BE_DISPATCHED <= InvSnapshot.EOD_AVAILABLE_QUANTITY',
            'status': 'PASS' if validation_valid else 'FAIL',
            'notes': f"All records valid" if validation_valid else f"{invalid_count} records invalid"
        }
    except Exception as e:
        return {
            'test_case': 'AllTrucks.UNITS_TO_BE_DISPATCHED <= InvSnapshot.EOD_AVAILABLE_QUANTITY',
            'status': 'ERROR',
            'notes': f"Error during validation: {str(e)}"
        }

def validate_mop_vs_dispatchable(dataframes):
    """
    Validates that the Quantity in SourceMop matches the Quantity in Dispatchable
    at dealer, SKU, plant level.
    """
    try:
        # Aggregate MOP quantities by dealer, SKU, plant
        mop_quantities = dataframes['SourceMop'].groupby(['DEALER', 'SKU', 'PLANT'])['QUANTITY'].sum().reset_index()
        
        # Aggregate dispatchable quantities by dealer, SKU, plant
        dispatchable_quantities = dataframes['Dispatchable'].groupby(['DEALER', 'SKU', 'PLANT'])['QUANTITY'].sum().reset_index()
        
        # Merge datasets for comparison
        merged_data = dispatchable_quantities.merge(mop_quantities, on=['DEALER', 'SKU', 'PLANT'], 
                                          how='left', suffixes=( '_dispatchable','_mop'))
        
        # Fill any missing values with 0
        merged_data['QUANTITY_dispatchable'] = merged_data['QUANTITY_dispatchable'].fillna(0)
        
        # Check if quantities match
        merged_data['is_valid'] = merged_data['QUANTITY_mop'] == merged_data['QUANTITY_dispatchable']
       
        validation_valid = merged_data['is_valid'].all()
        invalid_count = (~merged_data['is_valid']).sum()
        
        return {
            'test_case': 'SourceMop.QUANTITY == Dispatchable.QUANTITY',
            'status': 'PASS' if validation_valid else 'FAIL',
            'notes': f"All records valid" if validation_valid else f"{invalid_count} records invalid"
        }
    except Exception as e:
        return {
            'test_case': 'SourceMop.QUANTITY == Dispatchable.QUANTITY',
            'status': 'ERROR',
            'notes': f"Error during validation: {str(e)}"
        }


def validate_alltrucks_dispatch_range(dataframes):
    """
    Validates that UNITS_TO_BE_DISPATCHED for AllTrucks is between FINAL_B2G and MAX_QUANTITY 
    of Dispatchable at dealer, SKU, plant level.
    """
    try:
        # Get AllTrucks dispatch data - select only required columns
        all_trucks_dispatch = dataframes['AllTrucks'].groupby(['DEALER', 'SKU', 'PLANT'])['UNITS_TO_BE_DISPATCHED'].sum().reset_index()
        
        # Get Dispatchable range limits - select only required columns
        dispatchable_limits = dataframes['Dispatchable'][['DEALER', 'SKU', 'PLANT', 'FINAL_B2G', 'MAX_QTY']].copy()
        
        # Merge for comparison
        merged_data = all_trucks_dispatch.merge(dispatchable_limits, on=['DEALER', 'SKU', 'PLANT'], how='left')
        
        # Fill missing values with safe defaults
        merged_data['FINAL_B2G'] = merged_data['FINAL_B2G'].fillna(0)
        merged_data['MAX_QTY'] = merged_data['MAX_QTY'].fillna(0)
        
        # Check if within range
        merged_data['is_valid_lower'] = merged_data['UNITS_TO_BE_DISPATCHED'] >= merged_data['FINAL_B2G']
        merged_data['is_valid_upper'] = merged_data['UNITS_TO_BE_DISPATCHED'] <= merged_data['MAX_QTY']
        merged_data['is_valid'] = merged_data['is_valid_upper']
        merged_data.loc[ merged_data['is_valid']==False]
        validation_valid = merged_data['is_valid'].all()
        invalid_count = (~merged_data['is_valid']).sum()
        
        # Create detailed notes for failures
        if not validation_valid:
            below_min = (~merged_data['is_valid_lower']).sum()
            above_max = (~merged_data['is_valid_upper']).sum()
            notes = f"{invalid_count} records invalid: {below_min} below minimum, {above_max} above maximum"
        else:
            notes = "All records valid"
        
        return {
            'test_case': 'FINAL_B2G <= AllTrucks.UNITS_TO_BE_DISPATCHED <= Dispatchable.MAX_QTY',
            'status': 'PASS' if validation_valid else 'FAIL',
            'notes': notes
        }
    except Exception as e:
        return {
            'test_case': 'FINAL_B2G <= AllTrucks.UNITS_TO_BE_DISPATCHED <= Dispatchable.MAX_QTY',
            'status': 'ERROR',
            'notes': f"Error during validation: {str(e)}"
        }

def validate_mintrucks_dispatch_range(dataframes):
    """
    Validates that UNITS_TO_BE_DISPATCHED for MinTrucks is between FINAL_B2G and MAX_QUANTITY 
    of Dispatchable at dealer, SKU, plant level.
    """
    try:
        # Get AllTrucks dispatch data - select only required columns
        all_trucks_dispatch = dataframes['MinTrucks'].groupby(['DEALER', 'SKU', 'PLANT'])['UNITS_TO_BE_DISPATCHED'].sum().reset_index()
        
        # Get Dispatchable range limits - select only required columns
        dispatchable_limits = dataframes['Dispatchable'][['DEALER', 'SKU', 'PLANT', 'FINAL_B2G', 'MAX_QTY']].copy()
        
        # Merge for comparison
        merged_data = all_trucks_dispatch.merge(dispatchable_limits, on=['DEALER', 'SKU', 'PLANT'], how='left')
        
        # Fill missing values with safe defaults
        merged_data['FINAL_B2G'] = merged_data['FINAL_B2G'].fillna(0)
        merged_data['MAX_QTY'] = merged_data['MAX_QTY'].fillna(0)
        
        # Check if within range
        merged_data['is_valid_lower'] = merged_data['UNITS_TO_BE_DISPATCHED'] >= merged_data['FINAL_B2G']
        merged_data['is_valid_upper'] = merged_data['UNITS_TO_BE_DISPATCHED'] <= merged_data['MAX_QTY']
        merged_data['is_valid'] = merged_data['is_valid_upper']
        merged_data.loc[ merged_data['is_valid']==False]
        validation_valid = merged_data['is_valid'].all()
        invalid_count = (~merged_data['is_valid']).sum()
        
        # Create detailed notes for failures
        if not validation_valid:
            below_min = (~merged_data['is_valid_lower']).sum()
            above_max = (~merged_data['is_valid_upper']).sum()
            notes = f"{invalid_count} records invalid: {below_min} below minimum, {above_max} above maximum"
        else:
            notes = "All records valid"
        
        return {
            'test_case': 'FINAL_B2G <= MinTrucks.UNITS_TO_BE_DISPATCHED <= Dispatchable.MAX_QTY',
            'status': 'PASS' if validation_valid else 'FAIL',
            'notes': notes
        }
    except Exception as e:
        return {
            'test_case': 'FINAL_B2G <= MinTrucks.UNITS_TO_BE_DISPATCHED <= Dispatchable.MAX_QTY',
            'status': 'ERROR',
            'notes': f"Error during validation: {str(e)}"
        }
def print_validation_summary(validation_df):
    """Prints a formatted summary of all validation results"""
    print("\nData Relationship Validation Summary:")
    print("=" * 100)
    for idx, row in validation_df.iterrows():
        print(f"Test Case: {row['test_case']}")
        print(f"Status: {row['status']}")
        print(f"Notes: {row['notes']}")
        print("-" * 100)


# Example usage:
dataframes = fetch_all_data(session)
check_dealer_sku_validity(dataframes)
check_trucks_dispatchable_validity(dataframes, "MinTrucks")
check_trucks_dispatchable_validity(dataframes, "MaxTrucks")
check_trucks_dispatchable_validity(dataframes, "AllTrucks")
check_trucks_dealer_sku_validity(dataframes, "MinTrucks")
check_trucks_dealer_sku_validity(dataframes, "MaxTrucks")
check_trucks_dealer_sku_validity(dataframes, "AllTrucks")
validation_df, all_relationships_valid = validate_data_relationships(dataframes)

In [ ]:
#Import libraries
import pandas as pd
from datetime import date
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# 2.1 DealerSKU query
query_dealersku = """
SELECT 
    RUN_DATE,
    RUN_CYCLE_ID,
    DEALER,
    SKU,
    LAST_N_DAY_SALES,
    SAFETY_STOCK,
    PREDICTED_SALES_SKU,
    LEAD_TIME_STOCK,
    CURRENT_AVAILABLE_STOCK,
    REORDER_LEVEL,
    MOP_PLANNED_QUANTITY,
    QUANTITY_DISPATCHED_TODAY,
    TOTAL_DISPATCH,
    TOTAL_B2G,
    TOTAL_OPEN_ORDER_QTY,
    MODIFIED_REORDER_URGENCY_SCORE,
    "Prorated_Quantity",
    "Prorated_B2G",
    PRIORITY_RANK_OVERALL,
    OBL_CR_LIMIT_AVAIL,
    BLOCK_STATUS_HHHD,
    BLOCK_STATUS_HHHG,
    BLOCK_STATUS_HHHU,
    BLOCK_STATUS_HM4N,
    BLOCK_STATUS_HM5V,
    BLOCK_STATUS_HM6C,
    CASE 
        WHEN OBL_CR_LIMIT_AVAIL >= 100000 THEN TRUE 
        ELSE FALSE 
    END AS FUND_STATUS
FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDealerSKUTable t
WHERE (t.run_date, t.run_cycle_id) IN (
    SELECT run_date, MAX(run_cycle_id)
    FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
    WHERE run_date = (
        SELECT MAX(run_date)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
    )
    GROUP BY run_date
)
;
"""

# 2.2 DispatchableInstances query (fixed)
query_dispatchable = """
SELECT 
    RUN_DATE,
    RUN_CYCLE_ID,
    DEALER,
    SKU,
    PLANT,
    QUANTITY,
    MODEL,
    VERSION,
    PREDICTED_SALES_SKU,
    AVG_LEAD_TIME_STOCK      AS LEAD_TIME_STOCK,
    SAFETY_STOCK,
    REORDER_LEVEL,
    CURRENT_AVAILABLE_STOCK,
    CURRENT_MONTH_SALES,
    LAST_N_DAY_SALES,
    QUANTITY_DISPATCHED_TODAY,
    QUANTITY_DISPATCHED,
    TOTAL_OPEN_ORDER_QTY,
    B2G,
    LOAD_SIZE_MULTIPLES,
    AVAILABLE_PLANT_STOCK,
    TODAY_PRODUCTION_STOCK,
    PLANT_SKU_OPEN_ORDER_QTY,
    EOD_AVAILABLE_QUANTITY,
    NORMALISED_B2G_URGENCY_SCORE,
    FINAL_B2G               AS MIN_B2G_FILLABALE,
    "Prorated_Quantity",
    MAX_QTY_V1,
    "Normalized_MAX_QTY",
    MAX_QTY                 AS MAX_B2G_FILLABALE,
    PRIORITY_RANK_OVERALL,
    N,
    PRORITY_RANK_PLANT_WISE
FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances t
WHERE (t.run_date, t.run_cycle_id) IN (
    SELECT run_date, MAX(run_cycle_id)
    FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
    WHERE run_date = (
        SELECT MAX(run_date)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
    )
    GROUP BY run_date
)
;
"""

# 2.3 AllTrucks query
query_alltrucks = """
SELECT 
    RUN_DATE,
    RUN_CYCLE_ID,
    KEY_IDENTIFIER,
    DEALER,
    SKU,
    PLANT,
    MULTIPLIER,
    "Truck_no",
    UNITS_TO_BE_DISPATCHED,
    IS_COMPLETE_TRUCK,
    FUND_AVAILABLE,
    CUMMULATIVE_COST,
    B2G_URGENCY_SCORE,
    "Max_Score",
    "Median_Score",
    "Max_Score_Rank",
    "Median_Score_Rank"
FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DAILYALLTRUCKS t
WHERE (t.run_date, t.run_cycle_id) IN (
    SELECT run_date, MAX(run_cycle_id)
    FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
    WHERE run_date = (
        SELECT MAX(run_date)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
    )
    GROUP BY run_date
)
;
"""

#Step 3 - Read the tables as pandas
df_dealersku = session.sql(query_dealersku).to_pandas()

df_dispatchable = session.sql(query_dispatchable).to_pandas()

df_alltrucks = session.sql(query_alltrucks).to_pandas()


# Step 4: Write all three DataFrames into a single Excel workbook
current_date = date.today().strftime("%Y-%m-%d")
output_filename = f"final_files_{current_date}.xlsx"
file_path = f"/tmp/{output_filename}"

with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
    df_dealersku.to_excel(writer, sheet_name='DealerSKU', index=False)
    df_dispatchable.to_excel(writer, sheet_name='DispatchableInstances', index=False)
    df_alltrucks.to_excel(writer, sheet_name='AllTrucks', index=False)

# import os
# file_path = os.path.join(os.getcwd(), f"final_files_{current_date}.xlsx")
# # session.file.put(file_path, "@DAILY_FILES_TRUCK_DISPATCH", auto_compress=False, overwrite=True)
# # print(f"✅ Data successfully exported to {output_filename}")


In [ ]:
session.file.put(file_path, "@MOP_DATABASE.SOQ.DAILY_FILES_TRUCK_DISPATCH", auto_compress=False, overwrite=True)

In [ ]:
# import snowflake.connector
# import pandas as pd
# from datetime import date

# # Step 1: Connect to Snowflake
# conn = snowflake.connector.connect(
#     user='YASHS',
#     password='Snowflake_PassKey_123@',
#     account='HEROMOTOCORP-ANALYTICS',
#     warehouse='DS_SPO_WH',
#     role='DS1_ROLE',
#     database='MOP_DATABASE',
#     schema='DISPATCH_AUTOMATION_OBL'
# )

# # Step 2: Define the three (corrected) queries

# # 2.1 DealerSKU query
# query_dealersku = """
# SELECT 
#     RUN_DATE,
#     RUN_CYCLE_ID,
#     DEALER,
#     SKU,
#     LAST_N_DAY_SALES,
#     SAFETY_STOCK,
#     PREDICTED_SALES_SKU,
#     LEAD_TIME_STOCK,
#     CURRENT_AVAILABLE_STOCK,
#     REORDER_LEVEL,
#     MOP_PLANNED_QUANTITY,
#     QUANTITY_DISPATCHED_TODAY,
#     TOTAL_DISPATCH,
#     TOTAL_B2G,
#     TOTAL_OPEN_ORDER_QTY,
#     MODIFIED_REORDER_URGENCY_SCORE,
#     "Prorated_Quantity",
#     "Prorated_B2G",
#     PRIORITY_RANK_OVERALL,
#     OBL_CR_LIMIT_AVAIL,
#     BLOCK_STATUS_HHHD,
#     BLOCK_STATUS_HHHG,
#     BLOCK_STATUS_HHHU,
#     BLOCK_STATUS_HM4N,
#     BLOCK_STATUS_HM5V,
#     BLOCK_STATUS_HM6C,
#     CASE 
#         WHEN OBL_CR_LIMIT_AVAIL >= 100000 THEN TRUE 
#         ELSE FALSE 
#     END AS FUND_STATUS
# FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDealerSKUTable t
# WHERE (t.run_date, t.run_cycle_id) IN (
#     SELECT run_date, MAX(run_cycle_id)
#     FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
#     WHERE run_date = (
#         SELECT MAX(run_date)
#         FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
#     )
#     GROUP BY run_date
# )
# ;
# """

# # 2.2 DispatchableInstances query (fixed)
# query_dispatchable = """
# SELECT 
#     RUN_DATE,
#     RUN_CYCLE_ID,
#     DEALER,
#     SKU,
#     PLANT,
#     QUANTITY,
#     MODEL,
#     VERSION,
#     PREDICTED_SALES_SKU,
#     AVG_LEAD_TIME_STOCK      AS LEAD_TIME_STOCK,
#     SAFETY_STOCK,
#     REORDER_LEVEL,
#     CURRENT_AVAILABLE_STOCK,
#     CURRENT_MONTH_SALES,
#     LAST_N_DAY_SALES,
#     QUANTITY_DISPATCHED_TODAY,
#     QUANTITY_DISPATCHED,
#     TOTAL_OPEN_ORDER_QTY,
#     B2G,
#     LOAD_SIZE_MULTIPLES,
#     AVAILABLE_PLANT_STOCK,
#     TODAY_PRODUCTION_STOCK,
#     PLANT_SKU_OPEN_ORDER_QTY,
#     EOD_AVAILABLE_QUANTITY,
#     NORMALISED_B2G_URGENCY_SCORE,
#     FINAL_B2G               AS MIN_B2G_FILLABALE,
#     "Prorated_Quantity",
#     MAX_QTY_V1,
#     "Normalized_MAX_QTY",
#     MAX_QTY                 AS MAX_B2G_FILLABALE,
#     PRIORITY_RANK_OVERALL,
#     N,
#     PRORITY_RANK_PLANT_WISE
# FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances t
# WHERE (t.run_date, t.run_cycle_id) IN (
#     SELECT run_date, MAX(run_cycle_id)
#     FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
#     WHERE run_date = (
#         SELECT MAX(run_date)
#         FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
#     )
#     GROUP BY run_date
# )
# ;
# """

# # 2.3 AllTrucks query
# query_alltrucks = """
# SELECT 
#     RUN_DATE,
#     RUN_CYCLE_ID,
#     KEY_IDENTIFIER,
#     DEALER,
#     SKU,
#     PLANT,
#     MULTIPLIER,
#     "Truck_no",
#     UNITS_TO_BE_DISPATCHED,
#     IS_COMPLETE_TRUCK,
#     FUND_AVAILABLE,
#     CUMMULATIVE_COST,
#     B2G_URGENCY_SCORE,
#     "Max_Score",
#     "Median_Score",
#     "Max_Score_Rank",
#     "Median_Score_Rank"
# FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DAILYALLTRUCKS t
# WHERE (t.run_date, t.run_cycle_id) IN (
#     SELECT run_date, MAX(run_cycle_id)
#     FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
#     WHERE run_date = (
#         SELECT MAX(run_date)
#         FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
#     )
#     GROUP BY run_date
# )
# ;
# """

# # Step 3: Execute each query and load into a DataFrame
# cursor = conn.cursor()

# cursor.execute(query_dealersku)
# df_dealersku = cursor.fetch_pandas_all()

# cursor.execute(query_dispatchable)
# df_dispatchable = cursor.fetch_pandas_all()
 
# cursor.execute(query_alltrucks)
# df_alltrucks = cursor.fetch_pandas_all()

# # cursor.close()

# # Step 4: Write all three DataFrames into a single Excel workbook
# current_date = date.today().strftime("%Y-%m-%d")
# output_filename = f"final_files_{current_date}.xlsx"

# with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
#     df_dealersku.to_excel(writer, sheet_name='DealerSKU', index=False)
#     df_dispatchable.to_excel(writer, sheet_name='DispatchableInstances', index=False)
#     df_alltrucks.to_excel(writer, sheet_name='AllTrucks', index=False)

# # Step 5: Close the Snowflake connection
# # conn.close()

# print(f"✅ Data successfully exported to {output_filename}")


In [ ]:
df1=pd.merge(All_trucks[['DEALER','SKU','KEY_IDENTIFIER','UNITS_TO_BE_DISPATCHED']],prioritisation[['DEALER','SKU','MOP_PLANNED_QUANTITY','TOTAL_B2G']], on =['DEALER','SKU'], how='left')

In [ ]:
df1['Diff_PLAN'] = df1.MOP_PLANNED_QUANTITY - df1.UNITS_TO_BE_DISPATCHED
df1['Diff_B2G'] = df1.TOTAL_B2G - df1.UNITS_TO_BE_DISPATCHED
df1.loc[df1.Diff_PLAN<0,:]
df1.loc[df1.Diff_B2G<0,:]

In [ ]:
prioritisation.loc[(prioritisation.DEALER=='10273') & (prioritisation.SKU=='HDESHDRLMFIPBK')]
#final_df.loc[(final_df.DEALER=='11646') &(final_df.SKU=='HSPLMIRSCFIRPB')]

In [ ]:
dispatchable_data.loc[(dispatchable_data.DEALER=='10273') & (dispatchable_data.SKU=='HDESHDRLMFIPBK')]

In [ ]:
final_df.loc[(final_df.DEALER=='10273') & (final_df.SKU=='HDESHDRLMFIPBK')]

In [ ]:
import datetime 
print(f"Execution completed at {datetime.datetime.now()}")